# Video games domain — `videogames`

Final unified notebook.  It keeps the successful pieces from the earlier runs: fast L1–L2 / L3–L4 generation profiles plus the v14 quality L5 generator.  By default generation is disabled, because the domain has already been generated and this notebook is mainly a reproducibility/formality artifact.


In [ ]:
# Load common helpers only if this domain notebook is run standalone.
from pathlib import Path

if "BenchmarkExample" not in globals():
    helper_path = Path("common_helpers.py")
    if not helper_path.exists():
        helper_path = Path("/mnt/data/common_helpers.py")
    exec(helper_path.read_text(encoding="utf-8"), globals())



## Generator and final profiles

Default: `RUN_VIDEOGAMES_GENERATION = False`, so running the notebook only registers helpers/templates and performs sanity checks.

Available profiles, if regeneration is ever needed:

- `l1_l2_success` — fast direct/simple patterns for the already-successful L1–L2 part.
- `l3_l4_success` — multihop-heavy L3–L4 tail using the faster direct hidden-bridge implementation.
- `l5_quality` — strict v14 L5-only generator: distinct double-hidden bridges, visible genre/platform constraint, earliest-publication-date semantics.
- `full_formal` — one-shot 130-record formal profile combining the above plans. Use only if you intentionally want a full regeneration.

To generate a profile manually, set for example:

```python
RUN_VIDEOGAMES_GENERATION = True
VIDEOGAMES_GENERATION_PROFILE = 'l5_quality'
```


In [ ]:
import json
import random
import re
import time
from collections import Counter, defaultdict
from contextlib import contextmanager
from dataclasses import asdict, fields
from pathlib import Path
from typing import Any, Callable, Dict, Iterable, List, Optional, Sequence, Tuple

try:
    from tqdm.auto import tqdm
except Exception:  # pragma: no cover
    tqdm = None

# Core class: Wikidata item for "video game".
Q_VIDEO_GAME = "Q7889"

VIDEOGAMES_SEED = globals().get("VIDEOGAMES_SEED", 20260522)
VIDEOGAMES_RNG = random.Random(VIDEOGAMES_SEED)

# Requested tail-only run: generate only L5 examples; keep earlier L1-L4 outputs untouched.
VIDEOGAMES_TARGET_PLAN: Dict[str, int] = {
    "L1": 0,
    "L2": 0,
    "L3": 0,
    "L4": 0,
    "L5": 30,
}
VIDEOGAMES_REQUESTED_BY_LEVEL = {"L1": 5, "L2": 5, "L3": 4, "L4": 3, "L5": 3}

# Gold policy. 301 is probe limit, 300 is max saved gold size; by default we skip
# truncated probes, so accepted records are very likely to have complete golds.
VIDEOGAMES_PROBE_LIMIT = int(globals().get("VIDEOGAMES_PROBE_LIMIT", 301))
VIDEOGAMES_MAX_ACCEPTED_GOLD = int(globals().get("VIDEOGAMES_MAX_ACCEPTED_GOLD", 300))
VIDEOGAMES_SKIP_TRUNCATED_GOLD = bool(globals().get("VIDEOGAMES_SKIP_TRUNCATED_GOLD", True))
VIDEOGAMES_USE_WDQS_CACHE = bool(globals().get("VIDEOGAMES_USE_WDQS_CACHE", True))
VIDEOGAMES_ENFORCE_UNIQUE_PUBLIC_GOLD_LABELS = bool(globals().get("VIDEOGAMES_ENFORCE_UNIQUE_PUBLIC_GOLD_LABELS", True))

# WDQS candidate probes fail fast.  A failed random candidate should not hang the whole run.
VIDEOGAMES_WDQS_FAIL_FAST = bool(globals().get("VIDEOGAMES_WDQS_FAIL_FAST", True))
# Do not fail good candidates too aggressively: videogame WDQS queries often need more than 18s.
VIDEOGAMES_WDQS_FAST_TIMEOUT_SECONDS = int(globals().get("VIDEOGAMES_WDQS_FAST_TIMEOUT_SECONDS", 45))
VIDEOGAMES_WDQS_FAST_MAX_RETRIES = int(globals().get("VIDEOGAMES_WDQS_FAST_MAX_RETRIES", 3))
VIDEOGAMES_POOL_LIMIT = int(globals().get("VIDEOGAMES_POOL_LIMIT", 700))
VIDEOGAMES_SEED_POOL_LIMIT = int(globals().get("VIDEOGAMES_SEED_POOL_LIMIT", 800))

# Important stability switch.  Earlier versions tried to build broad property pools such as
# `videogames_platform_pool_v9` through WDQS.  Those broad pool queries are exactly what
# caused repeated `Read timed out` / `Truncated JSON` messages.  By default v12 uses
# deterministic API/curated anchor pools and does NOT build broad pools unless you opt in.
VIDEOGAMES_USE_BROAD_POOLS = bool(globals().get("VIDEOGAMES_USE_BROAD_POOLS", False))
VIDEOGAMES_VALIDATE_ANCHOR_POOLS_WITH_WDQS = bool(globals().get("VIDEOGAMES_VALIDATE_ANCHOR_POOLS_WITH_WDQS", False))
VIDEOGAMES_RESOLVE_MISSING_ANCHORS_WITH_API = bool(globals().get("VIDEOGAMES_RESOLVE_MISSING_ANCHORS_WITH_API", False))

VIDEOGAMES_DOMAIN_OUT_DIR = Path("out_wikidata_benchmark/domain_outputs")
VIDEOGAMES_DOMAIN_OUT_DIR.mkdir(parents=True, exist_ok=True)
VIDEOGAMES_OUTPUT_PATH = VIDEOGAMES_DOMAIN_OUT_DIR / "videogames_l5.jsonl"
VIDEOGAMES_AUDIT_PATH = VIDEOGAMES_DOMAIN_OUT_DIR / "videogames_l5_generation_audit.json"
VIDEOGAMES_CHECKPOINT_PATH = VIDEOGAMES_DOMAIN_OUT_DIR / "videogames_l5_generation_checkpoint.json"

# Set to False before running/importing the notebook if you only want the generator functions.
RUN_VIDEOGAMES_GENERATION = bool(globals().get("RUN_VIDEOGAMES_GENERATION", False))
OVERWRITE_VIDEOGAMES_OUTPUT = bool(globals().get("OVERWRITE_VIDEOGAMES_OUTPUT", True))
VIDEOGAMES_STRICT_TARGET = bool(globals().get("VIDEOGAMES_STRICT_TARGET", True))

# Preferred labels are used for better human-facing constraints.  The actual QIDs
# are loaded from Wikidata pools, not hard-coded, so this stays robust to label/QID drift.
VG_PREFERRED: Dict[str, set] = {
    "genre": {
        "role-playing video game", "platform game", "strategy video game", "shooter game",
        "puzzle video game", "adventure game", "simulation video game", "racing video game",
        "stealth game", "survival horror", "fighting game", "roguelike", "tower defense",
        "action-adventure game", "sports video game", "massively multiplayer online role-playing game",
    },
    "platform": {
        "Nintendo Switch", "PlayStation 4", "PlayStation 5", "PlayStation 3", "PlayStation 2",
        "Xbox One", "Xbox Series X and Series S", "Nintendo DS", "Game Boy Advance",
        "Microsoft Windows", "Android", "iOS", "Wii", "Nintendo 3DS",
    },
    "developer": {
        "Nintendo", "Ubisoft", "Capcom", "Konami", "Valve", "Bethesda Game Studios",
        "id Software", "Rockstar Games", "CD Projekt Red", "FromSoftware", "Square Enix",
        "Sega", "Electronic Arts", "Naughty Dog", "BioWare", "Blizzard Entertainment",
    },
    "publisher": {
        "Nintendo", "Electronic Arts", "Ubisoft", "Square Enix", "Sony Interactive Entertainment",
        "Xbox Game Studios", "Activision", "Sega", "Konami", "Bandai Namco Entertainment",
        "Capcom", "Bethesda Softworks", "Take-Two Interactive", "Rockstar Games",
    },
    "series": {
        "Mario", "The Legend of Zelda", "Final Fantasy", "Resident Evil", "Pokémon",
        "Assassin's Creed", "Halo", "Civilization", "Call of Duty", "Street Fighter",
        "Mortal Kombat", "Metal Gear", "Sonic the Hedgehog", "The Elder Scrolls",
    },
    "engine": {
        "Unreal Engine", "Unity", "Source", "id Tech", "RE Engine", "Creation Engine",
        "Frostbite", "CryEngine", "RAGE", "Gamebryo", "Havok",
    },
}


# Exact-label anchor pools are used first.  They make generation much more stable
# than relying on arbitrary WDQS LIMIT ordering in broad property pools.  Broad
# property pools are still used as a fallback if an anchor query is temporarily
# unavailable or returns too little variety.
VG_ANCHOR_LABELS: Dict[str, Tuple[str, ...]] = {
    key: tuple(sorted(vals))
    for key, vals in VG_PREFERRED.items()
}

# Fallback QIDs for highly common labels.  These are used only to bootstrap the
# anchor pools without running broad WDQS scans; every final answer is still
# validated by the generated SELECT/ASK SPARQL.  Labels that are ambiguous or
# not listed here are resolved through Wikidata's entity-search API at runtime.
VG_ANCHOR_FALLBACK_QIDS: Dict[str, Dict[str, str]] = {
    "genre": {
        "role-playing video game": "Q744038",
        "platform game": "Q828322",
        "strategy video game": "Q472055",
        "shooter game": "Q4282636",
        "puzzle video game": "Q54767",
        "adventure game": "Q23916",
        "simulation video game": "Q1610017",
        "racing video game": "Q860750",
        "stealth game": "Q682892",
        "survival horror": "Q200876",
        "fighting game": "Q846224",
        "roguelike": "Q1143132",
        "tower defense": "Q132311",
        "action-adventure game": "Q343568",
        "sports video game": "Q868217",
        "massively multiplayer online role-playing game": "Q175173",
    },
    "platform": {
        "Nintendo Switch": "Q19610114",
        "PlayStation 4": "Q5014725",
        "PlayStation 5": "Q63184502",
        "PlayStation 3": "Q10677",
        "PlayStation 2": "Q10680",
        "Xbox One": "Q13361286",
        "Nintendo DS": "Q170323",
        "Game Boy Advance": "Q188642",
        "Microsoft Windows": "Q1406",
        "Android": "Q94",
        "iOS": "Q48493",
        "Wii": "Q8079",
        "Nintendo 3DS": "Q203597",
    },
    "developer": {
        "Nintendo": "Q8093",
        "Ubisoft": "Q188273",
        "Capcom": "Q14428",
        "Konami": "Q45700",
        "Valve": "Q193559",
        "id Software": "Q207922",
        "Rockstar Games": "Q94933",
        "FromSoftware": "Q2743982",
        "Square Enix": "Q207784",
        "Sega": "Q122741",
        "Electronic Arts": "Q173941",
        "Naughty Dog": "Q130522",
        "BioWare": "Q522484",
        "Blizzard Entertainment": "Q178824",
    },
    "publisher": {
        "Nintendo": "Q8093",
        "Electronic Arts": "Q173941",
        "Ubisoft": "Q188273",
        "Square Enix": "Q207784",
        "Activision": "Q200491",
        "Sega": "Q122741",
        "Konami": "Q45700",
        "Capcom": "Q14428",
        "Bethesda Softworks": "Q684425",
        "Take-Two Interactive": "Q94939",
    },
    "series": {
        "The Legend of Zelda": "Q744989",
        "Final Fantasy": "Q123886",
        "Resident Evil": "Q734654",
        "Assassin's Creed": "Q420292",
        "Halo": "Q175736",
        "Call of Duty": "Q200432",
        "Metal Gear": "Q746263",
        "The Elder Scrolls": "Q689778",
    },
    "engine": {
        "Unreal Engine": "Q209711",
        "Unity": "Q272629",
        "Source": "Q193564",
        "RE Engine": "Q28910549",
        "Creation Engine": "Q6064111",
        "Frostbite": "Q1137099",
        "CryEngine": "Q1056652",
    },
}

# Russian display overrides for clean prompt text.  If absent, the English label is used.
VG_RU_OVERRIDES: Dict[str, str] = {
    "role-playing video game": "ролевая видеоигра",
    "platform game": "платформер",
    "strategy video game": "стратегическая видеоигра",
    "shooter game": "шутер",
    "puzzle video game": "головоломка",
    "adventure game": "приключенческая игра",
    "simulation video game": "симулятор",
    "racing video game": "гоночная игра",
    "stealth game": "стелс-игра",
    "survival horror": "survival horror",
    "fighting game": "файтинг",
    "action-adventure game": "action-adventure",
    "sports video game": "спортивная видеоигра",
    "massively multiplayer online role-playing game": "MMORPG",
    "Microsoft Windows": "Windows",
    "Xbox Series X and Series S": "Xbox Series X/S",
    "Sony Interactive Entertainment": "Sony Interactive Entertainment",
    "Bandai Namco Entertainment": "Bandai Namco Entertainment",
    "The Legend of Zelda": "The Legend of Zelda",
    "Sonic the Hedgehog": "Sonic the Hedgehog",
    "The Elder Scrolls": "The Elder Scrolls",
}

VG_PROPERTY_PIDS = {
    "genre": "P136",
    "platform": "P400",
    "developer": "P178",
    "publisher": "P123",
    "series": "P179",
    "engine": "P408",
}

VG_BRIDGES = {
    "developer": {"pid": "P178", "public_key": "same_developer_as", "ru": "тем же разработчиком", "en": "the same developer"},
    "publisher": {"pid": "P123", "public_key": "same_publisher_as", "ru": "тем же издателем", "en": "the same publisher"},
    "series": {"pid": "P179", "public_key": "same_series_as", "ru": "той же серии", "en": "the same series"},
    "engine": {"pid": "P408", "public_key": "same_game_engine_as", "ru": "тем же игровым движком", "en": "the same game engine"},
}

VG_BAD_LABEL_RE = re.compile(
    r"(soundtrack|album|novel|book|film|television series|episode|downloadable content|DLC pack|expansion pack)$",
    flags=re.I,
)

def _vg_norm(s: Any) -> str:
    return re.sub(r"\s+", " ", str(s or "").strip())


def _vg_norm_key(s: Any) -> str:
    return _vg_norm(s).casefold()


def _vg_clean_label(s: Any) -> str:
    s = _vg_norm(s)
    if re.fullmatch(r"Q\d+", s):
        return ""
    return s


def _vg_clean_qid(s: Any) -> str:
    """Normalize a Wikidata QID without treating it as a bad public label."""
    s = _vg_norm(s)
    return s if re.fullmatch(r"Q\d+", s) else ""


def _vg_display_ru(label_en: str, label_ru: Optional[str] = None) -> str:
    en = _vg_clean_label(label_en)
    ru = _vg_clean_label(label_ru)
    return VG_RU_OVERRIDES.get(en) or ru or en


def _vg_entity_from_row(row: Any, prefix: str = "") -> Dict[str, str]:
    def get(col: str) -> Any:
        if isinstance(row, dict):
            return row.get(col)
        return getattr(row, col)
    qid = _vg_clean_qid(get(prefix + "qid"))
    label_en = _vg_clean_label(get(prefix + "label_en"))
    label_ru = _vg_display_ru(label_en, get(prefix + "label_ru"))
    return {"qid": qid, "en": label_en, "ru": label_ru}


def _vg_is_good_public_label(label: str) -> bool:
    label = _vg_clean_label(label)
    if not label:
        return False
    if VG_BAD_LABEL_RE.search(label):
        return False
    return True


def _vg_year_phrase_ru(y1: Optional[int], y2: Optional[int]) -> str:
    if y1 is None or y2 is None:
        return ""
    if int(y1) == int(y2):
        return f"в {int(y1)} году"
    return f"в период {int(y1)}–{int(y2)} годов"


def _vg_year_phrase_en(y1: Optional[int], y2: Optional[int]) -> str:
    if y1 is None or y2 is None:
        return ""
    if int(y1) == int(y2):
        return f"in {int(y1)}"
    return f"from {int(y1)} to {int(y2)}"


def _vg_counted_video_games_ru(k: int) -> str:
    """Return a grammatically safe Russian counted phrase for video games."""
    k = int(k)
    last_two = k % 100
    last = k % 10
    if last_two not in range(11, 15) and last == 1:
        noun = "видеоигру"
    elif last_two not in range(11, 15) and last in {2, 3, 4}:
        noun = "видеоигры"
    else:
        noun = "видеоигр"
    return f"{k} {noun}"


def _vg_clause_join_ru(parts: Sequence[str]) -> str:
    return ", ".join([p for p in parts if p])


def _vg_clause_join_en(parts: Sequence[str]) -> str:
    return ", ".join([p for p in parts if p])


@contextmanager
def _vg_wdqs_fail_fast_context():
    if not VIDEOGAMES_WDQS_FAIL_FAST:
        yield
        return
    wd_obj = globals().get("wd")
    if wd_obj is None:
        yield
        return
    old_timeout = getattr(wd_obj, "timeout", None)
    old_retries = getattr(wd_obj, "max_retries", None)
    try:
        try:
            wd_obj.timeout = VIDEOGAMES_WDQS_FAST_TIMEOUT_SECONDS
            wd_obj.max_retries = VIDEOGAMES_WDQS_FAST_MAX_RETRIES
        except Exception:
            pass
        yield
    finally:
        try:
            if old_timeout is not None:
                wd_obj.timeout = old_timeout
            if old_retries is not None:
                wd_obj.max_retries = old_retries
        except Exception:
            pass


def _vg_rows_from_sparql(query: str, use_cache: bool = VIDEOGAMES_USE_WDQS_CACHE) -> List[Dict[str, str]]:
    return rows_from_select(wd.sparql_select(query, use_cache=use_cache))




# Runtime memory cache: if a transient pool query fails once, do not hammer WDQS
# with the same broad query dozens of times in the same run.
_VG_POOL_MEMORY: Dict[str, pd.DataFrame] = {}


def _vg_empty_pool(cols: Sequence[str]) -> pd.DataFrame:
    return pd.DataFrame(columns=list(cols))


def _vg_load_or_build_pool_once(name: str, builder: Callable[[], pd.DataFrame], cols: Sequence[str]) -> pd.DataFrame:
    if name in _VG_POOL_MEMORY:
        return _VG_POOL_MEMORY[name].copy()
    df = load_pool_df(name)
    if df is not None and len(df) > 0:
        _VG_POOL_MEMORY[name] = df.copy()
        return df.copy()
    try:
        df = builder()
    except Exception as e:
        print(f"[WARN] pool '{name}' build failed once and will be skipped for this run: {e}")
        df = _vg_empty_pool(cols)
    if df is None or len(df) == 0:
        df = _vg_empty_pool(cols)
    else:
        try:
            save_pool_df(name, df)
        except Exception as e:
            print(f"[WARN] pool '{name}' save failed: {e}")
    _VG_POOL_MEMORY[name] = df.copy()
    return df.copy()


def _vg_build_api_anchor_pool(prop_name: str) -> pd.DataFrame:
    """Build a small deterministic pool through Wikidata API/entity search.

    This avoids the broad WDQS scans that were timing out for platform/developer
    pools.  The final SELECT/ASK queries still validate all constraints in WDQS.
    """
    labels = list(VG_ANCHOR_LABELS.get(prop_name, ()))
    fallbacks = VG_ANCHOR_FALLBACK_QIDS.get(prop_name, {})
    data: List[Dict[str, str]] = []
    for en in labels:
        en = _vg_clean_label(en)
        if not _vg_is_good_public_label(en):
            continue
        qid = _vg_clean_qid(fallbacks.get(en))
        if not qid and VIDEOGAMES_RESOLVE_MISSING_ANCHORS_WITH_API:
            try:
                qid = _vg_clean_qid(resolve_qid(en, "en"))
            except Exception:
                qid = ""
        if qid:
            data.append({"qid": qid, "label_en": en, "label_ru": _vg_display_ru(en)})
    if not data:
        return _vg_empty_pool(["qid", "label_en", "label_ru"])
    return pd.DataFrame(data).drop_duplicates(subset=["qid", "label_en"]).reset_index(drop=True)

def _vg_build_anchor_property_pool(prop_name: str, pid: str) -> pd.DataFrame:
    labels = list(VG_ANCHOR_LABELS.get(prop_name, ()))
    if not labels:
        return pd.DataFrame(columns=["qid", "label_en", "label_ru"])
    values = " ".join(json.dumps(x) + "@en" for x in labels)
    sparql = f"""
    SELECT DISTINCT ?val ?valLabelEn ?valLabelRu WHERE {{
      VALUES ?valLabelEn {{ {values} }}
      ?val rdfs:label ?valLabelEn FILTER(LANG(?valLabelEn) = "en") .
      ?item wdt:P31/wdt:P279* wd:{Q_VIDEO_GAME} ;
            wdt:{pid} ?val .
      OPTIONAL {{ ?val rdfs:label ?valLabelRu FILTER(LANG(?valLabelRu) = "ru") . }}
    }}
    """
    rows = _vg_rows_from_sparql(sparql)
    data = []
    for r in rows:
        qid = uri_to_qid(r.get("val", ""))
        en = _vg_clean_label(r.get("valLabelEn"))
        ru = _vg_display_ru(en, r.get("valLabelRu"))
        if qid and _vg_is_good_public_label(en):
            data.append({"qid": qid, "label_en": en, "label_ru": ru})
    if not data:
        return pd.DataFrame(columns=["qid", "label_en", "label_ru"])
    return pd.DataFrame(data).drop_duplicates(subset=["qid", "label_en"]).reset_index(drop=True)

def _vg_build_property_pool(prop_name: str, pid: str, limit: int = VIDEOGAMES_POOL_LIMIT) -> pd.DataFrame:
    sparql = f"""
    SELECT DISTINCT ?val ?valLabelEn ?valLabelRu WHERE {{
      ?item wdt:P31/wdt:P279* wd:{Q_VIDEO_GAME} ;
            wdt:{pid} ?val .
      ?val rdfs:label ?valLabelEn FILTER(LANG(?valLabelEn) = "en") .
      OPTIONAL {{ ?val rdfs:label ?valLabelRu FILTER(LANG(?valLabelRu) = "ru") . }}
    }}
    LIMIT {int(limit)}
    """
    rows = _vg_rows_from_sparql(sparql)
    data = []
    for r in rows:
        qid = uri_to_qid(r.get("val", ""))
        en = _vg_clean_label(r.get("valLabelEn"))
        ru = _vg_display_ru(en, r.get("valLabelRu"))
        if qid and _vg_is_good_public_label(en):
            data.append({"qid": qid, "label_en": en, "label_ru": ru})
    if not data:
        return pd.DataFrame(columns=["qid", "label_en", "label_ru"])
    return pd.DataFrame(data).drop_duplicates(subset=["qid", "label_en"]).reset_index(drop=True)


def _vg_pool(prop_name: str) -> pd.DataFrame:
    pid = VG_PROPERTY_PIDS[prop_name]
    cols = ["qid", "label_en", "label_ru"]

    # 1) Fast deterministic API/curated anchors.  This is the default source.
    api_anchor = _vg_load_or_build_pool_once(
        f"videogames_{prop_name}_api_anchor_pool_v12",
        lambda: _vg_build_api_anchor_pool(prop_name),
        cols,
    )
    frames = [api_anchor] if api_anchor is not None and len(api_anchor) > 0 else []

    # 2) Optional WDQS validation of anchors.  Disabled by default because the
    # final gold/ASK queries already validate records, and pool validation was a
    # major source of timeouts on WDQS.
    if VIDEOGAMES_VALIDATE_ANCHOR_POOLS_WITH_WDQS:
        wdqs_anchor = _vg_load_or_build_pool_once(
            f"videogames_{prop_name}_anchor_pool_v12",
            lambda: _vg_build_anchor_property_pool(prop_name, pid),
            cols,
        )
        if wdqs_anchor is not None and len(wdqs_anchor) > 0:
            frames.append(wdqs_anchor)

    # 3) Optional broad pool.  Opt-in only.  This is intentionally not run by
    # default because broad pools like videogames_platform_pool_v9/v10 repeatedly
    # timed out and prevented generation from starting.
    if VIDEOGAMES_USE_BROAD_POOLS:
        broad = _vg_load_or_build_pool_once(
            f"videogames_{prop_name}_pool_v12",
            lambda: _vg_build_property_pool(prop_name, pid),
            cols,
        )
        if broad is not None and len(broad) > 0:
            frames.append(broad)

    if not frames:
        return _vg_empty_pool(cols)

    out = pd.concat(frames, ignore_index=True)
    for col in cols:
        if col not in out.columns:
            out[col] = ""
    out["qid"] = out["qid"].map(_vg_clean_qid)
    out["label_en"] = out["label_en"].map(_vg_clean_label)
    out["label_ru"] = [_vg_display_ru(en, ru) for en, ru in zip(out["label_en"], out["label_ru"])]
    out = out[out["qid"].str.fullmatch(r"Q\d+").fillna(False)]
    out = out[out["label_en"].map(_vg_is_good_public_label)]
    anchors = set(VG_ANCHOR_LABELS.get(prop_name, ()))
    out["_anchor_rank"] = out["label_en"].map(lambda x: 0 if x in anchors else 1)
    out = out.sort_values(["_anchor_rank", "label_en", "qid"]).drop(columns=["_anchor_rank"])
    return out.drop_duplicates(subset=["qid", "label_en"]).reset_index(drop=True)

def _vg_pick(prop_name: str, rng: random.Random, preferred_bias: float = 0.78) -> Dict[str, str]:
    df = _vg_pool(prop_name)
    if df is None or len(df) == 0:
        raise ValueError(f"Empty videogames pool: {prop_name}")
    preferred = VG_PREFERRED.get(prop_name, set())
    sub = df[df["label_en"].isin(preferred)].copy() if preferred else pd.DataFrame()
    use_sub = len(sub) >= 3 and rng.random() < preferred_bias
    pick_df = sub if use_sub else df
    row = pick_df.sample(1, random_state=rng.randint(0, 10**9)).iloc[0]
    return _vg_entity_from_row(row)


def _vg_build_seed_pool(bridge: str, limit: int = VIDEOGAMES_SEED_POOL_LIMIT) -> pd.DataFrame:
    pid = VG_BRIDGES[bridge]["pid"]
    sparql = f"""
    SELECT DISTINCT ?seed ?seedLabelEn ?seedLabelRu ?bridgeVal ?bridgeValLabelEn ?bridgeValLabelRu WHERE {{
      ?seed wdt:P31/wdt:P279* wd:{Q_VIDEO_GAME} ;
            wdt:{pid} ?bridgeVal ;
            wdt:P577 ?seedDate .
      ?seed rdfs:label ?seedLabelEn FILTER(LANG(?seedLabelEn) = "en") .
      OPTIONAL {{ ?seed rdfs:label ?seedLabelRu FILTER(LANG(?seedLabelRu) = "ru") . }}
      ?bridgeVal rdfs:label ?bridgeValLabelEn FILTER(LANG(?bridgeValLabelEn) = "en") .
      OPTIONAL {{ ?bridgeVal rdfs:label ?bridgeValLabelRu FILTER(LANG(?bridgeValLabelRu) = "ru") . }}
      FILTER(!REGEX(?seedLabelEn, "soundtrack|album|film|novel", "i")) .
    }}
    LIMIT {int(limit)}
    """
    rows = _vg_rows_from_sparql(sparql)
    data = []
    for r in rows:
        seed_qid = uri_to_qid(r.get("seed", ""))
        bridge_qid = uri_to_qid(r.get("bridgeVal", ""))
        seed_en = _vg_clean_label(r.get("seedLabelEn"))
        bridge_en = _vg_clean_label(r.get("bridgeValLabelEn"))
        if not (seed_qid and bridge_qid and _vg_is_good_public_label(seed_en) and _vg_is_good_public_label(bridge_en)):
            continue
        data.append({
            "seed_qid": seed_qid,
            "seed_label_en": seed_en,
            "seed_label_ru": _vg_display_ru(seed_en, r.get("seedLabelRu")),
            "bridge_qid": bridge_qid,
            "bridge_label_en": bridge_en,
            "bridge_label_ru": _vg_display_ru(bridge_en, r.get("bridgeValLabelRu")),
        })
    if not data:
        return pd.DataFrame(columns=["seed_qid", "seed_label_en", "seed_label_ru", "bridge_qid", "bridge_label_en", "bridge_label_ru"])
    return pd.DataFrame(data).drop_duplicates(subset=["seed_qid", "bridge_qid"]).reset_index(drop=True)


def _vg_seed_pool(bridge: str) -> pd.DataFrame:
    df = load_or_build_pool(f"videogames_seed_pool_{bridge}_v10", lambda: _vg_build_seed_pool(bridge))
    if df is None or len(df) == 0:
        return pd.DataFrame(columns=["seed_qid", "seed_label_en", "seed_label_ru", "bridge_qid", "bridge_label_en", "bridge_label_ru"])
    out = df.copy()
    for col in ["seed_qid", "seed_label_en", "seed_label_ru", "bridge_qid", "bridge_label_en", "bridge_label_ru"]:
        if col not in out.columns:
            out[col] = ""
    out["seed_qid"] = out["seed_qid"].map(_vg_clean_qid)
    out["bridge_qid"] = out["bridge_qid"].map(_vg_clean_qid)
    for col in ["seed_label_en", "seed_label_ru", "bridge_label_en", "bridge_label_ru"]:
        out[col] = out[col].map(_vg_clean_label)
    out = out[out["seed_qid"].str.fullmatch(r"Q\d+").fillna(False)]
    out = out[out["bridge_qid"].str.fullmatch(r"Q\d+").fillna(False)]
    out = out[out["seed_label_en"].map(_vg_is_good_public_label)]
    out = out[out["bridge_label_en"].map(_vg_is_good_public_label)]
    return out.drop_duplicates(subset=["seed_qid", "bridge_qid"]).reset_index(drop=True)


def _vg_seed_pool_for_bridge_value(bridge: str, bridge_value: Dict[str, str]) -> pd.DataFrame:
    """Find seed games for one concrete developer/publisher/series/engine.

    Earlier versions built a huge generic seed pool per bridge.  This targeted
    query is smaller and also makes the hidden bridge value high-coverage because
    it is picked from the curated property pool first.
    """
    pid = VG_BRIDGES[bridge]["pid"]
    bq = _vg_clean_qid(bridge_value.get("qid"))
    cols = ["seed_qid", "seed_label_en", "seed_label_ru", "bridge_qid", "bridge_label_en", "bridge_label_ru"]
    if not bq:
        return _vg_empty_pool(cols)
    name = f"videogames_seed_pool_{bridge}_{bq}_v12"

    def builder() -> pd.DataFrame:
        sparql = f"""
        SELECT DISTINCT ?seed ?seedLabelEn ?seedLabelRu WHERE {{
          ?seed wdt:P31/wdt:P279* wd:{Q_VIDEO_GAME} ;
                wdt:{pid} wd:{bq} ;
                wdt:P577 ?seedDate .
          ?seed rdfs:label ?seedLabelEn FILTER(LANG(?seedLabelEn) = "en") .
          OPTIONAL {{ ?seed rdfs:label ?seedLabelRu FILTER(LANG(?seedLabelRu) = "ru") . }}
          FILTER(!REGEX(?seedLabelEn, "soundtrack|album|film|novel", "i")) .
        }}
        LIMIT 120
        """
        rows = _vg_rows_from_sparql(sparql)
        data = []
        for r in rows:
            seed_qid = uri_to_qid(r.get("seed", ""))
            seed_en = _vg_clean_label(r.get("seedLabelEn"))
            if seed_qid and _vg_is_good_public_label(seed_en):
                data.append({
                    "seed_qid": seed_qid,
                    "seed_label_en": seed_en,
                    "seed_label_ru": _vg_display_ru(seed_en, r.get("seedLabelRu")),
                    "bridge_qid": bq,
                    "bridge_label_en": bridge_value.get("en", ""),
                    "bridge_label_ru": bridge_value.get("ru", ""),
                })
        if not data:
            return _vg_empty_pool(cols)
        return pd.DataFrame(data).drop_duplicates(subset=["seed_qid", "bridge_qid"]).reset_index(drop=True)

    return _vg_load_or_build_pool_once(name, builder, cols)


def _vg_pick_seed(bridge: str, rng: random.Random) -> Dict[str, Dict[str, str]]:
    # Pick a high-coverage bridge value first, then find a seed game for it.
    values = _vg_pool(bridge)
    if values is None or len(values) == 0:
        raise ValueError(f"Empty videogames bridge-value pool for: {bridge}")

    preferred = VG_PREFERRED.get(bridge, set())
    if preferred:
        values_pref = values[values["label_en"].isin(preferred)].copy()
    else:
        values_pref = pd.DataFrame()
    candidate_df = values_pref if len(values_pref) > 0 else values
    order = list(candidate_df.sample(frac=1, random_state=rng.randint(0, 10**9)).to_dict("records"))

    for row in order[:12]:
        bridge_value = _vg_entity_from_row(row)
        seed_df = _vg_seed_pool_for_bridge_value(bridge, bridge_value)
        if seed_df is None or len(seed_df) == 0:
            continue
        seed_row = seed_df.sample(1, random_state=rng.randint(0, 10**9)).iloc[0]
        seed_en = _vg_clean_label(seed_row.seed_label_en)
        bridge_en = _vg_clean_label(seed_row.bridge_label_en)
        return {
            "seed": {"qid": seed_row.seed_qid, "en": seed_en, "ru": _vg_display_ru(seed_en, seed_row.seed_label_ru)},
            "bridge_value": {"qid": seed_row.bridge_qid, "en": bridge_en, "ru": _vg_display_ru(bridge_en, seed_row.bridge_label_ru)},
            "bridge": bridge,
        }

    raise ValueError(f"Empty videogames seed pool for bridge: {bridge}")


def _vg_compatible_value_pool(bridge: str, bridge_value: Dict[str, str], prop_name: str) -> pd.DataFrame:
    """Values of prop_name that actually co-occur with a concrete bridge value."""
    bridge_pid = VG_BRIDGES[bridge]["pid"]
    prop_pid = VG_PROPERTY_PIDS[prop_name]
    bq = _vg_clean_qid(bridge_value.get("qid"))
    cols = ["qid", "label_en", "label_ru"]
    if not bq:
        return _vg_empty_pool(cols)
    name = f"videogames_compatible_{bridge}_{bq}_{prop_name}_v12"

    def builder() -> pd.DataFrame:
        sparql = f"""
        SELECT DISTINCT ?val ?valLabelEn ?valLabelRu WHERE {{
          ?item wdt:P31/wdt:P279* wd:{Q_VIDEO_GAME} ;
                wdt:{bridge_pid} wd:{bq} ;
                wdt:{prop_pid} ?val .
          ?val rdfs:label ?valLabelEn FILTER(LANG(?valLabelEn) = "en") .
          OPTIONAL {{ ?val rdfs:label ?valLabelRu FILTER(LANG(?valLabelRu) = "ru") . }}
        }}
        LIMIT 120
        """
        rows = _vg_rows_from_sparql(sparql)
        data = []
        for r in rows:
            qid = uri_to_qid(r.get("val", ""))
            en = _vg_clean_label(r.get("valLabelEn"))
            if qid and _vg_is_good_public_label(en):
                data.append({"qid": qid, "label_en": en, "label_ru": _vg_display_ru(en, r.get("valLabelRu"))})
        if not data:
            return _vg_empty_pool(cols)
        return pd.DataFrame(data).drop_duplicates(subset=["qid", "label_en"]).reset_index(drop=True)

    return _vg_load_or_build_pool_once(name, builder, cols)


def _vg_pick_visible_for_bridge(bridge: str, bridge_value: Dict[str, str], prop_name: str, rng: random.Random) -> Dict[str, str]:
    compatible = _vg_compatible_value_pool(bridge, bridge_value, prop_name)
    if compatible is not None and len(compatible) > 0:
        preferred = VG_PREFERRED.get(prop_name, set())
        sub = compatible[compatible["label_en"].isin(preferred)].copy() if preferred else pd.DataFrame()
        pick_df = sub if len(sub) > 0 and rng.random() < 0.65 else compatible
        row = pick_df.sample(1, random_state=rng.randint(0, 10**9)).iloc[0]
        return _vg_entity_from_row(row)
    # Last resort: global pool.  The final gold query still decides acceptance.
    return _vg_pick(prop_name, rng)

def _vg_year_window(complexity: str, rng: random.Random) -> Tuple[int, int]:
    choices = {
        "L1": [(1990, 2025)],
        "L2": [(1990, 2005), (2000, 2015), (2010, 2025)],
        "L3": [(1996, 2008), (2004, 2013), (2010, 2018), (2016, 2025)],
        "L4": [(1998, 2007), (2006, 2014), (2012, 2020), (2018, 2025)],
        "L5": [(2000, 2008), (2008, 2015), (2014, 2021), (2018, 2025)],
    }
    return rng.choice(choices.get(complexity, choices["L3"]))


def _vg_year_where(y1: Optional[int], y2: Optional[int]) -> List[str]:
    if y1 is None or y2 is None:
        return []
    return [
        "?item wdt:P577 ?releaseDate .",
        "BIND(YEAR(?releaseDate) AS ?releaseYear) .",
        f"FILTER(?releaseYear >= {int(y1)} && ?releaseYear <= {int(y2)}) .",
    ]


def _vg_select_sparql(where_lines: Sequence[str], limit: int = VIDEOGAMES_PROBE_LIMIT) -> str:
    where = "\n      ".join([w for w in where_lines if w])
    return f"""
    SELECT DISTINCT ?item ?itemLabelEn ?itemLabelRu WHERE {{
      ?item wdt:P31/wdt:P279* wd:{Q_VIDEO_GAME} .
      {where}
      ?item rdfs:label ?itemLabelEn FILTER(LANG(?itemLabelEn) = "en") .
      OPTIONAL {{ ?item rdfs:label ?itemLabelRu FILTER(LANG(?itemLabelRu) = "ru") . }}
      FILTER(!REGEX(?itemLabelEn, "soundtrack|album|film|novel", "i")) .
    }}
    LIMIT {int(limit)}
    """.strip()


def _vg_ask_validator(where_lines: Sequence[str]) -> str:
    where = "\n      ".join([w for w in where_lines if w])
    return f"""
    # WDQS-only validator. Replace {{ITEM}} with a candidate QID.
    ASK WHERE {{
      BIND(wd:{{ITEM}} AS ?item)
      ?item wdt:P31/wdt:P279* wd:{Q_VIDEO_GAME} .
      {where}
    }}
    """.strip()


def _vg_collect_gold(where_lines: Sequence[str], limit: int = VIDEOGAMES_PROBE_LIMIT) -> Tuple[str, List[Dict[str, str]], Dict[str, Any]]:
    sparql = _vg_select_sparql(where_lines, limit=limit)
    rows = _vg_rows_from_sparql(sparql, use_cache=VIDEOGAMES_USE_WDQS_CACHE)

    gold: List[Dict[str, str]] = []
    seen_qids = set()
    seen_public_labels = set()
    dropped_no_qid = 0
    dropped_no_en = 0
    dropped_duplicate_label = 0
    dropped_bad_label = 0
    label_sources = {"ru_label": 0, "en_fallback_for_ru": 0}

    for r in rows:
        qid = uri_to_qid(r.get("item", ""))
        if not qid:
            dropped_no_qid += 1
            continue
        if qid in seen_qids:
            continue
        en = _vg_clean_label(r.get("itemLabelEn"))
        if not en:
            dropped_no_en += 1
            continue
        if not _vg_is_good_public_label(en):
            dropped_bad_label += 1
            continue
        ru_raw = _vg_clean_label(r.get("itemLabelRu"))
        ru = ru_raw or en
        if ru_raw:
            label_sources["ru_label"] += 1
        else:
            label_sources["en_fallback_for_ru"] += 1
        if VIDEOGAMES_ENFORCE_UNIQUE_PUBLIC_GOLD_LABELS:
            public_key = _vg_norm_key(en)
            if public_key in seen_public_labels:
                dropped_duplicate_label += 1
                continue
            seen_public_labels.add(public_key)
        seen_qids.add(qid)
        gold.append({"qid": qid, "label_en": en, "label_ru": ru})

    truncated_by_wdqs = len(rows) >= int(limit)
    gold_total_before_local_limit = len(gold)
    if len(gold) > VIDEOGAMES_MAX_ACCEPTED_GOLD:
        gold = gold[:VIDEOGAMES_MAX_ACCEPTED_GOLD]

    meta = {
        "source": "wikidata_sparql",
        "wdqs_candidate_limit": int(limit),
        "rows_returned_by_wdqs": len(rows),
        "gold_returned_before_limits": gold_total_before_local_limit,
        "dropped_no_qid_count": dropped_no_qid,
        "dropped_no_en_label_count": dropped_no_en,
        "dropped_bad_label_count": dropped_bad_label,
        "dropped_duplicate_public_label_count": dropped_duplicate_label,
        "label_sources": label_sources,
        "gold_may_be_incomplete_due_to_wdqs_limit": bool(truncated_by_wdqs),
        "gold_limit": VIDEOGAMES_MAX_ACCEPTED_GOLD,
        "gold_returned": len(gold),
        "gold_total_before_limit": gold_total_before_local_limit,
        "gold_truncated_by_local_limit": gold_total_before_local_limit > VIDEOGAMES_MAX_ACCEPTED_GOLD,
        "constraints_are_wdqs_only": True,
    }
    return sparql, gold, meta


def _vg_public_constraints(**kwargs: Any) -> Dict[str, Any]:
    out = {"kind": "video_game"}
    for k, v in kwargs.items():
        if v is None:
            continue
        if isinstance(v, dict):
            # Public constraints keep only English labels, never QIDs.
            val = v.get("en") or v.get("label_en")
            if val:
                out[k] = val
        elif isinstance(v, (list, tuple)):
            vals = []
            for x in v:
                if isinstance(x, dict):
                    val = x.get("en") or x.get("label_en")
                    if val:
                        vals.append(val)
                elif x is not None:
                    vals.append(x)
            if vals:
                out[k] = vals
        else:
            out[k] = v
    return out


def _vg_constraint_qids(**kwargs: Any) -> Dict[str, Any]:
    out: Dict[str, Any] = {}
    for k, v in kwargs.items():
        if v is None:
            continue
        if isinstance(v, dict):
            if v.get("qid"):
                out[k] = {"qid": v.get("qid"), "label_en": v.get("en"), "label_ru": v.get("ru")}
        elif isinstance(v, (list, tuple)):
            vals = []
            for x in v:
                if isinstance(x, dict) and x.get("qid"):
                    vals.append({"qid": x.get("qid"), "label_en": x.get("en"), "label_ru": x.get("ru")})
            if vals:
                out[k] = vals
        else:
            out[k] = v
    return out


def _vg_build_record(
    *,
    idx: int,
    complexity: str,
    template_id: str,
    template_family: str,
    query_text_ru: str,
    query_text_en: str,
    constraints: Dict[str, Any],
    qid_constraints: Dict[str, Any],
    where_lines: Sequence[str],
    requested_count: int,
) -> Optional[BenchmarkExample]:
    sparql, gold, meta = _vg_collect_gold(where_lines, limit=VIDEOGAMES_PROBE_LIMIT)
    if len(gold) < requested_count:
        return None
    if VIDEOGAMES_SKIP_TRUNCATED_GOLD and meta["gold_may_be_incomplete_due_to_wdqs_limit"]:
        return None

    meta.update({
        "template_id": template_id,
        "template_family": template_family,
        "constraints_with_qids": qid_constraints,
    })
    ask = _vg_ask_validator(where_lines)

    return BenchmarkExample(
        id=f"videogames_{complexity.lower()}_{idx:04d}",
        domain="videogames",
        complexity=complexity,
        query_text_ru=query_text_ru,
        constraints=constraints,
        requested_count=int(requested_count),
        gold_answer_qids=[x["qid"] for x in gold],
        gold_answer_labels_ru=[x["label_ru"] for x in gold],
        sparql_query=sparql,
        created_at=utc_now_z(),
        query_text_en=query_text_en,
        gold_answer_labels_en=[x["label_en"] for x in gold],
        is_advanced=(complexity in {"L4", "L5"}),
        template_id=template_id,
        template_family=template_family,
        gold_truncated=bool(meta["gold_may_be_incomplete_due_to_wdqs_limit"] or meta["gold_truncated_by_local_limit"]),
        ask_validator_sparql=ask,
        local_validator={
            "type": "none_wdqs_only",
            "source": "Wikidata Query Service",
            "applies_after": "ask_validator_sparql",
            "filters": constraints,
            "label_matching_used": False,
            "note": "All video-game constraints for this task are represented in the WDQS ASK validator; no external local validator is required.",
        },
        gold_collection_meta=meta,
    )

def _vg_requested(complexity: str) -> int:
    return int(VIDEOGAMES_REQUESTED_BY_LEVEL.get(complexity, 3))


def _vg_tpl_direct_genre_platform_year(complexity: str, rng: random.Random) -> Dict[str, Any]:
    genre = _vg_pick("genre", rng)
    platform = _vg_pick("platform", rng)
    y1, y2 = _vg_year_window(complexity, rng)
    where = [
        f"?item wdt:P136 wd:{genre['qid']} .",
        f"?item wdt:P400 wd:{platform['qid']} .",
    ] + _vg_year_where(y1, y2)
    k = _vg_requested(complexity)
    k_ru = _vg_counted_video_games_ru(k)
    ru = f"Назови {k_ru}, которые относятся к жанру «{genre['ru']}» и были выпущены на платформе {platform['ru']} {_vg_year_phrase_ru(y1, y2)}."
    en = f"Name {k} video games in the {genre['en']} genre, released for {platform['en']} {_vg_year_phrase_en(y1, y2)}."
    return {
        "template_id": "vg_direct_genre_platform_year",
        "template_family": "direct_visible_multi_criteria",
        "query_text_ru": ru,
        "query_text_en": en,
        "constraints": _vg_public_constraints(genre=genre, platform=platform, release_year_from=y1, release_year_to=y2),
        "qid_constraints": _vg_constraint_qids(genre=genre, platform=platform, release_year_from=y1, release_year_to=y2),
        "where_lines": where,
        "requested_count": k,
    }


def _vg_tpl_direct_developer_platform_year(complexity: str, rng: random.Random) -> Dict[str, Any]:
    developer = _vg_pick("developer", rng)
    platform = _vg_pick("platform", rng)
    y1, y2 = _vg_year_window(complexity, rng)
    where = [
        f"?item wdt:P178 wd:{developer['qid']} .",
        f"?item wdt:P400 wd:{platform['qid']} .",
    ] + _vg_year_where(y1, y2)
    k = _vg_requested(complexity)
    k_ru = _vg_counted_video_games_ru(k)
    ru = f"Назови {k_ru}, которые разработаны {developer['ru']} и выпущены на платформе {platform['ru']} {_vg_year_phrase_ru(y1, y2)}."
    en = f"Name {k} video games developed by {developer['en']} and released for {platform['en']} {_vg_year_phrase_en(y1, y2)}."
    return {
        "template_id": "vg_direct_developer_platform_year",
        "template_family": "direct_visible_multi_criteria",
        "query_text_ru": ru,
        "query_text_en": en,
        "constraints": _vg_public_constraints(developer=developer, platform=platform, release_year_from=y1, release_year_to=y2),
        "qid_constraints": _vg_constraint_qids(developer=developer, platform=platform, release_year_from=y1, release_year_to=y2),
        "where_lines": where,
        "requested_count": k,
    }


def _vg_tpl_direct_publisher_genre_year(complexity: str, rng: random.Random) -> Dict[str, Any]:
    publisher = _vg_pick("publisher", rng)
    genre = _vg_pick("genre", rng)
    y1, y2 = _vg_year_window(complexity, rng)
    where = [
        f"?item wdt:P123 wd:{publisher['qid']} .",
        f"?item wdt:P136 wd:{genre['qid']} .",
    ] + _vg_year_where(y1, y2)
    k = _vg_requested(complexity)
    k_ru = _vg_counted_video_games_ru(k)
    ru = f"Назови {k_ru}, которые относятся к жанру «{genre['ru']}» и изданы {publisher['ru']} {_vg_year_phrase_ru(y1, y2)}."
    en = f"Name {k} video games in the {genre['en']} genre, published by {publisher['en']} {_vg_year_phrase_en(y1, y2)}."
    return {
        "template_id": "vg_direct_publisher_genre_year",
        "template_family": "direct_visible_multi_criteria",
        "query_text_ru": ru,
        "query_text_en": en,
        "constraints": _vg_public_constraints(publisher=publisher, genre=genre, release_year_from=y1, release_year_to=y2),
        "qid_constraints": _vg_constraint_qids(publisher=publisher, genre=genre, release_year_from=y1, release_year_to=y2),
        "where_lines": where,
        "requested_count": k,
    }


def _vg_tpl_direct_series_platform_year(complexity: str, rng: random.Random) -> Dict[str, Any]:
    series = _vg_pick("series", rng)
    platform = _vg_pick("platform", rng)
    y1, y2 = _vg_year_window(complexity, rng)
    where = [
        f"?item wdt:P179 wd:{series['qid']} .",
        f"?item wdt:P400 wd:{platform['qid']} .",
    ] + _vg_year_where(y1, y2)
    k = _vg_requested(complexity)
    k_ru = _vg_counted_video_games_ru(k)
    ru = f"Назови {k_ru}, которые входят в серию «{series['ru']}» и были выпущены на платформе {platform['ru']} {_vg_year_phrase_ru(y1, y2)}."
    en = f"Name {k} video games from the {series['en']} series, released for {platform['en']} {_vg_year_phrase_en(y1, y2)}."
    return {
        "template_id": "vg_direct_series_platform_year",
        "template_family": "direct_visible_multi_criteria",
        "query_text_ru": ru,
        "query_text_en": en,
        "constraints": _vg_public_constraints(series=series, platform=platform, release_year_from=y1, release_year_to=y2),
        "qid_constraints": _vg_constraint_qids(series=series, platform=platform, release_year_from=y1, release_year_to=y2),
        "where_lines": where,
        "requested_count": k,
    }


def _vg_tpl_same_bridge_as_seed(
    complexity: str,
    rng: random.Random,
    *,
    bridge: str,
    visible: Sequence[str] = (),
) -> Dict[str, Any]:
    seed_pack = _vg_pick_seed(bridge, rng)
    seed = seed_pack["seed"]
    bridge_value = seed_pack["bridge_value"]
    pid = VG_BRIDGES[bridge]["pid"]
    public_key = VG_BRIDGES[bridge]["public_key"]
    y1, y2 = _vg_year_window(complexity, rng)

    where = [
        f"BIND(wd:{seed['qid']} AS ?seed) .",
        f"?seed wdt:{pid} ?bridgeValue .",
        f"?item wdt:{pid} ?bridgeValue .",
        "FILTER(?item != ?seed) .",
    ]
    public_kwargs: Dict[str, Any] = {public_key: seed, "release_year_from": y1, "release_year_to": y2}
    qid_kwargs: Dict[str, Any] = {public_key: seed, "hidden_bridge_value": bridge_value, "release_year_from": y1, "release_year_to": y2}

    visible_phrases_ru: List[str] = []
    visible_phrases_en: List[str] = []

    if "genre" in visible:
        genre = _vg_pick_visible_for_bridge(bridge, bridge_value, "genre", rng)
        where.append(f"?item wdt:P136 wd:{genre['qid']} .")
        public_kwargs["genre"] = genre
        qid_kwargs["genre"] = genre
        visible_phrases_ru.append(f"относятся к жанру «{genre['ru']}»")
        visible_phrases_en.append(f"in the {genre['en']} genre")
    if "platform" in visible:
        platform = _vg_pick_visible_for_bridge(bridge, bridge_value, "platform", rng)
        where.append(f"?item wdt:P400 wd:{platform['qid']} .")
        public_kwargs["platform"] = platform
        qid_kwargs["platform"] = platform
        visible_phrases_ru.append(f"были выпущены на платформе {platform['ru']}")
        visible_phrases_en.append(f"released for {platform['en']}")
    if "publisher" in visible:
        publisher = _vg_pick_visible_for_bridge(bridge, bridge_value, "publisher", rng)
        where.append(f"?item wdt:P123 wd:{publisher['qid']} .")
        public_kwargs["publisher"] = publisher
        qid_kwargs["publisher"] = publisher
        visible_phrases_ru.append(f"изданы {publisher['ru']}")
        visible_phrases_en.append(f"published by {publisher['en']}")
    if "developer" in visible:
        developer = _vg_pick_visible_for_bridge(bridge, bridge_value, "developer", rng)
        where.append(f"?item wdt:P178 wd:{developer['qid']} .")
        public_kwargs["developer"] = developer
        qid_kwargs["developer"] = developer
        visible_phrases_ru.append(f"разработаны {developer['ru']}")
        visible_phrases_en.append(f"developed by {developer['en']}")

    where += _vg_year_where(y1, y2)
    k = _vg_requested(complexity)

    bridge_ru = {
        "developer": f"разработаны тем же разработчиком, что и «{seed['ru']}»",
        "publisher": f"изданы тем же издателем, что и «{seed['ru']}»",
        "series": f"входят в ту же серию, что и «{seed['ru']}»",
        "engine": f"используют тот же игровой движок, что и «{seed['ru']}»",
    }[bridge]
    bridge_en = {
        "developer": f"developed by the same developer as {seed['en']}",
        "publisher": f"published by the same publisher as {seed['en']}",
        "series": f"from the same series as {seed['en']}",
        "engine": f"using the same game engine as {seed['en']}",
    }[bridge]

    ru_parts = [bridge_ru] + visible_phrases_ru + [_vg_year_phrase_ru(y1, y2)]
    en_parts = [bridge_en] + visible_phrases_en + [_vg_year_phrase_en(y1, y2)]

    visible_suffix = "_" + "_".join(visible) if visible else ""
    template_id = f"vg_same_{bridge}_as_seed{visible_suffix}_year"
    template_family = "hidden_seed_bridge"
    k_ru = _vg_counted_video_games_ru(k)
    return {
        "template_id": template_id,
        "template_family": template_family,
        "query_text_ru": f"Назови {k_ru}, которые " + _vg_clause_join_ru(ru_parts) + ".",
        "query_text_en": f"Name {k} video games " + _vg_clause_join_en(en_parts) + ".",
        "constraints": _vg_public_constraints(**public_kwargs),
        "qid_constraints": _vg_constraint_qids(**qid_kwargs),
        "where_lines": where,
        "requested_count": k,
    }


VG_TEMPLATE_BUILDERS: Dict[str, Callable[[str, random.Random], Dict[str, Any]]] = {
    "vg_direct_genre_platform_year": _vg_tpl_direct_genre_platform_year,
    "vg_direct_developer_platform_year": _vg_tpl_direct_developer_platform_year,
    "vg_direct_publisher_genre_year": _vg_tpl_direct_publisher_genre_year,
    "vg_direct_series_platform_year": _vg_tpl_direct_series_platform_year,
    "vg_same_developer_as_seed_year": lambda c, r: _vg_tpl_same_bridge_as_seed(c, r, bridge="developer"),
    "vg_same_publisher_as_seed_year": lambda c, r: _vg_tpl_same_bridge_as_seed(c, r, bridge="publisher"),
    "vg_same_series_as_seed_year": lambda c, r: _vg_tpl_same_bridge_as_seed(c, r, bridge="series"),
    "vg_same_engine_as_seed_year": lambda c, r: _vg_tpl_same_bridge_as_seed(c, r, bridge="engine"),
    "vg_same_developer_as_seed_platform_year": lambda c, r: _vg_tpl_same_bridge_as_seed(c, r, bridge="developer", visible=("platform",)),
    "vg_same_publisher_as_seed_genre_year": lambda c, r: _vg_tpl_same_bridge_as_seed(c, r, bridge="publisher", visible=("genre",)),
    "vg_same_series_as_seed_platform_year": lambda c, r: _vg_tpl_same_bridge_as_seed(c, r, bridge="series", visible=("platform",)),
    "vg_same_engine_as_seed_genre_year": lambda c, r: _vg_tpl_same_bridge_as_seed(c, r, bridge="engine", visible=("genre",)),
    "vg_same_developer_as_seed_genre_platform_year": lambda c, r: _vg_tpl_same_bridge_as_seed(c, r, bridge="developer", visible=("genre", "platform")),
    "vg_same_developer_as_seed_genre_publisher_year": lambda c, r: _vg_tpl_same_bridge_as_seed(c, r, bridge="developer", visible=("genre", "publisher")),
    "vg_same_publisher_as_seed_genre_platform_year": lambda c, r: _vg_tpl_same_bridge_as_seed(c, r, bridge="publisher", visible=("genre", "platform")),
    "vg_same_engine_as_seed_genre_platform_year": lambda c, r: _vg_tpl_same_bridge_as_seed(c, r, bridge="engine", visible=("genre", "platform")),
    "vg_same_series_as_seed_genre_platform_year": lambda c, r: _vg_tpl_same_bridge_as_seed(c, r, bridge="series", visible=("genre", "platform")),
}

VG_TEMPLATE_PLAN_BY_LEVEL: Dict[str, Tuple[str, ...]] = {
    "L1": ("vg_direct_genre_platform_year",),
    "L2": ("vg_direct_genre_platform_year", "vg_direct_developer_platform_year"),
    "L3": (
        "vg_direct_genre_platform_year",
        "vg_direct_developer_platform_year",
        "vg_direct_publisher_genre_year",
        "vg_same_developer_as_seed_year",
        "vg_same_publisher_as_seed_year",
    ),
    "L4": (
        "vg_same_developer_as_seed_platform_year",
        "vg_same_publisher_as_seed_genre_year",
        "vg_same_series_as_seed_platform_year",
        "vg_same_engine_as_seed_genre_year",
        "vg_direct_series_platform_year",
    ),
    "L5": (
        "vg_same_developer_as_seed_genre_platform_year",
        "vg_same_publisher_as_seed_genre_platform_year",
        "vg_same_engine_as_seed_genre_platform_year",
        "vg_same_series_as_seed_genre_platform_year",
        "vg_same_developer_as_seed_platform_year",
    ),
}


def _vg_slot_try_plan(level: str, accepted_index_within_level: int) -> List[str]:
    plan = list(VG_TEMPLATE_PLAN_BY_LEVEL.get(level, ()))
    if not plan:
        raise ValueError(f"No template plan for videogames {level}")
    start = accepted_index_within_level % len(plan)
    rotated = plan[start:] + plan[:start]
    return rotated[:5]


VG_EXPECTED_JSON_KEYS = [f.name for f in fields(BenchmarkExample)]


def _vg_validate_record_schema(ex: BenchmarkExample) -> None:
    d = asdict(ex)
    assert list(d.keys()) == VG_EXPECTED_JSON_KEYS, "JSON key order/structure diverged from BenchmarkExample"
    assert ex.domain == "videogames"
    assert ex.query_text_ru and ex.query_text_en
    assert ex.requested_count > 0
    assert len(ex.gold_answer_qids) == len(ex.gold_answer_labels_ru) == len(ex.gold_answer_labels_en)
    assert len(ex.gold_answer_qids) >= ex.requested_count
    assert ex.sparql_query and ex.ask_validator_sparql
    assert _vg_public_constraints_are_clean(ex.constraints)
    assert isinstance(ex.gold_collection_meta, dict)
    assert isinstance(ex.local_validator, dict)
    assert ex.gold_collection_meta.get("constraints_with_qids"), "QID constraints must be stored in gold_collection_meta"


def _vg_validate_output_jsonl(path: Path, expected_count: Optional[int] = None) -> None:
    if not path.exists():
        raise AssertionError(f"Output JSONL was not created: {path}")
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            if not line.strip():
                continue
            rec = json.loads(line)
            assert list(rec.keys()) == VG_EXPECTED_JSON_KEYS, f"Bad key structure at line {line_no}"
            assert rec.get("domain") == "videogames", f"Bad domain at line {line_no}"
            assert rec.get("query_text_ru") and rec.get("query_text_en"), f"Missing query text at line {line_no}"
            assert len(rec.get("gold_answer_qids", [])) == len(rec.get("gold_answer_labels_ru", [])) == len(rec.get("gold_answer_labels_en", [])), f"Gold list length mismatch at line {line_no}"
            assert len(rec.get("gold_answer_qids", [])) >= int(rec.get("requested_count", 0)), f"Too few golds at line {line_no}"
            assert _vg_public_constraints_are_clean(rec.get("constraints", {})), f"Dirty public constraints at line {line_no}"
            rows.append(rec)
    if expected_count is not None:
        assert len(rows) == int(expected_count), f"Expected {expected_count} JSONL rows, got {len(rows)}"

def generate_videogames_example(
    complexity: str,
    idx: int,
    rng: random.Random,
    max_attempts: int = 140,
    forced_template_id: Optional[str] = None,
) -> BenchmarkExample:
    complexity = str(complexity)
    template_ids = [forced_template_id] if forced_template_id else list(VG_TEMPLATE_PLAN_BY_LEVEL.get(complexity, ()))
    if not template_ids:
        raise ValueError(f"Unknown videogames complexity: {complexity}")

    errors: List[str] = []
    with _vg_wdqs_fail_fast_context():
        for attempt in range(max_attempts):
            tid = forced_template_id or rng.choice(template_ids)
            builder = VG_TEMPLATE_BUILDERS[tid]
            try:
                spec = builder(complexity, rng)
                ex = _vg_build_record(
                    idx=idx,
                    complexity=complexity,
                    template_id=spec["template_id"],
                    template_family=spec["template_family"],
                    query_text_ru=spec["query_text_ru"],
                    query_text_en=spec["query_text_en"],
                    constraints=spec["constraints"],
                    qid_constraints=spec["qid_constraints"],
                    where_lines=spec["where_lines"],
                    requested_count=spec["requested_count"],
                )
                if ex is not None and len(ex.gold_answer_qids) >= ex.requested_count:
                    return ex
                errors.append(f"{tid}: insufficient_or_truncated_gold")
            except Exception as e:
                errors.append(f"{tid}: {type(e).__name__}: {e}")
                if len(errors) > 15:
                    errors = errors[-15:]
                continue

    tail = "; ".join(errors[-8:])
    raise RuntimeError(f"Failed to generate videogames example {complexity} after {max_attempts} attempts. Recent errors: {tail}")


# Backward-compatible aliases for possible external runners.
generate_video_games_example = generate_videogames_example

if "DOMAIN_GENERATORS" in globals():
    DOMAIN_GENERATORS["videogames"] = generate_videogames_example


def _vg_record_key(ex: BenchmarkExample) -> Tuple[str, str, str]:
    public_constraints = json.dumps(ex.constraints, ensure_ascii=False, sort_keys=True)
    return (str(ex.complexity), str(ex.template_id), public_constraints)


def _vg_append_jsonl(path: Path, record: Dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def _vg_write_json(path: Path, obj: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")
    tmp.replace(path)


def _vg_audit(records: Sequence[BenchmarkExample], skipped: Sequence[Dict[str, Any]]) -> Dict[str, Any]:
    return {
        "domain": "videogames",
        "target_plan": dict(VIDEOGAMES_TARGET_PLAN),
        "generated": len(records),
        "counts_by_complexity": dict(Counter(x.complexity for x in records)),
        "counts_by_template_family": dict(Counter(x.template_family for x in records)),
        "counts_by_template_id": dict(Counter(x.template_id for x in records)),
        "gold_policy": {
            "probe_limit": VIDEOGAMES_PROBE_LIMIT,
            "max_accepted_gold": VIDEOGAMES_MAX_ACCEPTED_GOLD,
            "skip_truncated_gold": VIDEOGAMES_SKIP_TRUNCATED_GOLD,
            "enforce_unique_public_gold_labels": VIDEOGAMES_ENFORCE_UNIQUE_PUBLIC_GOLD_LABELS,
        },
        "gold_size_by_complexity": {
            lvl: {
                "min": min([len(x.gold_answer_qids) for x in records if x.complexity == lvl] or [0]),
                "max": max([len(x.gold_answer_qids) for x in records if x.complexity == lvl] or [0]),
                "avg": round(sum(len(x.gold_answer_qids) for x in records if x.complexity == lvl) / max(1, sum(1 for x in records if x.complexity == lvl)), 2),
            }
            for lvl in VIDEOGAMES_TARGET_PLAN
        },
        "template_plan": {k: list(v) for k, v in VG_TEMPLATE_PLAN_BY_LEVEL.items()},
        "skipped_count": len(skipped),
        "skipped_preview": list(skipped)[-80:],
        "seed": VIDEOGAMES_SEED,
        "output_path": str(VIDEOGAMES_OUTPUT_PATH),
    }


def _vg_public_constraints_are_clean(constraints: Dict[str, Any]) -> bool:
    for k, v in constraints.items():
        if k.endswith("_qid") or k.endswith("_ru"):
            return False
        if isinstance(v, str) and re.fullmatch(r"Q\d+", v):
            return False
        if isinstance(v, dict):
            return False
    return True


def _vg_run_sanity_checks() -> None:
    assert Q_VIDEO_GAME == "Q7889"
    assert _vg_clean_label("Q123") == "" and _vg_clean_qid("Q123") == "Q123"
    assert _vg_counted_video_games_ru(3) == "3 видеоигры" and _vg_counted_video_games_ru(5) == "5 видеоигр"
    assert VG_EXPECTED_JSON_KEYS == [f.name for f in fields(BenchmarkExample)]
    assert sum(VIDEOGAMES_TARGET_PLAN.values()) == 30, "Target plan must generate exactly 30 L5 examples"
    assert VIDEOGAMES_TARGET_PLAN == {"L1": 0, "L2": 0, "L3": 0, "L4": 0, "L5": 30}
    assert VIDEOGAMES_TARGET_PLAN.get("L4", 0) == 0 and VIDEOGAMES_TARGET_PLAN.get("L5", 0) == 30
    assert VIDEOGAMES_REQUESTED_BY_LEVEL["L3"] == 4 and VIDEOGAMES_REQUESTED_BY_LEVEL["L5"] == 3
    assert VIDEOGAMES_PROBE_LIMIT == VIDEOGAMES_MAX_ACCEPTED_GOLD + 1
    for lvl in ("L3", "L4", "L5"):
        assert len(VG_TEMPLATE_PLAN_BY_LEVEL[lvl]) >= 5
    sample_constraints = _vg_public_constraints(
        genre={"qid": "Q1", "en": "role-playing video game", "ru": "ролевая видеоигра"},
        platform={"qid": "Q2", "en": "Nintendo Switch", "ru": "Nintendo Switch"},
        release_year_from=2018,
        release_year_to=2025,
    )
    assert sample_constraints == {
        "kind": "video_game",
        "genre": "role-playing video game",
        "platform": "Nintendo Switch",
        "release_year_from": 2018,
        "release_year_to": 2025,
    }
    assert _vg_public_constraints_are_clean(sample_constraints), "Public constraints must be English labels only, no QIDs/RU fields"
    static_where = ["?item wdt:P136 wd:Q1 ."] + _vg_year_where(2010, 2020)
    ask = _vg_ask_validator(static_where)
    assert "ASK WHERE" in ask and "wd:{ITEM}" in ask and "wdt:P31/wdt:P279* wd:Q7889" in ask
    sel = _vg_select_sparql(static_where, limit=301)
    assert "?itemLabelEn" in sel and "?itemLabelRu" in sel and "LIMIT 301" in sel


_vg_run_sanity_checks()
print("✅ videogames quality sanity checks passed: L5-only target (30 L5), clean English constraints, RU/EN labels, WDQS validators, full-gold probe policy, WDQS-safe anchor pools")


def generate_videogames_dataset(
    target_plan: Optional[Dict[str, int]] = None,
    output_path: Path = VIDEOGAMES_OUTPUT_PATH,
    audit_path: Path = VIDEOGAMES_AUDIT_PATH,
    overwrite: bool = OVERWRITE_VIDEOGAMES_OUTPUT,
    seed: int = VIDEOGAMES_SEED,
    max_visible_attempts_per_level: int = 260,
) -> List[BenchmarkExample]:
    target_plan = dict(target_plan or VIDEOGAMES_TARGET_PLAN)
    rng = random.Random(seed)
    records: List[BenchmarkExample] = []
    skipped: List[Dict[str, Any]] = []
    seen: set[Tuple[str, str, str]] = set()

    if overwrite and output_path.exists():
        output_path.unlink()

    total_target = sum(target_plan.values())
    overall_bar = tqdm(total=total_target, desc="videogames total") if tqdm is not None else None
    idx = 1

    try:
        for level, target in target_plan.items():
            if target <= 0:
                continue
            ok = 0
            attempts = 0
            level_bar = tqdm(total=target, desc=f"videogames {level}") if tqdm is not None else None
            while ok < target and attempts < max_visible_attempts_per_level:
                accepted: Optional[BenchmarkExample] = None
                for template_id in _vg_slot_try_plan(level, ok):
                    attempts += 1
                    try:
                        ex = generate_videogames_example(
                            level,
                            idx,
                            rng=rng,
                            max_attempts=28,
                            forced_template_id=template_id,
                        )
                    except Exception as e:
                        skipped.append({"complexity": level, "template_id": template_id, "reason": type(e).__name__, "message": str(e)[:500]})
                        if attempts >= max_visible_attempts_per_level:
                            break
                        continue

                    if not ex.gold_answer_qids:
                        skipped.append({"complexity": level, "template_id": template_id, "reason": "empty_gold"})
                        continue
                    if not ex.query_text_en or not ex.gold_answer_labels_en:
                        skipped.append({"complexity": level, "template_id": template_id, "reason": "missing_en_fields"})
                        continue
                    if not _vg_public_constraints_are_clean(ex.constraints):
                        skipped.append({"complexity": level, "template_id": template_id, "reason": "dirty_public_constraints"})
                        continue
                    key = _vg_record_key(ex)
                    if key in seen:
                        skipped.append({"complexity": level, "template_id": template_id, "reason": "duplicate_public_constraints"})
                        continue
                    accepted = ex
                    break

                if accepted is None:
                    continue

                _vg_validate_record_schema(accepted)
                seen.add(_vg_record_key(accepted))
                records.append(accepted)
                _vg_append_jsonl(output_path, asdict(accepted))
                idx += 1
                ok += 1

                if level_bar is not None:
                    level_bar.update(1)
                    level_bar.set_postfix({"attempts": attempts, "gold": len(accepted.gold_answer_qids), "tpl": accepted.template_id})
                if overall_bar is not None:
                    overall_bar.update(1)
                    overall_bar.set_postfix({"level": level, "gold": len(accepted.gold_answer_qids)})
                _vg_write_json(VIDEOGAMES_CHECKPOINT_PATH, {"last_record": asdict(accepted), "audit": _vg_audit(records, skipped)})

            if level_bar is not None:
                level_bar.close()
            if ok < target:
                print(f"[WARN] videogames {level}: generated {ok}/{target} after {attempts} visible attempts")
            else:
                print(f"OK videogames {level}: generated {ok}/{target} after {attempts} visible attempts")
    finally:
        if overall_bar is not None:
            overall_bar.close()
        audit = _vg_audit(records, skipped)
        _vg_write_json(audit_path, audit)
        print("saved:", output_path.resolve())
        print("audit:", audit_path.resolve())
        print("records:", len(records))
        print("counts:", dict(Counter(x.complexity for x in records)))
        print("families:", dict(Counter(x.template_family for x in records)))
        print("skipped:", len(skipped))

    if output_path.exists():
        _vg_validate_output_jsonl(output_path, expected_count=len(records))
    if VIDEOGAMES_STRICT_TARGET and len(records) != total_target:
        raise RuntimeError(
            f"videogames generation produced {len(records)}/{total_target} records. "
            f"See audit for skipped reasons: {audit_path}"
        )
    _vg_validate_output_jsonl(output_path, expected_count=total_target)
    return records


# === v6 quality patch: stricter gold, clearer date/platform semantics, harder multihop L3-L5 ===

# Accept only tasks whose full gold set is known to be <= 100.  With probe=101,
# any result returning 101 rows is treated as incomplete/too broad and skipped.
VIDEOGAMES_PROBE_LIMIT = int(globals().get("VIDEOGAMES_V6_PROBE_LIMIT", 101))
VIDEOGAMES_MAX_ACCEPTED_GOLD = int(globals().get("VIDEOGAMES_V6_MAX_ACCEPTED_GOLD", 100))
VIDEOGAMES_WDQS_FAST_TIMEOUT_SECONDS = int(globals().get("VIDEOGAMES_V6_WDQS_FAST_TIMEOUT_SECONDS", 55))
VIDEOGAMES_WDQS_FAST_MAX_RETRIES = int(globals().get("VIDEOGAMES_V6_WDQS_FAST_MAX_RETRIES", 3))

# Harder diversity defaults.  These are intentionally conservative: they remove
# exact/near-duplicate tasks such as repeated Assassin's Creed same-series items.
VIDEOGAMES_MAX_SAME_HIDDEN_BRIDGE_GLOBAL = int(globals().get("VIDEOGAMES_MAX_SAME_HIDDEN_BRIDGE_GLOBAL", 3))
VIDEOGAMES_MAX_SAME_HIDDEN_BRIDGE_PER_LEVEL = int(globals().get("VIDEOGAMES_MAX_SAME_HIDDEN_BRIDGE_PER_LEVEL", 2))
VIDEOGAMES_MAX_GOLD_JACCARD_OVERLAP = float(globals().get("VIDEOGAMES_MAX_GOLD_JACCARD_OVERLAP", 0.88))

# Ambiguous high-noise genres are not used as anchors in the regenerated dataset.
# They can still appear only if broad pools are explicitly enabled by the user.
VG_AMBIGUOUS_DIRECT_GENRES = {
    "racing video game", "sports video game", "simulation video game", "action-adventure game",
}
VG_PREFERRED["genre"] = set(x for x in VG_PREFERRED.get("genre", set()) if x not in VG_AMBIGUOUS_DIRECT_GENRES)
VG_ANCHOR_LABELS["genre"] = tuple(x for x in VG_ANCHOR_LABELS.get("genre", ()) if x not in VG_AMBIGUOUS_DIRECT_GENRES)

# Stronger label blacklist.  Important: use word boundaries around demo/pack so
# titles like "Demon Gaze" or "Doko Demo Issyo" are not removed by accident.
VG_BAD_LABEL_RE = re.compile(
    r"(?:"
    r"\bseason\s+pass\b|\bbattle\s+pass\b|\bdlc\b|\bdownloadable\s+content\b|"
    r"\bexpansion\s+pack\b|\bstarter\s+pack\b|\bmission\s+pack\b|\bmap\s+pack\b|\bcontent\s+pack\b|"
    r"\bmega\s+bundle\b|\bbundle\b|\bupgrade\b|\bcollection\b|\bcompilation\b|\banthology\b|"
    r"\bsoundtrack\b|\balbum\b|\bguide\b|\bmanual\b|\btrailer\b|\bwalkthrough\b|\bdemo\b"
    r")",
    flags=re.I,
)
VG_BAD_LABEL_SPARQL_RE = (
    r"season\\s+pass|battle\\s+pass|downloadable\\s+content|expansion\\s+pack|"
    r"starter\\s+pack|mission\\s+pack|map\\s+pack|content\\s+pack|mega\\s+bundle|"
    r"\\bDLC\\b|\\bbundle\\b|\\bupgrade\\b|\\bcollection\\b|\\bcompilation\\b|"
    r"\\banthology\\b|\\bsoundtrack\\b|\\balbum\\b|\\bguide\\b|\\bmanual\\b|"
    r"\\btrailer\\b|\\bwalkthrough\\b|\\bdemo\\b"
)

_VG_POOL_MEMORY.clear()


def _vg_is_good_public_label(label: str) -> bool:
    label = _vg_clean_label(label)
    if not label:
        return False
    if VG_BAD_LABEL_RE.search(label):
        return False
    return True


def _vg_year_window(complexity: str, rng: random.Random) -> Tuple[int, int]:
    # L1/L2 are still direct, but no longer use the huge 1990-2025 window that
    # created 250+ golds.  L3-L5 use tighter windows to produce clean <=100 golds.
    choices = {
        "L1": [(2000, 2008), (2006, 2014), (2012, 2020), (2018, 2025)],
        "L2": [(1996, 2008), (2004, 2013), (2010, 2018), (2016, 2025)],
        "L3": [(1998, 2007), (2006, 2014), (2012, 2020), (2018, 2025)],
        "L4": [(1998, 2007), (2006, 2014), (2012, 2020), (2018, 2025)],
        "L5": [(2002, 2009), (2008, 2015), (2014, 2021), (2018, 2025)],
    }
    return rng.choice(choices.get(complexity, choices["L3"]))


def _vg_year_clause_ru(y1: Optional[int], y2: Optional[int]) -> str:
    return f"были опубликованы {_vg_year_phrase_ru(y1, y2)}" if y1 is not None and y2 is not None else ""


def _vg_year_clause_en(y1: Optional[int], y2: Optional[int]) -> str:
    return f"published {_vg_year_phrase_en(y1, y2)}" if y1 is not None and y2 is not None else ""


def _vg_platform_phrase_ru(platform: Dict[str, str]) -> str:
    return f"имеют платформу {platform['ru']}"


def _vg_platform_phrase_en(platform: Dict[str, str]) -> str:
    return f"have {platform['en']} among their platforms"


def _vg_select_sparql(where_lines: Sequence[str], limit: int = VIDEOGAMES_PROBE_LIMIT) -> str:
    where = "\n      ".join([w for w in where_lines if w])
    return f"""
    SELECT DISTINCT ?item ?itemLabelEn ?itemLabelRu WHERE {{
      ?item wdt:P31/wdt:P279* wd:{Q_VIDEO_GAME} .
      {where}
      ?item rdfs:label ?itemLabelEn FILTER(LANG(?itemLabelEn) = "en") .
      OPTIONAL {{ ?item rdfs:label ?itemLabelRu FILTER(LANG(?itemLabelRu) = "ru") . }}
      FILTER(!REGEX(?itemLabelEn, "{VG_BAD_LABEL_SPARQL_RE}", "i")) .
    }}
    LIMIT {int(limit)}
    """.strip()


def _vg_collect_gold(where_lines: Sequence[str], limit: int = VIDEOGAMES_PROBE_LIMIT) -> Tuple[str, List[Dict[str, str]], Dict[str, Any]]:
    sparql = _vg_select_sparql(where_lines, limit=limit)
    rows = _vg_rows_from_sparql(sparql, use_cache=VIDEOGAMES_USE_WDQS_CACHE)

    gold: List[Dict[str, str]] = []
    seen_qids = set()
    seen_public_labels_en = set()
    seen_public_labels_ru = set()
    dropped_no_qid = 0
    dropped_no_en = 0
    dropped_duplicate_label = 0
    dropped_bad_label = 0
    label_sources = {"ru_label": 0, "en_fallback_for_ru": 0}

    for r in rows:
        qid = uri_to_qid(r.get("item", ""))
        if not qid:
            dropped_no_qid += 1
            continue
        if qid in seen_qids:
            continue
        en = _vg_clean_label(r.get("itemLabelEn"))
        if not en:
            dropped_no_en += 1
            continue
        ru_raw = _vg_clean_label(r.get("itemLabelRu"))
        ru = ru_raw or en
        if not (_vg_is_good_public_label(en) and _vg_is_good_public_label(ru)):
            dropped_bad_label += 1
            continue
        en_key = _vg_norm_key(en)
        ru_key = _vg_norm_key(ru)
        if VIDEOGAMES_ENFORCE_UNIQUE_PUBLIC_GOLD_LABELS and (
            en_key in seen_public_labels_en or ru_key in seen_public_labels_ru
        ):
            dropped_duplicate_label += 1
            continue
        seen_qids.add(qid)
        seen_public_labels_en.add(en_key)
        seen_public_labels_ru.add(ru_key)
        if ru_raw:
            label_sources["ru_label"] += 1
        else:
            label_sources["en_fallback_for_ru"] += 1
        gold.append({"qid": qid, "label_en": en, "label_ru": ru})

    truncated_by_wdqs = len(rows) >= int(limit)
    gold_total_before_local_limit = len(gold)
    if len(gold) > VIDEOGAMES_MAX_ACCEPTED_GOLD:
        gold = gold[:VIDEOGAMES_MAX_ACCEPTED_GOLD]

    meta = {
        "source": "wikidata_sparql",
        "wdqs_candidate_limit": int(limit),
        "rows_returned_by_wdqs": len(rows),
        "gold_returned_before_limits": gold_total_before_local_limit,
        "dropped_no_qid_count": dropped_no_qid,
        "dropped_no_en_label_count": dropped_no_en,
        "dropped_bad_label_count": dropped_bad_label,
        "dropped_duplicate_public_label_count": dropped_duplicate_label,
        "dedupe_public_labels": "en_and_ru_normalized",
        "label_sources": label_sources,
        "gold_may_be_incomplete_due_to_wdqs_limit": truncated_by_wdqs,
        "gold_limit": VIDEOGAMES_MAX_ACCEPTED_GOLD,
        "gold_returned": len(gold),
        "gold_total_before_limit": gold_total_before_local_limit,
        "gold_truncated_by_local_limit": gold_total_before_local_limit > VIDEOGAMES_MAX_ACCEPTED_GOLD,
        "constraints_are_wdqs_only": True,
    }
    return sparql, gold, meta


def _vg_pool(prop_name: str) -> pd.DataFrame:
    pid = VG_PROPERTY_PIDS[prop_name]
    cols = ["qid", "label_en", "label_ru"]
    frames: List[pd.DataFrame] = []
    api_anchor = _vg_load_or_build_pool_once(
        f"videogames_{prop_name}_api_anchor_pool_v13",
        lambda: _vg_build_api_anchor_pool(prop_name),
        cols,
    )
    if api_anchor is not None and len(api_anchor) > 0:
        frames.append(api_anchor)
    if VIDEOGAMES_VALIDATE_ANCHOR_POOLS_WITH_WDQS:
        wdqs_anchor = _vg_load_or_build_pool_once(
            f"videogames_{prop_name}_anchor_pool_v13",
            lambda: _vg_build_anchor_property_pool(prop_name, pid),
            cols,
        )
        if wdqs_anchor is not None and len(wdqs_anchor) > 0:
            frames.append(wdqs_anchor)
    if VIDEOGAMES_USE_BROAD_POOLS:
        broad = _vg_load_or_build_pool_once(
            f"videogames_{prop_name}_pool_v13",
            lambda: _vg_build_property_pool(prop_name, pid),
            cols,
        )
        if broad is not None and len(broad) > 0:
            frames.append(broad)
    if not frames:
        return _vg_empty_pool(cols)
    out = pd.concat(frames, ignore_index=True)
    for col in cols:
        if col not in out.columns:
            out[col] = ""
    out["qid"] = out["qid"].map(_vg_clean_qid)
    out["label_en"] = out["label_en"].map(_vg_clean_label)
    out["label_ru"] = [_vg_display_ru(en, ru) for en, ru in zip(out["label_en"], out["label_ru"])]
    out = out[out["qid"].str.fullmatch(r"Q\d+").fillna(False)]
    out = out[out["label_en"].map(_vg_is_good_public_label)]
    out = out[out["label_ru"].map(_vg_is_good_public_label)]
    anchors = set(VG_ANCHOR_LABELS.get(prop_name, ()))
    out["_anchor_rank"] = out["label_en"].map(lambda x: 0 if x in anchors else 1)
    out = out.sort_values(["_anchor_rank", "label_en", "qid"]).drop(columns=["_anchor_rank"])
    return out.drop_duplicates(subset=["qid", "label_en"]).reset_index(drop=True)


def _vg_seed_pool_for_bridge_value(bridge: str, bridge_value: Dict[str, str]) -> pd.DataFrame:
    pid = VG_BRIDGES[bridge]["pid"]
    bq = _vg_clean_qid(bridge_value.get("qid"))
    cols = ["seed_qid", "seed_label_en", "seed_label_ru", "bridge_qid", "bridge_label_en", "bridge_label_ru"]
    if not bq:
        return _vg_empty_pool(cols)
    name = f"videogames_seed_pool_{bridge}_{bq}_v13"

    def builder() -> pd.DataFrame:
        sparql = f"""
        SELECT DISTINCT ?seed ?seedLabelEn ?seedLabelRu WHERE {{
          ?seed wdt:P31/wdt:P279* wd:{Q_VIDEO_GAME} ;
                wdt:{pid} wd:{bq} ;
                wdt:P577 ?seedDate .
          ?seed rdfs:label ?seedLabelEn FILTER(LANG(?seedLabelEn) = "en") .
          OPTIONAL {{ ?seed rdfs:label ?seedLabelRu FILTER(LANG(?seedLabelRu) = "ru") . }}
          FILTER(!REGEX(?seedLabelEn, "{VG_BAD_LABEL_SPARQL_RE}", "i")) .
        }}
        LIMIT 120
        """
        rows = _vg_rows_from_sparql(sparql)
        data = []
        for r in rows:
            seed_qid = uri_to_qid(r.get("seed", ""))
            seed_en = _vg_clean_label(r.get("seedLabelEn"))
            seed_ru = _vg_display_ru(seed_en, r.get("seedLabelRu"))
            if seed_qid and _vg_is_good_public_label(seed_en) and _vg_is_good_public_label(seed_ru):
                data.append({
                    "seed_qid": seed_qid,
                    "seed_label_en": seed_en,
                    "seed_label_ru": seed_ru,
                    "bridge_qid": bq,
                    "bridge_label_en": bridge_value.get("en", ""),
                    "bridge_label_ru": bridge_value.get("ru", ""),
                })
        if not data:
            return _vg_empty_pool(cols)
        return pd.DataFrame(data).drop_duplicates(subset=["seed_qid", "bridge_qid"]).reset_index(drop=True)

    df = _vg_load_or_build_pool_once(name, builder, cols)
    if df is None or len(df) == 0:
        return _vg_empty_pool(cols)
    out = df.copy()
    for col in cols:
        if col not in out.columns:
            out[col] = ""
    out["seed_qid"] = out["seed_qid"].map(_vg_clean_qid)
    out["bridge_qid"] = out["bridge_qid"].map(_vg_clean_qid)
    out["seed_label_en"] = out["seed_label_en"].map(_vg_clean_label)
    out["seed_label_ru"] = [_vg_display_ru(en, ru) for en, ru in zip(out["seed_label_en"], out["seed_label_ru"])]
    out = out[out["seed_qid"].str.fullmatch(r"Q\d+").fillna(False)]
    out = out[out["bridge_qid"].str.fullmatch(r"Q\d+").fillna(False)]
    out = out[out["seed_label_en"].map(_vg_is_good_public_label)]
    out = out[out["seed_label_ru"].map(_vg_is_good_public_label)]
    return out.drop_duplicates(subset=["seed_qid", "bridge_qid"]).reset_index(drop=True)


def _vg_compatible_value_pool(bridge: str, bridge_value: Dict[str, str], prop_name: str) -> pd.DataFrame:
    bridge_pid = VG_BRIDGES[bridge]["pid"]
    prop_pid = VG_PROPERTY_PIDS[prop_name]
    bq = _vg_clean_qid(bridge_value.get("qid"))
    cols = ["qid", "label_en", "label_ru"]
    if not bq:
        return _vg_empty_pool(cols)
    name = f"videogames_compatible_{bridge}_{bq}_{prop_name}_v13"

    def builder() -> pd.DataFrame:
        sparql = f"""
        SELECT DISTINCT ?val ?valLabelEn ?valLabelRu WHERE {{
          ?item wdt:P31/wdt:P279* wd:{Q_VIDEO_GAME} ;
                wdt:{bridge_pid} wd:{bq} ;
                wdt:{prop_pid} ?val .
          ?val rdfs:label ?valLabelEn FILTER(LANG(?valLabelEn) = "en") .
          OPTIONAL {{ ?val rdfs:label ?valLabelRu FILTER(LANG(?valLabelRu) = "ru") . }}
        }}
        LIMIT 160
        """
        rows = _vg_rows_from_sparql(sparql)
        data = []
        for r in rows:
            qid = uri_to_qid(r.get("val", ""))
            en = _vg_clean_label(r.get("valLabelEn"))
            ru = _vg_display_ru(en, r.get("valLabelRu"))
            if qid and _vg_is_good_public_label(en) and _vg_is_good_public_label(ru):
                data.append({"qid": qid, "label_en": en, "label_ru": ru})
        if not data:
            return _vg_empty_pool(cols)
        return pd.DataFrame(data).drop_duplicates(subset=["qid", "label_en"]).reset_index(drop=True)

    df = _vg_load_or_build_pool_once(name, builder, cols)
    if df is None or len(df) == 0:
        return _vg_empty_pool(cols)
    out = df.copy()
    for col in cols:
        if col not in out.columns:
            out[col] = ""
    out["qid"] = out["qid"].map(_vg_clean_qid)
    out["label_en"] = out["label_en"].map(_vg_clean_label)
    out["label_ru"] = [_vg_display_ru(en, ru) for en, ru in zip(out["label_en"], out["label_ru"])]
    out = out[out["qid"].str.fullmatch(r"Q\d+").fillna(False)]
    out = out[out["label_en"].map(_vg_is_good_public_label)]
    out = out[out["label_ru"].map(_vg_is_good_public_label)]
    return out.drop_duplicates(subset=["qid", "label_en"]).reset_index(drop=True)


def _vg_tpl_direct_genre_platform_year(complexity: str, rng: random.Random) -> Dict[str, Any]:
    genre = _vg_pick("genre", rng)
    platform = _vg_pick("platform", rng)
    y1, y2 = _vg_year_window(complexity, rng)
    where = [
        f"?item wdt:P136 wd:{genre['qid']} .",
        f"?item wdt:P400 wd:{platform['qid']} .",
    ] + _vg_year_where(y1, y2)
    k = _vg_requested(complexity)
    k_ru = _vg_counted_video_games_ru(k)
    ru = f"Назови {k_ru}, которые относятся к жанру «{genre['ru']}», {_vg_platform_phrase_ru(platform)} и {_vg_year_clause_ru(y1, y2)}."
    en = f"Name {k} video games in the {genre['en']} genre that {_vg_platform_phrase_en(platform)} and were {_vg_year_clause_en(y1, y2)}."
    return {
        "template_id": "vg_direct_genre_platform_year",
        "template_family": "direct_visible_multi_criteria",
        "query_text_ru": ru,
        "query_text_en": en,
        "constraints": _vg_public_constraints(genre=genre, platform=platform, release_year_from=y1, release_year_to=y2),
        "qid_constraints": _vg_constraint_qids(genre=genre, platform=platform, release_year_from=y1, release_year_to=y2),
        "where_lines": where,
        "requested_count": k,
    }


def _vg_tpl_direct_developer_platform_year(complexity: str, rng: random.Random) -> Dict[str, Any]:
    developer = _vg_pick("developer", rng)
    platform = _vg_pick("platform", rng)
    y1, y2 = _vg_year_window(complexity, rng)
    where = [
        f"?item wdt:P178 wd:{developer['qid']} .",
        f"?item wdt:P400 wd:{platform['qid']} .",
    ] + _vg_year_where(y1, y2)
    k = _vg_requested(complexity)
    k_ru = _vg_counted_video_games_ru(k)
    ru = f"Назови {k_ru}, которые разработаны {developer['ru']}, {_vg_platform_phrase_ru(platform)} и {_vg_year_clause_ru(y1, y2)}."
    en = f"Name {k} video games developed by {developer['en']} that {_vg_platform_phrase_en(platform)} and were {_vg_year_clause_en(y1, y2)}."
    return {
        "template_id": "vg_direct_developer_platform_year",
        "template_family": "direct_visible_multi_criteria",
        "query_text_ru": ru,
        "query_text_en": en,
        "constraints": _vg_public_constraints(developer=developer, platform=platform, release_year_from=y1, release_year_to=y2),
        "qid_constraints": _vg_constraint_qids(developer=developer, platform=platform, release_year_from=y1, release_year_to=y2),
        "where_lines": where,
        "requested_count": k,
    }


def _vg_tpl_same_bridge_as_seed(complexity: str, rng: random.Random, *, bridge: str, visible: Sequence[str] = ()) -> Dict[str, Any]:
    seed_pack = _vg_pick_seed(bridge, rng)
    seed = seed_pack["seed"]
    bridge_value = seed_pack["bridge_value"]
    pid = VG_BRIDGES[bridge]["pid"]
    public_key = VG_BRIDGES[bridge]["public_key"]
    y1, y2 = _vg_year_window(complexity, rng)
    where = [
        f"BIND(wd:{seed['qid']} AS ?seed) .",
        f"?seed wdt:{pid} ?bridgeValue .",
        f"?item wdt:{pid} ?bridgeValue .",
        "FILTER(?item != ?seed) .",
    ]
    public_kwargs: Dict[str, Any] = {public_key: seed, "release_year_from": y1, "release_year_to": y2}
    qid_kwargs: Dict[str, Any] = {public_key: seed, "hidden_bridge_value": bridge_value, "release_year_from": y1, "release_year_to": y2}
    visible_phrases_ru: List[str] = []
    visible_phrases_en: List[str] = []
    if "genre" in visible:
        genre = _vg_pick_visible_for_bridge(bridge, bridge_value, "genre", rng)
        where.append(f"?item wdt:P136 wd:{genre['qid']} .")
        public_kwargs["genre"] = genre
        qid_kwargs["genre"] = genre
        visible_phrases_ru.append(f"относятся к жанру «{genre['ru']}»")
        visible_phrases_en.append(f"in the {genre['en']} genre")
    if "platform" in visible:
        platform = _vg_pick_visible_for_bridge(bridge, bridge_value, "platform", rng)
        where.append(f"?item wdt:P400 wd:{platform['qid']} .")
        public_kwargs["platform"] = platform
        qid_kwargs["platform"] = platform
        visible_phrases_ru.append(_vg_platform_phrase_ru(platform))
        visible_phrases_en.append(_vg_platform_phrase_en(platform))
    if "publisher" in visible:
        publisher = _vg_pick_visible_for_bridge(bridge, bridge_value, "publisher", rng)
        where.append(f"?item wdt:P123 wd:{publisher['qid']} .")
        public_kwargs["publisher"] = publisher
        qid_kwargs["publisher"] = publisher
        visible_phrases_ru.append(f"изданы {publisher['ru']}")
        visible_phrases_en.append(f"published by {publisher['en']}")
    if "developer" in visible:
        developer = _vg_pick_visible_for_bridge(bridge, bridge_value, "developer", rng)
        where.append(f"?item wdt:P178 wd:{developer['qid']} .")
        public_kwargs["developer"] = developer
        qid_kwargs["developer"] = developer
        visible_phrases_ru.append(f"разработаны {developer['ru']}")
        visible_phrases_en.append(f"developed by {developer['en']}")
    where += _vg_year_where(y1, y2)
    k = _vg_requested(complexity)
    bridge_ru = {
        "developer": f"разработаны тем же разработчиком, что и «{seed['ru']}»",
        "publisher": f"изданы тем же издателем, что и «{seed['ru']}»",
        "series": f"входят в ту же серию, что и «{seed['ru']}»",
        "engine": f"используют тот же игровой движок, что и «{seed['ru']}»",
    }[bridge]
    bridge_en = {
        "developer": f"developed by the same developer as {seed['en']}",
        "publisher": f"published by the same publisher as {seed['en']}",
        "series": f"from the same series as {seed['en']}",
        "engine": f"using the same game engine as {seed['en']}",
    }[bridge]
    ru_parts = [bridge_ru] + visible_phrases_ru + [_vg_year_clause_ru(y1, y2)]
    en_parts = [bridge_en] + visible_phrases_en + [f"were {_vg_year_clause_en(y1, y2)}"]
    visible_suffix = "_" + "_".join(visible) if visible else ""
    template_id = f"vg_same_{bridge}_as_seed{visible_suffix}_year"
    k_ru = _vg_counted_video_games_ru(k)
    return {
        "template_id": template_id,
        "template_family": "hidden_seed_bridge",
        "query_text_ru": f"Назови {k_ru}, которые " + _vg_clause_join_ru(ru_parts) + ".",
        "query_text_en": f"Name {k} video games that " + _vg_clause_join_en(en_parts) + ".",
        "constraints": _vg_public_constraints(**public_kwargs),
        "qid_constraints": _vg_constraint_qids(**qid_kwargs),
        "where_lines": where,
        "requested_count": k,
    }


def _vg_pick_compatible_value(primary_bridge: str, primary_value: Dict[str, str], prop_name: str, rng: random.Random) -> Dict[str, str]:
    df = _vg_compatible_value_pool(primary_bridge, primary_value, prop_name)
    if df is None or len(df) == 0:
        raise ValueError(f"No compatible {prop_name} values for {primary_bridge}={primary_value.get('en')}")
    preferred = VG_PREFERRED.get(prop_name, set())
    sub = df[df["label_en"].isin(preferred)].copy() if preferred else pd.DataFrame()
    pick_df = sub if len(sub) > 0 and rng.random() < 0.75 else df
    row = pick_df.sample(1, random_state=rng.randint(0, 10**9)).iloc[0]
    return _vg_entity_from_row(row)


def _vg_tpl_two_seed_bridges(complexity: str, rng: random.Random, *, bridge_a: str, bridge_b: str, visible: Sequence[str] = ()) -> Dict[str, Any]:
    if bridge_a == bridge_b:
        raise ValueError("Two-seed bridge template requires different bridge types")
    pack_a = _vg_pick_seed(bridge_a, rng)
    seed_a = pack_a["seed"]
    value_a = pack_a["bridge_value"]
    value_b = _vg_pick_compatible_value(bridge_a, value_a, bridge_b, rng)
    seed_b_df = _vg_seed_pool_for_bridge_value(bridge_b, value_b)
    if seed_b_df is None or len(seed_b_df) == 0:
        raise ValueError(f"No seed for secondary bridge {bridge_b}={value_b.get('en')}")
    seed_b_row = seed_b_df.sample(1, random_state=rng.randint(0, 10**9)).iloc[0]
    seed_b = {
        "qid": seed_b_row.seed_qid,
        "en": _vg_clean_label(seed_b_row.seed_label_en),
        "ru": _vg_display_ru(seed_b_row.seed_label_en, seed_b_row.seed_label_ru),
    }
    y1, y2 = _vg_year_window(complexity, rng)
    pid_a = VG_BRIDGES[bridge_a]["pid"]
    pid_b = VG_BRIDGES[bridge_b]["pid"]
    key_a = VG_BRIDGES[bridge_a]["public_key"]
    key_b = VG_BRIDGES[bridge_b]["public_key"]
    where = [
        f"BIND(wd:{seed_a['qid']} AS ?seedA) .",
        f"?seedA wdt:{pid_a} ?bridgeValueA .",
        f"?item wdt:{pid_a} ?bridgeValueA .",
        f"BIND(wd:{seed_b['qid']} AS ?seedB) .",
        f"?seedB wdt:{pid_b} ?bridgeValueB .",
        f"?item wdt:{pid_b} ?bridgeValueB .",
        "FILTER(?item != ?seedA && ?item != ?seedB) .",
    ]
    public_kwargs: Dict[str, Any] = {key_a: seed_a, key_b: seed_b, "release_year_from": y1, "release_year_to": y2}
    qid_kwargs: Dict[str, Any] = {
        key_a: seed_a,
        key_b: seed_b,
        "hidden_bridge_values": [
            {"bridge": bridge_a, **value_a},
            {"bridge": bridge_b, **value_b},
        ],
        "release_year_from": y1,
        "release_year_to": y2,
    }
    ru_parts = []
    en_parts = []
    ru_map = {
        "developer": lambda s: f"разработаны тем же разработчиком, что и «{s['ru']}»",
        "publisher": lambda s: f"изданы тем же издателем, что и «{s['ru']}»",
        "series": lambda s: f"входят в ту же серию, что и «{s['ru']}»",
        "engine": lambda s: f"используют тот же игровой движок, что и «{s['ru']}»",
    }
    en_map = {
        "developer": lambda s: f"developed by the same developer as {s['en']}",
        "publisher": lambda s: f"published by the same publisher as {s['en']}",
        "series": lambda s: f"from the same series as {s['en']}",
        "engine": lambda s: f"using the same game engine as {s['en']}",
    }
    ru_parts += [ru_map[bridge_a](seed_a), ru_map[bridge_b](seed_b)]
    en_parts += [en_map[bridge_a](seed_a), en_map[bridge_b](seed_b)]
    for prop in visible:
        val = _vg_pick_compatible_value(bridge_a, value_a, prop, rng)
        pid = VG_PROPERTY_PIDS[prop]
        where.append(f"?item wdt:{pid} wd:{val['qid']} .")
        public_kwargs[prop] = val
        qid_kwargs[prop] = val
        if prop == "genre":
            ru_parts.append(f"относятся к жанру «{val['ru']}»")
            en_parts.append(f"in the {val['en']} genre")
        elif prop == "platform":
            ru_parts.append(_vg_platform_phrase_ru(val))
            en_parts.append(_vg_platform_phrase_en(val))
        elif prop == "publisher":
            ru_parts.append(f"изданы {val['ru']}")
            en_parts.append(f"published by {val['en']}")
        elif prop == "developer":
            ru_parts.append(f"разработаны {val['ru']}")
            en_parts.append(f"developed by {val['en']}")
    where += _vg_year_where(y1, y2)
    ru_parts.append(_vg_year_clause_ru(y1, y2))
    en_parts.append(f"were {_vg_year_clause_en(y1, y2)}")
    template_id = f"vg_same_{bridge_a}_and_{bridge_b}_as_seeds"
    if visible:
        template_id += "_" + "_".join(visible)
    template_id += "_year"
    k = _vg_requested(complexity)
    k_ru = _vg_counted_video_games_ru(k)
    return {
        "template_id": template_id,
        "template_family": "double_hidden_seed_bridge",
        "query_text_ru": f"Назови {k_ru}, которые " + _vg_clause_join_ru(ru_parts) + ".",
        "query_text_en": f"Name {k} video games that " + _vg_clause_join_en(en_parts) + ".",
        "constraints": _vg_public_constraints(**public_kwargs),
        "qid_constraints": _vg_constraint_qids(**qid_kwargs),
        "where_lines": where,
        "requested_count": k,
    }


VG_TEMPLATE_BUILDERS.update({
    "vg_direct_genre_platform_year": _vg_tpl_direct_genre_platform_year,
    "vg_direct_developer_platform_year": _vg_tpl_direct_developer_platform_year,
    "vg_same_developer_as_seed_genre_year": lambda c, r: _vg_tpl_same_bridge_as_seed(c, r, bridge="developer", visible=("genre",)),
    "vg_same_publisher_as_seed_platform_year": lambda c, r: _vg_tpl_same_bridge_as_seed(c, r, bridge="publisher", visible=("platform",)),
    "vg_same_series_as_seed_genre_year": lambda c, r: _vg_tpl_same_bridge_as_seed(c, r, bridge="series", visible=("genre",)),
    "vg_same_engine_as_seed_platform_year": lambda c, r: _vg_tpl_same_bridge_as_seed(c, r, bridge="engine", visible=("platform",)),
    "vg_same_developer_and_publisher_as_seeds_genre_year": lambda c, r: _vg_tpl_two_seed_bridges(c, r, bridge_a="developer", bridge_b="publisher", visible=("genre",)),
    "vg_same_developer_and_publisher_as_seeds_platform_year": lambda c, r: _vg_tpl_two_seed_bridges(c, r, bridge_a="developer", bridge_b="publisher", visible=("platform",)),
    "vg_same_publisher_and_series_as_seeds_genre_year": lambda c, r: _vg_tpl_two_seed_bridges(c, r, bridge_a="publisher", bridge_b="series", visible=("genre",)),
    "vg_same_developer_and_series_as_seeds_platform_year": lambda c, r: _vg_tpl_two_seed_bridges(c, r, bridge_a="developer", bridge_b="series", visible=("platform",)),
    "vg_same_developer_and_engine_as_seeds_platform_year": lambda c, r: _vg_tpl_two_seed_bridges(c, r, bridge_a="developer", bridge_b="engine", visible=("platform",)),
})

# New difficulty ladder:
# L1-L2 = visible multicriteria only; L3 = one hidden hop + visible criterion;
# L4 = one hidden hop + two visible criteria; L5 = two hidden hops or strongest L4 fallback.
VG_TEMPLATE_PLAN_BY_LEVEL = {
    "L1": (
        "vg_direct_genre_platform_year",
    ),
    "L2": (
        "vg_direct_genre_platform_year",
        "vg_direct_developer_platform_year",
        "vg_direct_publisher_genre_year",
    ),
    "L3": (
        "vg_same_developer_as_seed_genre_year",
        "vg_same_publisher_as_seed_platform_year",
        "vg_same_series_as_seed_genre_year",
        "vg_same_engine_as_seed_platform_year",
        "vg_same_developer_as_seed_platform_year",
        "vg_same_publisher_as_seed_genre_year",
    ),
    "L4": (
        "vg_same_developer_as_seed_genre_platform_year",
        "vg_same_publisher_as_seed_genre_platform_year",
        "vg_same_series_as_seed_genre_platform_year",
        "vg_same_engine_as_seed_genre_platform_year",
        "vg_same_developer_as_seed_genre_publisher_year",
    ),
    "L5": (
        "vg_same_developer_and_publisher_as_seeds_genre_year",
        "vg_same_developer_and_publisher_as_seeds_platform_year",
        "vg_same_publisher_and_series_as_seeds_genre_year",
        "vg_same_developer_and_series_as_seeds_platform_year",
        "vg_same_developer_and_engine_as_seeds_platform_year",
        "vg_same_developer_as_seed_genre_platform_year",
        "vg_same_publisher_as_seed_genre_platform_year",
    ),
}


def _vg_hidden_bridge_keys(ex: BenchmarkExample) -> List[Tuple[str, str, str]]:
    cq = (ex.gold_collection_meta or {}).get("constraints_with_qids") or {}
    out: List[Tuple[str, str, str]] = []
    hv = cq.get("hidden_bridge_value")
    if isinstance(hv, dict) and hv.get("qid"):
        out.append((ex.complexity, str(hv.get("qid")), str(hv.get("label_en") or hv.get("en") or "")))
    hvs = cq.get("hidden_bridge_values")
    if isinstance(hvs, list):
        for x in hvs:
            if isinstance(x, dict) and x.get("qid"):
                out.append((ex.complexity, str(x.get("qid")), str(x.get("label_en") or x.get("en") or "")))
    return out


def _vg_gold_set_key(ex: BenchmarkExample) -> Tuple[str, ...]:
    return tuple(sorted(ex.gold_answer_qids))


def _vg_gold_jaccard(a: Sequence[str], b: Sequence[str]) -> float:
    sa, sb = set(a), set(b)
    if not sa or not sb:
        return 0.0
    return len(sa & sb) / len(sa | sb)


def _vg_diversity_reject_reason(
    ex: BenchmarkExample,
    *,
    seen_public_keys: set,
    seen_gold_sets: set,
    accepted_records: Sequence[BenchmarkExample],
    hidden_counts_global: Counter,
    hidden_counts_by_level: Counter,
) -> Optional[str]:
    if len(ex.gold_answer_qids) > VIDEOGAMES_MAX_ACCEPTED_GOLD:
        return "gold_count_over_max_accepted"
    if _vg_record_key(ex) in seen_public_keys:
        return "duplicate_public_constraints"
    gs = _vg_gold_set_key(ex)
    if gs in seen_gold_sets:
        return "duplicate_exact_gold_set"
    for prev in accepted_records:
        if ex.complexity == prev.complexity or ex.template_family == prev.template_family:
            if _vg_gold_jaccard(ex.gold_answer_qids, prev.gold_answer_qids) >= VIDEOGAMES_MAX_GOLD_JACCARD_OVERLAP:
                return f"near_duplicate_gold_set:{prev.id}"
    for level, qid, label in _vg_hidden_bridge_keys(ex):
        if hidden_counts_global[(qid, label)] >= VIDEOGAMES_MAX_SAME_HIDDEN_BRIDGE_GLOBAL:
            return f"hidden_bridge_overused_global:{label or qid}"
        if hidden_counts_by_level[(level, qid, label)] >= VIDEOGAMES_MAX_SAME_HIDDEN_BRIDGE_PER_LEVEL:
            return f"hidden_bridge_overused_level:{label or qid}"
    return None


def _vg_mark_diversity_accept(
    ex: BenchmarkExample,
    *,
    seen_public_keys: set,
    seen_gold_sets: set,
    hidden_counts_global: Counter,
    hidden_counts_by_level: Counter,
) -> None:
    seen_public_keys.add(_vg_record_key(ex))
    seen_gold_sets.add(_vg_gold_set_key(ex))
    for level, qid, label in _vg_hidden_bridge_keys(ex):
        hidden_counts_global[(qid, label)] += 1
        hidden_counts_by_level[(level, qid, label)] += 1


def _vg_validate_record_schema(ex: BenchmarkExample) -> None:
    d = asdict(ex)
    assert list(d.keys()) == VG_EXPECTED_JSON_KEYS, "JSON key order/structure diverged from BenchmarkExample"
    assert ex.domain == "videogames"
    assert ex.query_text_ru and ex.query_text_en
    assert ex.requested_count > 0
    assert len(ex.gold_answer_qids) == len(ex.gold_answer_labels_ru) == len(ex.gold_answer_labels_en)
    assert len(ex.gold_answer_qids) >= ex.requested_count
    assert len(ex.gold_answer_qids) <= VIDEOGAMES_MAX_ACCEPTED_GOLD
    assert len({_vg_norm_key(x) for x in ex.gold_answer_labels_en}) == len(ex.gold_answer_labels_en)
    assert len({_vg_norm_key(x) for x in ex.gold_answer_labels_ru}) == len(ex.gold_answer_labels_ru)
    assert all(_vg_is_good_public_label(x) for x in ex.gold_answer_labels_en)
    assert all(_vg_is_good_public_label(x) for x in ex.gold_answer_labels_ru)
    assert ex.sparql_query and ex.ask_validator_sparql
    assert _vg_public_constraints_are_clean(ex.constraints)
    assert isinstance(ex.gold_collection_meta, dict)
    assert isinstance(ex.local_validator, dict)
    assert ex.gold_collection_meta.get("constraints_with_qids"), "QID constraints must be stored in gold_collection_meta"


def _vg_validate_output_jsonl(path: Path, expected_count: Optional[int] = None) -> None:
    if not path.exists():
        raise AssertionError(f"Output JSONL was not created: {path}")
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            if not line.strip():
                continue
            rec = json.loads(line)
            assert list(rec.keys()) == VG_EXPECTED_JSON_KEYS, f"Bad key structure at line {line_no}"
            assert rec.get("domain") == "videogames", f"Bad domain at line {line_no}"
            assert rec.get("query_text_ru") and rec.get("query_text_en"), f"Missing query text at line {line_no}"
            assert len(rec.get("gold_answer_qids", [])) == len(rec.get("gold_answer_labels_ru", [])) == len(rec.get("gold_answer_labels_en", [])), f"Gold list length mismatch at line {line_no}"
            assert len(rec.get("gold_answer_qids", [])) >= int(rec.get("requested_count", 0)), f"Too few golds at line {line_no}"
            assert len(rec.get("gold_answer_qids", [])) <= VIDEOGAMES_MAX_ACCEPTED_GOLD, f"Too many golds at line {line_no}"
            assert len({_vg_norm_key(x) for x in rec.get("gold_answer_labels_en", [])}) == len(rec.get("gold_answer_labels_en", [])), f"Duplicate EN public label at line {line_no}"
            assert len({_vg_norm_key(x) for x in rec.get("gold_answer_labels_ru", [])}) == len(rec.get("gold_answer_labels_ru", [])), f"Duplicate RU public label at line {line_no}"
            assert all(_vg_is_good_public_label(x) for x in rec.get("gold_answer_labels_en", [])), f"Bad EN gold label at line {line_no}"
            assert all(_vg_is_good_public_label(x) for x in rec.get("gold_answer_labels_ru", [])), f"Bad RU gold label at line {line_no}"
            assert _vg_public_constraints_are_clean(rec.get("constraints", {})), f"Dirty public constraints at line {line_no}"
            rows.append(rec)
    if expected_count is not None:
        assert len(rows) == int(expected_count), f"Expected {expected_count} JSONL rows, got {len(rows)}"


def generate_videogames_dataset(
    target_plan: Optional[Dict[str, int]] = None,
    output_path: Path = VIDEOGAMES_OUTPUT_PATH,
    audit_path: Path = VIDEOGAMES_AUDIT_PATH,
    overwrite: bool = OVERWRITE_VIDEOGAMES_OUTPUT,
    seed: int = VIDEOGAMES_SEED,
    max_visible_attempts_per_level: int = 620,
) -> List[BenchmarkExample]:
    target_plan = dict(target_plan or VIDEOGAMES_TARGET_PLAN)
    rng = random.Random(seed)
    records: List[BenchmarkExample] = []
    skipped: List[Dict[str, Any]] = []
    seen_public_keys: set = set()
    seen_gold_sets: set = set()
    hidden_counts_global: Counter = Counter()
    hidden_counts_by_level: Counter = Counter()

    if overwrite and output_path.exists():
        output_path.unlink()

    total_target = sum(target_plan.values())
    overall_bar = tqdm(total=total_target, desc="videogames total") if tqdm is not None else None
    idx = 1

    try:
        for level, target in target_plan.items():
            if target <= 0:
                continue
            ok = 0
            attempts = 0
            level_bar = tqdm(total=target, desc=f"videogames {level}") if tqdm is not None else None
            while ok < target and attempts < max_visible_attempts_per_level:
                accepted: Optional[BenchmarkExample] = None
                for template_id in _vg_slot_try_plan(level, ok + attempts):
                    attempts += 1
                    try:
                        ex = generate_videogames_example(
                            level,
                            idx,
                            rng=rng,
                            max_attempts=32,
                            forced_template_id=template_id,
                        )
                    except Exception as e:
                        skipped.append({"complexity": level, "template_id": template_id, "reason": type(e).__name__, "message": str(e)[:500]})
                        if attempts >= max_visible_attempts_per_level:
                            break
                        continue
                    reason = _vg_diversity_reject_reason(
                        ex,
                        seen_public_keys=seen_public_keys,
                        seen_gold_sets=seen_gold_sets,
                        accepted_records=records,
                        hidden_counts_global=hidden_counts_global,
                        hidden_counts_by_level=hidden_counts_by_level,
                    )
                    if reason:
                        skipped.append({"complexity": level, "template_id": template_id, "reason": reason, "record_id": ex.id})
                        if attempts >= max_visible_attempts_per_level:
                            break
                        continue
                    accepted = ex
                    break
                if accepted is None:
                    continue
                _vg_validate_record_schema(accepted)
                _vg_mark_diversity_accept(
                    accepted,
                    seen_public_keys=seen_public_keys,
                    seen_gold_sets=seen_gold_sets,
                    hidden_counts_global=hidden_counts_global,
                    hidden_counts_by_level=hidden_counts_by_level,
                )
                records.append(accepted)
                _vg_append_jsonl(output_path, asdict(accepted))
                idx += 1
                ok += 1
                if level_bar is not None:
                    level_bar.update(1)
                    level_bar.set_postfix({"attempts": attempts, "gold": len(accepted.gold_answer_qids), "tpl": accepted.template_id})
                if overall_bar is not None:
                    overall_bar.update(1)
                    overall_bar.set_postfix({"level": level, "gold": len(accepted.gold_answer_qids)})
                _vg_write_json(VIDEOGAMES_CHECKPOINT_PATH, {"last_record": asdict(accepted), "audit": _vg_audit(records, skipped)})
            if level_bar is not None:
                level_bar.close()
            if ok < target:
                print(f"[WARN] videogames {level}: generated {ok}/{target} after {attempts} visible attempts")
            else:
                print(f"OK videogames {level}: generated {ok}/{target} after {attempts} visible attempts")
    finally:
        if overall_bar is not None:
            overall_bar.close()
        audit = _vg_audit(records, skipped)
        audit.update({
            "v6_quality_patch": True,
            "max_same_hidden_bridge_global": VIDEOGAMES_MAX_SAME_HIDDEN_BRIDGE_GLOBAL,
            "max_same_hidden_bridge_per_level": VIDEOGAMES_MAX_SAME_HIDDEN_BRIDGE_PER_LEVEL,
            "max_gold_jaccard_overlap": VIDEOGAMES_MAX_GOLD_JACCARD_OVERLAP,
            "gold_max_accepted_enforced": VIDEOGAMES_MAX_ACCEPTED_GOLD,
            "ambiguous_genres_removed_from_anchors": sorted(VG_AMBIGUOUS_DIRECT_GENRES),
            "template_plan": {k: list(v) for k, v in VG_TEMPLATE_PLAN_BY_LEVEL.items()},
        })
        _vg_write_json(audit_path, audit)
        print("saved:", output_path.resolve())
        print("audit:", audit_path.resolve())
        print("records:", len(records))
        print("counts:", dict(Counter(x.complexity for x in records)))
        print("families:", dict(Counter(x.template_family for x in records)))
        print("skipped:", len(skipped))

    if output_path.exists():
        _vg_validate_output_jsonl(output_path, expected_count=len(records))
    if VIDEOGAMES_STRICT_TARGET and len(records) != total_target:
        raise RuntimeError(
            f"videogames generation produced {len(records)}/{total_target} records. "
            f"See audit for skipped reasons: {audit_path}"
        )
    _vg_validate_output_jsonl(output_path, expected_count=total_target)
    return records


# === v7 prompt wording patch: explicit independent constraints ===
# This block changes ONLY human-facing query_text_ru/query_text_en wording.
# SPARQL, constraints, gold collection, filters, and template plan are intentionally unchanged.


def _vg_year_clause_ru(y1: Optional[int], y2: Optional[int]) -> str:
    if y1 is None or y2 is None:
        return ""
    return f"общая дата публикации игры — с {int(y1)} по {int(y2)} год включительно"


def _vg_year_clause_en(y1: Optional[int], y2: Optional[int]) -> str:
    if y1 is None or y2 is None:
        return ""
    return f"the game's overall publication date is from {int(y1)} to {int(y2)} inclusive"


def _vg_platform_phrase_ru(platform: Dict[str, str]) -> str:
    return f"среди платформ указана {platform['ru']}"


def _vg_platform_phrase_en(platform: Dict[str, str]) -> str:
    return f"{platform['en']} is listed among the platforms"


def _vg_genre_phrase_ru(genre: Dict[str, str]) -> str:
    return f"жанр — «{genre['ru']}»"


def _vg_genre_phrase_en(genre: Dict[str, str]) -> str:
    return f"the genre is {genre['en']}"


def _vg_developer_phrase_ru(developer: Dict[str, str]) -> str:
    return f"разработчик — {developer['ru']}"


def _vg_developer_phrase_en(developer: Dict[str, str]) -> str:
    return f"the developer is {developer['en']}"


def _vg_publisher_phrase_ru(publisher: Dict[str, str]) -> str:
    return f"издатель — {publisher['ru']}"


def _vg_publisher_phrase_en(publisher: Dict[str, str]) -> str:
    return f"the publisher is {publisher['en']}"


def _vg_series_phrase_ru(series: Dict[str, str]) -> str:
    return f"серия — «{series['ru']}»"


def _vg_series_phrase_en(series: Dict[str, str]) -> str:
    return f"the series is {series['en']}"


def _vg_make_query_text_ru(k: int, parts: Sequence[str]) -> str:
    k_ru = _vg_counted_video_games_ru(k)
    clean = [p.strip().rstrip('.') for p in parts if p and str(p).strip()]
    return f"Назови {k_ru}, у которых: " + "; ".join(clean) + "."


def _vg_make_query_text_en(k: int, parts: Sequence[str]) -> str:
    clean = [p.strip().rstrip('.') for p in parts if p and str(p).strip()]
    return f"Name {int(k)} video games where: " + "; ".join(clean) + "."


def _vg_tpl_direct_genre_platform_year(complexity: str, rng: random.Random) -> Dict[str, Any]:
    genre = _vg_pick("genre", rng)
    platform = _vg_pick("platform", rng)
    y1, y2 = _vg_year_window(complexity, rng)
    where = [
        f"?item wdt:P136 wd:{genre['qid']} .",
        f"?item wdt:P400 wd:{platform['qid']} .",
    ] + _vg_year_where(y1, y2)
    k = _vg_requested(complexity)
    return {
        "template_id": "vg_direct_genre_platform_year",
        "template_family": "direct_visible_multi_criteria",
        "query_text_ru": _vg_make_query_text_ru(k, [_vg_genre_phrase_ru(genre), _vg_platform_phrase_ru(platform), _vg_year_clause_ru(y1, y2)]),
        "query_text_en": _vg_make_query_text_en(k, [_vg_genre_phrase_en(genre), _vg_platform_phrase_en(platform), _vg_year_clause_en(y1, y2)]),
        "constraints": _vg_public_constraints(genre=genre, platform=platform, release_year_from=y1, release_year_to=y2),
        "qid_constraints": _vg_constraint_qids(genre=genre, platform=platform, release_year_from=y1, release_year_to=y2),
        "where_lines": where,
        "requested_count": k,
    }


def _vg_tpl_direct_developer_platform_year(complexity: str, rng: random.Random) -> Dict[str, Any]:
    developer = _vg_pick("developer", rng)
    platform = _vg_pick("platform", rng)
    y1, y2 = _vg_year_window(complexity, rng)
    where = [
        f"?item wdt:P178 wd:{developer['qid']} .",
        f"?item wdt:P400 wd:{platform['qid']} .",
    ] + _vg_year_where(y1, y2)
    k = _vg_requested(complexity)
    return {
        "template_id": "vg_direct_developer_platform_year",
        "template_family": "direct_visible_multi_criteria",
        "query_text_ru": _vg_make_query_text_ru(k, [_vg_developer_phrase_ru(developer), _vg_platform_phrase_ru(platform), _vg_year_clause_ru(y1, y2)]),
        "query_text_en": _vg_make_query_text_en(k, [_vg_developer_phrase_en(developer), _vg_platform_phrase_en(platform), _vg_year_clause_en(y1, y2)]),
        "constraints": _vg_public_constraints(developer=developer, platform=platform, release_year_from=y1, release_year_to=y2),
        "qid_constraints": _vg_constraint_qids(developer=developer, platform=platform, release_year_from=y1, release_year_to=y2),
        "where_lines": where,
        "requested_count": k,
    }


def _vg_tpl_direct_publisher_genre_year(complexity: str, rng: random.Random) -> Dict[str, Any]:
    publisher = _vg_pick("publisher", rng)
    genre = _vg_pick("genre", rng)
    y1, y2 = _vg_year_window(complexity, rng)
    where = [
        f"?item wdt:P123 wd:{publisher['qid']} .",
        f"?item wdt:P136 wd:{genre['qid']} .",
    ] + _vg_year_where(y1, y2)
    k = _vg_requested(complexity)
    return {
        "template_id": "vg_direct_publisher_genre_year",
        "template_family": "direct_visible_multi_criteria",
        "query_text_ru": _vg_make_query_text_ru(k, [_vg_genre_phrase_ru(genre), _vg_publisher_phrase_ru(publisher), _vg_year_clause_ru(y1, y2)]),
        "query_text_en": _vg_make_query_text_en(k, [_vg_genre_phrase_en(genre), _vg_publisher_phrase_en(publisher), _vg_year_clause_en(y1, y2)]),
        "constraints": _vg_public_constraints(publisher=publisher, genre=genre, release_year_from=y1, release_year_to=y2),
        "qid_constraints": _vg_constraint_qids(publisher=publisher, genre=genre, release_year_from=y1, release_year_to=y2),
        "where_lines": where,
        "requested_count": k,
    }


def _vg_tpl_direct_series_platform_year(complexity: str, rng: random.Random) -> Dict[str, Any]:
    series = _vg_pick("series", rng)
    platform = _vg_pick("platform", rng)
    y1, y2 = _vg_year_window(complexity, rng)
    where = [
        f"?item wdt:P179 wd:{series['qid']} .",
        f"?item wdt:P400 wd:{platform['qid']} .",
    ] + _vg_year_where(y1, y2)
    k = _vg_requested(complexity)
    return {
        "template_id": "vg_direct_series_platform_year",
        "template_family": "direct_visible_multi_criteria",
        "query_text_ru": _vg_make_query_text_ru(k, [_vg_series_phrase_ru(series), _vg_platform_phrase_ru(platform), _vg_year_clause_ru(y1, y2)]),
        "query_text_en": _vg_make_query_text_en(k, [_vg_series_phrase_en(series), _vg_platform_phrase_en(platform), _vg_year_clause_en(y1, y2)]),
        "constraints": _vg_public_constraints(series=series, platform=platform, release_year_from=y1, release_year_to=y2),
        "qid_constraints": _vg_constraint_qids(series=series, platform=platform, release_year_from=y1, release_year_to=y2),
        "where_lines": where,
        "requested_count": k,
    }


def _vg_bridge_phrase_ru(bridge: str, seed: Dict[str, str]) -> str:
    return {
        "developer": f"разработчик совпадает с разработчиком игры «{seed['ru']}»",
        "publisher": f"издатель совпадает с издателем игры «{seed['ru']}»",
        "series": f"серия совпадает с серией игры «{seed['ru']}»",
        "engine": f"игровой движок совпадает с игровым движком игры «{seed['ru']}»",
    }[bridge]


def _vg_bridge_phrase_en(bridge: str, seed: Dict[str, str]) -> str:
    return {
        "developer": f"the developer matches the developer of {seed['en']}",
        "publisher": f"the publisher matches the publisher of {seed['en']}",
        "series": f"the series matches the series of {seed['en']}",
        "engine": f"the game engine matches the game engine of {seed['en']}",
    }[bridge]


def _vg_tpl_same_bridge_as_seed(complexity: str, rng: random.Random, *, bridge: str, visible: Sequence[str] = ()) -> Dict[str, Any]:
    seed_pack = _vg_pick_seed(bridge, rng)
    seed = seed_pack["seed"]
    bridge_value = seed_pack["bridge_value"]
    pid = VG_BRIDGES[bridge]["pid"]
    public_key = VG_BRIDGES[bridge]["public_key"]
    y1, y2 = _vg_year_window(complexity, rng)
    where = [
        f"BIND(wd:{seed['qid']} AS ?seed) .",
        f"?seed wdt:{pid} ?bridgeValue .",
        f"?item wdt:{pid} ?bridgeValue .",
        "FILTER(?item != ?seed) .",
    ]
    public_kwargs: Dict[str, Any] = {public_key: seed, "release_year_from": y1, "release_year_to": y2}
    qid_kwargs: Dict[str, Any] = {public_key: seed, "hidden_bridge_value": bridge_value, "release_year_from": y1, "release_year_to": y2}
    ru_parts: List[str] = [_vg_bridge_phrase_ru(bridge, seed)]
    en_parts: List[str] = [_vg_bridge_phrase_en(bridge, seed)]

    if "genre" in visible:
        genre = _vg_pick_visible_for_bridge(bridge, bridge_value, "genre", rng)
        where.append(f"?item wdt:P136 wd:{genre['qid']} .")
        public_kwargs["genre"] = genre
        qid_kwargs["genre"] = genre
        ru_parts.append(_vg_genre_phrase_ru(genre))
        en_parts.append(_vg_genre_phrase_en(genre))
    if "platform" in visible:
        platform = _vg_pick_visible_for_bridge(bridge, bridge_value, "platform", rng)
        where.append(f"?item wdt:P400 wd:{platform['qid']} .")
        public_kwargs["platform"] = platform
        qid_kwargs["platform"] = platform
        ru_parts.append(_vg_platform_phrase_ru(platform))
        en_parts.append(_vg_platform_phrase_en(platform))
    if "publisher" in visible:
        publisher = _vg_pick_visible_for_bridge(bridge, bridge_value, "publisher", rng)
        where.append(f"?item wdt:P123 wd:{publisher['qid']} .")
        public_kwargs["publisher"] = publisher
        qid_kwargs["publisher"] = publisher
        ru_parts.append(_vg_publisher_phrase_ru(publisher))
        en_parts.append(_vg_publisher_phrase_en(publisher))
    if "developer" in visible:
        developer = _vg_pick_visible_for_bridge(bridge, bridge_value, "developer", rng)
        where.append(f"?item wdt:P178 wd:{developer['qid']} .")
        public_kwargs["developer"] = developer
        qid_kwargs["developer"] = developer
        ru_parts.append(_vg_developer_phrase_ru(developer))
        en_parts.append(_vg_developer_phrase_en(developer))

    where += _vg_year_where(y1, y2)
    ru_parts.append(_vg_year_clause_ru(y1, y2))
    en_parts.append(_vg_year_clause_en(y1, y2))
    visible_suffix = "_" + "_".join(visible) if visible else ""
    template_id = f"vg_same_{bridge}_as_seed{visible_suffix}_year"
    k = _vg_requested(complexity)
    return {
        "template_id": template_id,
        "template_family": "hidden_seed_bridge",
        "query_text_ru": _vg_make_query_text_ru(k, ru_parts),
        "query_text_en": _vg_make_query_text_en(k, en_parts),
        "constraints": _vg_public_constraints(**public_kwargs),
        "qid_constraints": _vg_constraint_qids(**qid_kwargs),
        "where_lines": where,
        "requested_count": k,
    }


def _vg_tpl_two_seed_bridges(complexity: str, rng: random.Random, *, bridge_a: str, bridge_b: str, visible: Sequence[str] = ()) -> Dict[str, Any]:
    if bridge_a == bridge_b:
        raise ValueError("Two-seed bridge template requires different bridge types")
    pack_a = _vg_pick_seed(bridge_a, rng)
    seed_a = pack_a["seed"]
    value_a = pack_a["bridge_value"]
    value_b = _vg_pick_compatible_value(bridge_a, value_a, bridge_b, rng)
    seed_b_df = _vg_seed_pool_for_bridge_value(bridge_b, value_b)
    if seed_b_df is None or len(seed_b_df) == 0:
        raise ValueError(f"No seed for secondary bridge {bridge_b}={value_b.get('en')}")
    seed_b_row = seed_b_df.sample(1, random_state=rng.randint(0, 10**9)).iloc[0]
    seed_b = {
        "qid": seed_b_row.seed_qid,
        "en": _vg_clean_label(seed_b_row.seed_label_en),
        "ru": _vg_display_ru(seed_b_row.seed_label_en, seed_b_row.seed_label_ru),
    }
    y1, y2 = _vg_year_window(complexity, rng)
    pid_a = VG_BRIDGES[bridge_a]["pid"]
    pid_b = VG_BRIDGES[bridge_b]["pid"]
    key_a = VG_BRIDGES[bridge_a]["public_key"]
    key_b = VG_BRIDGES[bridge_b]["public_key"]
    where = [
        f"BIND(wd:{seed_a['qid']} AS ?seedA) .",
        f"?seedA wdt:{pid_a} ?bridgeValueA .",
        f"?item wdt:{pid_a} ?bridgeValueA .",
        f"BIND(wd:{seed_b['qid']} AS ?seedB) .",
        f"?seedB wdt:{pid_b} ?bridgeValueB .",
        f"?item wdt:{pid_b} ?bridgeValueB .",
        "FILTER(?item != ?seedA && ?item != ?seedB) .",
    ]
    public_kwargs: Dict[str, Any] = {key_a: seed_a, key_b: seed_b, "release_year_from": y1, "release_year_to": y2}
    qid_kwargs: Dict[str, Any] = {
        key_a: seed_a,
        key_b: seed_b,
        "hidden_bridge_values": [
            {"bridge": bridge_a, **value_a},
            {"bridge": bridge_b, **value_b},
        ],
        "release_year_from": y1,
        "release_year_to": y2,
    }
    ru_parts: List[str] = [_vg_bridge_phrase_ru(bridge_a, seed_a), _vg_bridge_phrase_ru(bridge_b, seed_b)]
    en_parts: List[str] = [_vg_bridge_phrase_en(bridge_a, seed_a), _vg_bridge_phrase_en(bridge_b, seed_b)]
    for prop in visible:
        val = _vg_pick_compatible_value(bridge_a, value_a, prop, rng)
        pid = VG_PROPERTY_PIDS[prop]
        where.append(f"?item wdt:{pid} wd:{val['qid']} .")
        public_kwargs[prop] = val
        qid_kwargs[prop] = val
        if prop == "genre":
            ru_parts.append(_vg_genre_phrase_ru(val))
            en_parts.append(_vg_genre_phrase_en(val))
        elif prop == "platform":
            ru_parts.append(_vg_platform_phrase_ru(val))
            en_parts.append(_vg_platform_phrase_en(val))
        elif prop == "publisher":
            ru_parts.append(_vg_publisher_phrase_ru(val))
            en_parts.append(_vg_publisher_phrase_en(val))
        elif prop == "developer":
            ru_parts.append(_vg_developer_phrase_ru(val))
            en_parts.append(_vg_developer_phrase_en(val))
    where += _vg_year_where(y1, y2)
    ru_parts.append(_vg_year_clause_ru(y1, y2))
    en_parts.append(_vg_year_clause_en(y1, y2))
    template_id = f"vg_same_{bridge_a}_and_{bridge_b}_as_seeds"
    if visible:
        template_id += "_" + "_".join(visible)
    template_id += "_year"
    k = _vg_requested(complexity)
    return {
        "template_id": template_id,
        "template_family": "double_hidden_seed_bridge",
        "query_text_ru": _vg_make_query_text_ru(k, ru_parts),
        "query_text_en": _vg_make_query_text_en(k, en_parts),
        "constraints": _vg_public_constraints(**public_kwargs),
        "qid_constraints": _vg_constraint_qids(**qid_kwargs),
        "where_lines": where,
        "requested_count": k,
    }

# Re-register builders so every generated record uses the explicit v7 wording.
VG_TEMPLATE_BUILDERS.update({
    "vg_direct_genre_platform_year": _vg_tpl_direct_genre_platform_year,
    "vg_direct_developer_platform_year": _vg_tpl_direct_developer_platform_year,
    "vg_direct_publisher_genre_year": _vg_tpl_direct_publisher_genre_year,
    "vg_direct_series_platform_year": _vg_tpl_direct_series_platform_year,
    "vg_same_developer_as_seed_genre_year": lambda c, r: _vg_tpl_same_bridge_as_seed(c, r, bridge="developer", visible=("genre",)),
    "vg_same_publisher_as_seed_platform_year": lambda c, r: _vg_tpl_same_bridge_as_seed(c, r, bridge="publisher", visible=("platform",)),
    "vg_same_series_as_seed_genre_year": lambda c, r: _vg_tpl_same_bridge_as_seed(c, r, bridge="series", visible=("genre",)),
    "vg_same_engine_as_seed_platform_year": lambda c, r: _vg_tpl_same_bridge_as_seed(c, r, bridge="engine", visible=("platform",)),
    "vg_same_developer_as_seed_platform_year": lambda c, r: _vg_tpl_same_bridge_as_seed(c, r, bridge="developer", visible=("platform",)),
    "vg_same_publisher_as_seed_genre_year": lambda c, r: _vg_tpl_same_bridge_as_seed(c, r, bridge="publisher", visible=("genre",)),
    "vg_same_developer_as_seed_genre_platform_year": lambda c, r: _vg_tpl_same_bridge_as_seed(c, r, bridge="developer", visible=("genre", "platform")),
    "vg_same_publisher_as_seed_genre_platform_year": lambda c, r: _vg_tpl_same_bridge_as_seed(c, r, bridge="publisher", visible=("genre", "platform")),
    "vg_same_series_as_seed_genre_platform_year": lambda c, r: _vg_tpl_same_bridge_as_seed(c, r, bridge="series", visible=("genre", "platform")),
    "vg_same_engine_as_seed_genre_platform_year": lambda c, r: _vg_tpl_same_bridge_as_seed(c, r, bridge="engine", visible=("genre", "platform")),
    "vg_same_developer_as_seed_genre_publisher_year": lambda c, r: _vg_tpl_same_bridge_as_seed(c, r, bridge="developer", visible=("genre", "publisher")),
    "vg_same_developer_and_publisher_as_seeds_genre_year": lambda c, r: _vg_tpl_two_seed_bridges(c, r, bridge_a="developer", bridge_b="publisher", visible=("genre",)),
    "vg_same_developer_and_publisher_as_seeds_platform_year": lambda c, r: _vg_tpl_two_seed_bridges(c, r, bridge_a="developer", bridge_b="publisher", visible=("platform",)),
    "vg_same_publisher_and_series_as_seeds_genre_year": lambda c, r: _vg_tpl_two_seed_bridges(c, r, bridge_a="publisher", bridge_b="series", visible=("genre",)),
    "vg_same_developer_and_series_as_seeds_platform_year": lambda c, r: _vg_tpl_two_seed_bridges(c, r, bridge_a="developer", bridge_b="series", visible=("platform",)),
    "vg_same_developer_and_engine_as_seeds_platform_year": lambda c, r: _vg_tpl_two_seed_bridges(c, r, bridge_a="developer", bridge_b="engine", visible=("platform",)),
})



# === v8 speed/stability patch ===
# The v6/v7 quality patch made L3+ substantially stricter: <=100 complete golds,
# DLC/bundle filtering, duplicate/near-duplicate filtering, and hidden-bridge quotas.
# Without a cheap preflight, many bad/random candidates still paid the cost of a full
# LIMIT 101 WDQS gold query and then got rejected.  This block keeps the same dataset
# semantics but reduces wasted WDQS work.

VIDEOGAMES_QUICK_PROBE_LIMIT = int(globals().get("VIDEOGAMES_V8_QUICK_PROBE_LIMIT", 16))
VIDEOGAMES_TEMPLATE_INNER_ATTEMPTS = int(globals().get("VIDEOGAMES_V8_TEMPLATE_INNER_ATTEMPTS", 7))
VIDEOGAMES_SLOT_TRY_WIDTH = int(globals().get("VIDEOGAMES_V8_SLOT_TRY_WIDTH", 5))
# Candidate queries should fail fast.  Accepted/full-gold candidates are cached, so
# two retries are enough; hanging for 55s*3 per random candidate is the main slowdown.
VIDEOGAMES_WDQS_FAST_TIMEOUT_SECONDS = int(globals().get("VIDEOGAMES_V8_WDQS_FAST_TIMEOUT_SECONDS", 32))
VIDEOGAMES_WDQS_FAST_MAX_RETRIES = int(globals().get("VIDEOGAMES_V8_WDQS_FAST_MAX_RETRIES", 2))


def _vg_slot_try_plan(level: str, accepted_index_within_level: int) -> List[str]:
    plan = list(VG_TEMPLATE_PLAN_BY_LEVEL.get(level, ()))
    if not plan:
        raise ValueError(f"No template plan for videogames {level}")
    start = accepted_index_within_level % len(plan)
    rotated = plan[start:] + plan[:start]
    return rotated[:max(1, int(VIDEOGAMES_SLOT_TRY_WIDTH))]


def _vg_spec_public_key(complexity: str, template_id: str, constraints: Dict[str, Any]) -> Tuple[str, str, str]:
    return (str(complexity), str(template_id), json.dumps(constraints, ensure_ascii=False, sort_keys=True))


def _vg_hidden_bridge_keys_from_qid_constraints(level: str, cq: Dict[str, Any]) -> List[Tuple[str, str, str]]:
    out: List[Tuple[str, str, str]] = []
    hv = (cq or {}).get("hidden_bridge_value")
    if isinstance(hv, dict) and hv.get("qid"):
        out.append((str(level), str(hv.get("qid")), str(hv.get("label_en") or hv.get("en") or "")))
    hvs = (cq or {}).get("hidden_bridge_values")
    if isinstance(hvs, list):
        for x in hvs:
            if isinstance(x, dict) and x.get("qid"):
                out.append((str(level), str(x.get("qid")), str(x.get("label_en") or x.get("en") or "")))
    return out


def _vg_pre_gold_reject_reason(
    *,
    complexity: str,
    template_id: str,
    constraints: Dict[str, Any],
    qid_constraints: Dict[str, Any],
    seen_public_keys: set,
    hidden_counts_global: Counter,
    hidden_counts_by_level: Counter,
) -> Optional[str]:
    if _vg_spec_public_key(complexity, template_id, constraints) in seen_public_keys:
        return "duplicate_public_constraints_pre_gold"
    for level, qid, label in _vg_hidden_bridge_keys_from_qid_constraints(complexity, qid_constraints):
        if hidden_counts_global[(qid, label)] >= VIDEOGAMES_MAX_SAME_HIDDEN_BRIDGE_GLOBAL:
            return f"hidden_bridge_overused_global_pre_gold:{label or qid}"
        if hidden_counts_by_level[(level, qid, label)] >= VIDEOGAMES_MAX_SAME_HIDDEN_BRIDGE_PER_LEVEL:
            return f"hidden_bridge_overused_level_pre_gold:{label or qid}"
    return None


def _vg_build_record(
    *,
    idx: int,
    complexity: str,
    template_id: str,
    template_family: str,
    query_text_ru: str,
    query_text_en: str,
    constraints: Dict[str, Any],
    qid_constraints: Dict[str, Any],
    where_lines: Sequence[str],
    requested_count: int,
) -> Optional[BenchmarkExample]:
    # Cheap preflight: reject too-narrow random candidates using a small LIMIT query.
    # We do NOT accept from this probe; it only prevents full LIMIT-101 gold queries
    # for candidates that obviously cannot provide enough answers after label cleanup.
    quick_limit = max(int(requested_count) * 3, int(VIDEOGAMES_QUICK_PROBE_LIMIT))
    quick_limit = min(quick_limit, int(VIDEOGAMES_PROBE_LIMIT))
    _, quick_gold, quick_meta = _vg_collect_gold(where_lines, limit=quick_limit)
    if len(quick_gold) < int(requested_count):
        return None

    # Full probe: keeps the v6 quality semantics: complete golds, <=100 accepted,
    # same SPARQL and ASK validator as before.
    sparql, gold, meta = _vg_collect_gold(where_lines, limit=VIDEOGAMES_PROBE_LIMIT)
    if len(gold) < requested_count:
        return None
    if VIDEOGAMES_SKIP_TRUNCATED_GOLD and meta["gold_may_be_incomplete_due_to_wdqs_limit"]:
        return None

    meta.update({
        "template_id": template_id,
        "template_family": template_family,
        "constraints_with_qids": qid_constraints,
        "v8_speed_patch": {
            "quick_probe_limit": int(quick_limit),
            "quick_gold_returned": int(len(quick_gold)),
            "quick_rows_returned_by_wdqs": int(quick_meta.get("rows_returned_by_wdqs", 0)),
        },
    })
    ask = _vg_ask_validator(where_lines)

    return BenchmarkExample(
        id=f"videogames_{complexity.lower()}_{idx:04d}",
        domain="videogames",
        complexity=complexity,
        query_text_ru=query_text_ru,
        constraints=constraints,
        requested_count=int(requested_count),
        gold_answer_qids=[x["qid"] for x in gold],
        gold_answer_labels_ru=[x["label_ru"] for x in gold],
        sparql_query=sparql,
        created_at=utc_now_z(),
        query_text_en=query_text_en,
        gold_answer_labels_en=[x["label_en"] for x in gold],
        is_advanced=(complexity in {"L4", "L5"}),
        template_id=template_id,
        template_family=template_family,
        gold_truncated=bool(meta["gold_may_be_incomplete_due_to_wdqs_limit"] or meta["gold_truncated_by_local_limit"]),
        ask_validator_sparql=ask,
        local_validator={
            "type": "none_wdqs_only",
            "source": "Wikidata Query Service",
            "applies_after": "ask_validator_sparql",
            "filters": constraints,
            "label_matching_used": False,
            "note": "All video-game constraints for this task are represented in the WDQS ASK validator; no external local validator is required.",
        },
        gold_collection_meta=meta,
    )


def _vg_generate_candidate_from_template(
    *,
    complexity: str,
    idx: int,
    template_id: str,
    rng: random.Random,
    seen_public_keys: set,
    hidden_counts_global: Counter,
    hidden_counts_by_level: Counter,
    skipped: List[Dict[str, Any]],
) -> Optional[BenchmarkExample]:
    builder = VG_TEMPLATE_BUILDERS[template_id]
    for local_attempt in range(max(1, int(VIDEOGAMES_TEMPLATE_INNER_ATTEMPTS))):
        try:
            spec = builder(complexity, rng)
            pre_reason = _vg_pre_gold_reject_reason(
                complexity=complexity,
                template_id=spec["template_id"],
                constraints=spec["constraints"],
                qid_constraints=spec["qid_constraints"],
                seen_public_keys=seen_public_keys,
                hidden_counts_global=hidden_counts_global,
                hidden_counts_by_level=hidden_counts_by_level,
            )
            if pre_reason:
                skipped.append({"complexity": complexity, "template_id": spec.get("template_id", template_id), "reason": pre_reason})
                continue
            ex = _vg_build_record(
                idx=idx,
                complexity=complexity,
                template_id=spec["template_id"],
                template_family=spec["template_family"],
                query_text_ru=spec["query_text_ru"],
                query_text_en=spec["query_text_en"],
                constraints=spec["constraints"],
                qid_constraints=spec["qid_constraints"],
                where_lines=spec["where_lines"],
                requested_count=spec["requested_count"],
            )
            if ex is not None and len(ex.gold_answer_qids) >= ex.requested_count:
                return ex
            skipped.append({"complexity": complexity, "template_id": spec.get("template_id", template_id), "reason": "insufficient_or_truncated_gold_fast_probe"})
        except Exception as e:
            skipped.append({"complexity": complexity, "template_id": template_id, "reason": type(e).__name__, "message": str(e)[:500]})
            continue
    return None


def generate_videogames_dataset(
    target_plan: Optional[Dict[str, int]] = None,
    output_path: Path = VIDEOGAMES_OUTPUT_PATH,
    audit_path: Path = VIDEOGAMES_AUDIT_PATH,
    overwrite: bool = OVERWRITE_VIDEOGAMES_OUTPUT,
    seed: int = VIDEOGAMES_SEED,
    max_visible_attempts_per_level: int = 620,
) -> List[BenchmarkExample]:
    target_plan = dict(target_plan or VIDEOGAMES_TARGET_PLAN)
    rng = random.Random(seed)
    records: List[BenchmarkExample] = []
    skipped: List[Dict[str, Any]] = []
    seen_public_keys: set = set()
    seen_gold_sets: set = set()
    hidden_counts_global: Counter = Counter()
    hidden_counts_by_level: Counter = Counter()

    if overwrite and output_path.exists():
        output_path.unlink()

    total_target = sum(target_plan.values())
    overall_bar = tqdm(total=total_target, desc="videogames total") if tqdm is not None else None
    idx = 1
    started_at = time.time()

    try:
        for level, target in target_plan.items():
            if target <= 0:
                continue
            ok = 0
            attempts = 0
            level_bar = tqdm(total=target, desc=f"videogames {level}") if tqdm is not None else None
            while ok < target and attempts < max_visible_attempts_per_level:
                accepted: Optional[BenchmarkExample] = None
                for template_id in _vg_slot_try_plan(level, ok + attempts):
                    attempts += 1
                    with _vg_wdqs_fail_fast_context():
                        ex = _vg_generate_candidate_from_template(
                            complexity=level,
                            idx=idx,
                            template_id=template_id,
                            rng=rng,
                            seen_public_keys=seen_public_keys,
                            hidden_counts_global=hidden_counts_global,
                            hidden_counts_by_level=hidden_counts_by_level,
                            skipped=skipped,
                        )
                    if ex is None:
                        if attempts >= max_visible_attempts_per_level:
                            break
                        continue
                    reason = _vg_diversity_reject_reason(
                        ex,
                        seen_public_keys=seen_public_keys,
                        seen_gold_sets=seen_gold_sets,
                        accepted_records=records,
                        hidden_counts_global=hidden_counts_global,
                        hidden_counts_by_level=hidden_counts_by_level,
                    )
                    if reason:
                        skipped.append({"complexity": level, "template_id": template_id, "reason": reason, "record_id": ex.id})
                        if attempts >= max_visible_attempts_per_level:
                            break
                        continue
                    accepted = ex
                    break
                if accepted is None:
                    continue
                _vg_validate_record_schema(accepted)
                _vg_mark_diversity_accept(
                    accepted,
                    seen_public_keys=seen_public_keys,
                    seen_gold_sets=seen_gold_sets,
                    hidden_counts_global=hidden_counts_global,
                    hidden_counts_by_level=hidden_counts_by_level,
                )
                records.append(accepted)
                _vg_append_jsonl(output_path, asdict(accepted))
                idx += 1
                ok += 1
                if level_bar is not None:
                    level_bar.update(1)
                    level_bar.set_postfix({"attempts": attempts, "gold": len(accepted.gold_answer_qids), "tpl": accepted.template_id})
                if overall_bar is not None:
                    overall_bar.update(1)
                    overall_bar.set_postfix({"level": level, "gold": len(accepted.gold_answer_qids)})
                _vg_write_json(VIDEOGAMES_CHECKPOINT_PATH, {"last_record": asdict(accepted), "audit": _vg_audit(records, skipped)})
            if level_bar is not None:
                level_bar.close()
            if ok < target:
                print(f"[WARN] videogames {level}: generated {ok}/{target} after {attempts} visible attempts")
            else:
                print(f"OK videogames {level}: generated {ok}/{target} after {attempts} visible attempts")
    finally:
        if overall_bar is not None:
            overall_bar.close()
        audit = _vg_audit(records, skipped)
        audit.update({
            "v8_speed_patch": True,
            "elapsed_seconds": round(time.time() - started_at, 2),
            "quick_probe_limit": VIDEOGAMES_QUICK_PROBE_LIMIT,
            "template_inner_attempts": VIDEOGAMES_TEMPLATE_INNER_ATTEMPTS,
            "slot_try_width": VIDEOGAMES_SLOT_TRY_WIDTH,
            "wdqs_fast_timeout_seconds": VIDEOGAMES_WDQS_FAST_TIMEOUT_SECONDS,
            "wdqs_fast_max_retries": VIDEOGAMES_WDQS_FAST_MAX_RETRIES,
            "max_same_hidden_bridge_global": VIDEOGAMES_MAX_SAME_HIDDEN_BRIDGE_GLOBAL,
            "max_same_hidden_bridge_per_level": VIDEOGAMES_MAX_SAME_HIDDEN_BRIDGE_PER_LEVEL,
            "max_gold_jaccard_overlap": VIDEOGAMES_MAX_GOLD_JACCARD_OVERLAP,
            "gold_max_accepted_enforced": VIDEOGAMES_MAX_ACCEPTED_GOLD,
            "ambiguous_genres_removed_from_anchors": sorted(VG_AMBIGUOUS_DIRECT_GENRES),
            "template_plan": {k: list(v) for k, v in VG_TEMPLATE_PLAN_BY_LEVEL.items()},
        })
        _vg_write_json(audit_path, audit)
        print("saved:", output_path.resolve())
        print("audit:", audit_path.resolve())
        print("records:", len(records))
        print("counts:", dict(Counter(x.complexity for x in records)))
        print("families:", dict(Counter(x.template_family for x in records)))
        print("skipped:", len(skipped))

    if output_path.exists():
        _vg_validate_output_jsonl(output_path, expected_count=len(records))
    if VIDEOGAMES_STRICT_TARGET and len(records) != total_target:
        raise RuntimeError(
            f"videogames generation produced {len(records)}/{total_target} records. "
            f"See audit for skipped reasons: {audit_path}"
        )
    _vg_validate_output_jsonl(output_path, expected_count=total_target)
    return records


# === v9 level-fill + diagnosability patch ===
# v8 did not actually "hang" at 54/130: it exhausted the L3 attempt budget
# (24/35 L3 after 620 visible attempts) and silently moved to L4.  For this
# domain we need exact per-level targets, so v9 must never skip an underfilled
# level.  It also broadens the L3-L5 template plan and relaxes only the most
# over-aggressive diversity guards while keeping the important quality rules:
# complete gold <=100, no DLC/bundle/collection labels, EN+RU gold-label dedupe,
# exact-gold-set duplicate ban, and explicit independent platform/year wording.

VIDEOGAMES_V9_PATCH = True
VIDEOGAMES_STRICT_LEVEL_TARGET = bool(globals().get("VIDEOGAMES_V9_STRICT_LEVEL_TARGET", True))
VIDEOGAMES_HEARTBEAT_EVERY_ATTEMPTS = int(globals().get("VIDEOGAMES_V9_HEARTBEAT_EVERY_ATTEMPTS", 75))
VIDEOGAMES_SLOT_TRY_WIDTH = int(globals().get("VIDEOGAMES_V9_SLOT_TRY_WIDTH", 99))
VIDEOGAMES_TEMPLATE_INNER_ATTEMPTS = int(globals().get("VIDEOGAMES_V9_TEMPLATE_INNER_ATTEMPTS", 6))

# v8 values (3 global / 2 per level / 0.88 Jaccard) were too strict for 100 hard
# L3-L5 records and caused premature pool exhaustion. Exact duplicates are still
# blocked; this only allows distinct questions with partially overlapping golds.
VIDEOGAMES_MAX_SAME_HIDDEN_BRIDGE_GLOBAL = int(globals().get("VIDEOGAMES_V9_MAX_SAME_HIDDEN_BRIDGE_GLOBAL", 5))
VIDEOGAMES_MAX_SAME_HIDDEN_BRIDGE_PER_LEVEL = int(globals().get("VIDEOGAMES_V9_MAX_SAME_HIDDEN_BRIDGE_PER_LEVEL", 3))
VIDEOGAMES_MAX_GOLD_JACCARD_OVERLAP = float(globals().get("VIDEOGAMES_V9_MAX_GOLD_JACCARD_OVERLAP", 0.95))

VIDEOGAMES_LEVEL_ATTEMPT_BUDGET = dict(globals().get("VIDEOGAMES_V9_LEVEL_ATTEMPT_BUDGET", {
    "L1": 220,
    "L2": 420,
    "L3": 1800,
    "L4": 2400,
    "L5": 2800,
}))

# L3 must be mostly true hidden-hop tasks, but it needs a broad enough pattern
# inventory to reach 35 under <=100-gold and duplicate filters. Direct templates
# are kept as last-resort L3 filler only. L4/L5 get more multihop options.
VG_TEMPLATE_PLAN_BY_LEVEL = {
    "L1": (
        "vg_direct_genre_platform_year",
    ),
    "L2": (
        "vg_direct_genre_platform_year",
        "vg_direct_developer_platform_year",
        "vg_direct_publisher_genre_year",
    ),
    "L3": (
        "vg_same_developer_as_seed_genre_year",
        "vg_same_publisher_as_seed_platform_year",
        "vg_same_series_as_seed_genre_year",
        "vg_same_engine_as_seed_platform_year",
        "vg_same_developer_as_seed_platform_year",
        "vg_same_publisher_as_seed_genre_year",
        "vg_same_series_as_seed_year",
        "vg_same_engine_as_seed_year",
        "vg_same_developer_as_seed_year",
        "vg_same_publisher_as_seed_year",
        "vg_direct_publisher_genre_year",
        "vg_direct_developer_platform_year",
        "vg_direct_genre_platform_year",
    ),
    "L4": (
        "vg_same_developer_as_seed_genre_platform_year",
        "vg_same_publisher_as_seed_genre_platform_year",
        "vg_same_series_as_seed_genre_platform_year",
        "vg_same_engine_as_seed_genre_platform_year",
        "vg_same_developer_as_seed_genre_publisher_year",
        "vg_same_developer_as_seed_genre_year",
        "vg_same_publisher_as_seed_platform_year",
        "vg_same_series_as_seed_genre_year",
        "vg_same_engine_as_seed_platform_year",
    ),
    "L5": (
        "vg_same_developer_and_publisher_as_seeds_genre_year",
        "vg_same_developer_and_publisher_as_seeds_platform_year",
        "vg_same_publisher_and_series_as_seeds_genre_year",
        "vg_same_developer_and_series_as_seeds_platform_year",
        "vg_same_developer_and_engine_as_seeds_platform_year",
        "vg_same_developer_as_seed_genre_platform_year",
        "vg_same_publisher_as_seed_genre_platform_year",
        "vg_same_series_as_seed_genre_platform_year",
        "vg_same_engine_as_seed_genre_platform_year",
        "vg_same_developer_as_seed_genre_publisher_year",
    ),
}


def _vg_slot_try_plan(level: str, accepted_index_within_level: int) -> List[str]:
    plan = list(VG_TEMPLATE_PLAN_BY_LEVEL.get(level, ()))
    if not plan:
        raise ValueError(f"No template plan for videogames {level}")
    start = accepted_index_within_level % len(plan)
    rotated = plan[start:] + plan[:start]
    width = min(len(rotated), max(1, int(VIDEOGAMES_SLOT_TRY_WIDTH)))
    return rotated[:width]


def _vg_skip_reason_counts(skipped: Sequence[Dict[str, Any]], level: Optional[str] = None, top_n: int = 5) -> Dict[str, int]:
    c = Counter()
    for row in skipped:
        if level is not None and row.get("complexity") != level:
            continue
        reason = str(row.get("reason") or "unknown")
        # Collapse noisy labels such as hidden_bridge_overused_global_pre_gold:Ubisoft
        reason = reason.split(":", 1)[0]
        c[reason] += 1
    return dict(c.most_common(top_n))


def _vg_level_attempt_budget(level: str, fallback: int) -> int:
    try:
        return int(VIDEOGAMES_LEVEL_ATTEMPT_BUDGET.get(level, fallback))
    except Exception:
        return int(fallback)


def generate_videogames_dataset(
    target_plan: Optional[Dict[str, int]] = None,
    output_path: Path = VIDEOGAMES_OUTPUT_PATH,
    audit_path: Path = VIDEOGAMES_AUDIT_PATH,
    overwrite: bool = OVERWRITE_VIDEOGAMES_OUTPUT,
    seed: int = VIDEOGAMES_SEED,
    max_visible_attempts_per_level: int = 620,
) -> List[BenchmarkExample]:
    target_plan = dict(target_plan or VIDEOGAMES_TARGET_PLAN)
    rng = random.Random(seed)
    records: List[BenchmarkExample] = []
    skipped: List[Dict[str, Any]] = []
    seen_public_keys: set = set()
    seen_gold_sets: set = set()
    hidden_counts_global: Counter = Counter()
    hidden_counts_by_level: Counter = Counter()

    if overwrite and output_path.exists():
        output_path.unlink()

    total_target = sum(target_plan.values())
    overall_bar = tqdm(total=total_target, desc="videogames total") if tqdm is not None else None
    idx = 1
    started_at = time.time()

    try:
        for level, target in target_plan.items():
            if target <= 0:
                continue
            ok = 0
            attempts = 0
            level_budget = _vg_level_attempt_budget(level, max_visible_attempts_per_level)
            level_bar = tqdm(total=target, desc=f"videogames {level}") if tqdm is not None else None

            while ok < target and attempts < level_budget:
                accepted: Optional[BenchmarkExample] = None
                for template_id in _vg_slot_try_plan(level, ok + attempts):
                    attempts += 1
                    with _vg_wdqs_fail_fast_context():
                        ex = _vg_generate_candidate_from_template(
                            complexity=level,
                            idx=idx,
                            template_id=template_id,
                            rng=rng,
                            seen_public_keys=seen_public_keys,
                            hidden_counts_global=hidden_counts_global,
                            hidden_counts_by_level=hidden_counts_by_level,
                            skipped=skipped,
                        )
                    if ex is None:
                        if attempts >= level_budget:
                            break
                        if VIDEOGAMES_HEARTBEAT_EVERY_ATTEMPTS > 0 and attempts % VIDEOGAMES_HEARTBEAT_EVERY_ATTEMPTS == 0:
                            msg = f"[INFO] videogames {level}: {ok}/{target}, attempts={attempts}/{level_budget}, recent_skip_reasons={_vg_skip_reason_counts(skipped, level)}"
                            print(msg)
                            if level_bar is not None:
                                level_bar.set_postfix({"attempts": attempts, "skips": _vg_skip_reason_counts(skipped, level, 2)})
                        continue
                    reason = _vg_diversity_reject_reason(
                        ex,
                        seen_public_keys=seen_public_keys,
                        seen_gold_sets=seen_gold_sets,
                        accepted_records=records,
                        hidden_counts_global=hidden_counts_global,
                        hidden_counts_by_level=hidden_counts_by_level,
                    )
                    if reason:
                        skipped.append({"complexity": level, "template_id": template_id, "reason": reason, "record_id": ex.id})
                        if attempts >= level_budget:
                            break
                        if VIDEOGAMES_HEARTBEAT_EVERY_ATTEMPTS > 0 and attempts % VIDEOGAMES_HEARTBEAT_EVERY_ATTEMPTS == 0:
                            msg = f"[INFO] videogames {level}: {ok}/{target}, attempts={attempts}/{level_budget}, recent_skip_reasons={_vg_skip_reason_counts(skipped, level)}"
                            print(msg)
                            if level_bar is not None:
                                level_bar.set_postfix({"attempts": attempts, "skips": _vg_skip_reason_counts(skipped, level, 2)})
                        continue
                    accepted = ex
                    break

                if accepted is None:
                    continue

                _vg_validate_record_schema(accepted)
                _vg_mark_diversity_accept(
                    accepted,
                    seen_public_keys=seen_public_keys,
                    seen_gold_sets=seen_gold_sets,
                    hidden_counts_global=hidden_counts_global,
                    hidden_counts_by_level=hidden_counts_by_level,
                )
                records.append(accepted)
                _vg_append_jsonl(output_path, asdict(accepted))
                idx += 1
                ok += 1

                if level_bar is not None:
                    level_bar.update(1)
                    level_bar.set_postfix({"attempts": attempts, "gold": len(accepted.gold_answer_qids), "tpl": accepted.template_id})
                if overall_bar is not None:
                    overall_bar.update(1)
                    overall_bar.set_postfix({"level": level, "gold": len(accepted.gold_answer_qids)})
                _vg_write_json(VIDEOGAMES_CHECKPOINT_PATH, {"last_record": asdict(accepted), "audit": _vg_audit(records, skipped)})

            if level_bar is not None:
                level_bar.close()
            if ok < target:
                level_counts = _vg_skip_reason_counts(skipped, level, top_n=12)
                msg = f"[ERROR] videogames {level}: generated {ok}/{target} after {attempts}/{level_budget} visible attempts; top_skip_reasons={level_counts}"
                print(msg)
                if VIDEOGAMES_STRICT_LEVEL_TARGET:
                    raise RuntimeError(
                        msg + " — stopping here instead of silently jumping to the next level. "
                        f"Increase VIDEOGAMES_V9_LEVEL_ATTEMPT_BUDGET['{level}'] or inspect {audit_path}."
                    )
            else:
                print(f"OK videogames {level}: generated {ok}/{target} after {attempts} visible attempts")

    finally:
        if overall_bar is not None:
            overall_bar.close()
        audit = _vg_audit(records, skipped)
        audit.update({
            "v9_level_fill_patch": True,
            "elapsed_seconds": round(time.time() - started_at, 2),
            "quick_probe_limit": VIDEOGAMES_QUICK_PROBE_LIMIT,
            "template_inner_attempts": VIDEOGAMES_TEMPLATE_INNER_ATTEMPTS,
            "slot_try_width": VIDEOGAMES_SLOT_TRY_WIDTH,
            "level_attempt_budget": VIDEOGAMES_LEVEL_ATTEMPT_BUDGET,
            "strict_level_target": VIDEOGAMES_STRICT_LEVEL_TARGET,
            "wdqs_fast_timeout_seconds": VIDEOGAMES_WDQS_FAST_TIMEOUT_SECONDS,
            "wdqs_fast_max_retries": VIDEOGAMES_WDQS_FAST_MAX_RETRIES,
            "max_same_hidden_bridge_global": VIDEOGAMES_MAX_SAME_HIDDEN_BRIDGE_GLOBAL,
            "max_same_hidden_bridge_per_level": VIDEOGAMES_MAX_SAME_HIDDEN_BRIDGE_PER_LEVEL,
            "max_gold_jaccard_overlap": VIDEOGAMES_MAX_GOLD_JACCARD_OVERLAP,
            "gold_max_accepted_enforced": VIDEOGAMES_MAX_ACCEPTED_GOLD,
            "ambiguous_genres_removed_from_anchors": sorted(VG_AMBIGUOUS_DIRECT_GENRES),
            "template_plan": {k: list(v) for k, v in VG_TEMPLATE_PLAN_BY_LEVEL.items()},
            "skip_reason_top": _vg_skip_reason_counts(skipped, None, top_n=20),
        })
        _vg_write_json(audit_path, audit)
        print("saved:", output_path.resolve())
        print("audit:", audit_path.resolve())
        print("records:", len(records))
        print("counts:", dict(Counter(x.complexity for x in records)))
        print("families:", dict(Counter(x.template_family for x in records)))
        print("skipped:", len(skipped))

    if output_path.exists():
        _vg_validate_output_jsonl(output_path, expected_count=len(records))
    if VIDEOGAMES_STRICT_TARGET and len(records) != total_target:
        raise RuntimeError(
            f"videogames generation produced {len(records)}/{total_target} records. "
            f"See audit for skipped reasons: {audit_path}"
        )
    _vg_validate_output_jsonl(output_path, expected_count=total_target)
    return records


# === v10 L4/L5 fast-generation patch ===
# This patch is intentionally focused on the hard tail only.  The previous v8/v9
# code was slow on L4/L5 because every random candidate could trigger:
#   1) seed-pool WDQS lookup;
#   2) compatible genre/platform/publisher WDQS pools;
#   3) quick gold probe;
#   4) full LIMIT-101 gold probe;
# and then be rejected by diversity/<=100/full-gold filters.  v10 keeps the same
# output schema and <=100 clean-gold rule, but removes the most expensive
# compatible-pool step and makes hidden bridge conditions direct in SPARQL.

VIDEOGAMES_V10_PATCH = True
VIDEOGAMES_TARGET_PLAN = {
    "L1": 0,
    "L2": 0,
    "L3": 0,
    "L4": 0,
    "L5": 30,
}
VIDEOGAMES_OUTPUT_PATH = VIDEOGAMES_DOMAIN_OUT_DIR / "videogames_l5.jsonl"
VIDEOGAMES_AUDIT_PATH = VIDEOGAMES_DOMAIN_OUT_DIR / "videogames_l5_generation_audit.json"
VIDEOGAMES_CHECKPOINT_PATH = VIDEOGAMES_DOMAIN_OUT_DIR / "videogames_l5_generation_checkpoint.json"

# Quality is still enforced by full LIMIT-101 probes.  These speed parameters
# only control random-candidate probing.  One retry is deliberate: a timed-out
# random candidate is cheaper to skip than to retry for minutes.
VIDEOGAMES_QUICK_PROBE_LIMIT = int(globals().get("VIDEOGAMES_V10_QUICK_PROBE_LIMIT", 8))
VIDEOGAMES_TEMPLATE_INNER_ATTEMPTS = int(globals().get("VIDEOGAMES_V10_TEMPLATE_INNER_ATTEMPTS", 3))
VIDEOGAMES_SLOT_TRY_WIDTH = int(globals().get("VIDEOGAMES_V10_SLOT_TRY_WIDTH", 99))
VIDEOGAMES_HEARTBEAT_EVERY_ATTEMPTS = int(globals().get("VIDEOGAMES_V10_HEARTBEAT_EVERY_ATTEMPTS", 25))
VIDEOGAMES_WDQS_FAST_TIMEOUT_SECONDS = int(globals().get("VIDEOGAMES_V10_WDQS_FAST_TIMEOUT_SECONDS", 24))
VIDEOGAMES_WDQS_FAST_MAX_RETRIES = int(globals().get("VIDEOGAMES_V10_WDQS_FAST_MAX_RETRIES", 1))

# Keep exact duplicates blocked, but do not let bridge quotas kill the hard tail.
# We will do final manual cleaning after generation, so this mode prioritizes
# getting enough L4/L5 candidates with complete golds.
VIDEOGAMES_MAX_SAME_HIDDEN_BRIDGE_GLOBAL = int(globals().get("VIDEOGAMES_V10_MAX_SAME_HIDDEN_BRIDGE_GLOBAL", 10))
VIDEOGAMES_MAX_SAME_HIDDEN_BRIDGE_PER_LEVEL = int(globals().get("VIDEOGAMES_V10_MAX_SAME_HIDDEN_BRIDGE_PER_LEVEL", 6))
VIDEOGAMES_MAX_GOLD_JACCARD_OVERLAP = float(globals().get("VIDEOGAMES_V10_MAX_GOLD_JACCARD_OVERLAP", 0.995))
VIDEOGAMES_STRICT_LEVEL_TARGET = bool(globals().get("VIDEOGAMES_V10_STRICT_LEVEL_TARGET", True))
VIDEOGAMES_STRICT_TARGET = bool(globals().get("VIDEOGAMES_V10_STRICT_TARGET", True))
VIDEOGAMES_LEVEL_ATTEMPT_BUDGET = dict(globals().get("VIDEOGAMES_V10_LEVEL_ATTEMPT_BUDGET", {
    "L1": 0,
    "L2": 0,
    "L3": 0,
    "L4": 0,
    "L5": 5200,
}))

# Fast high-coverage anchors.  They are used only to choose random visible
# constraints.  The generated SPARQL/ASK still validates every final record.
VG_FAST_BRIDGE_LABELS = {
    "developer": (
        "Nintendo", "Electronic Arts", "Ubisoft", "Sega", "Capcom", "Konami",
        "Square Enix", "Activision", "Valve", "Rockstar Games", "FromSoftware",
    ),
    "publisher": (
        "Nintendo", "Electronic Arts", "Ubisoft", "Sega", "Capcom", "Konami",
        "Square Enix", "Activision", "Sony Interactive Entertainment", "Bandai Namco Entertainment",
    ),
    "series": (
        "Final Fantasy", "Resident Evil", "Assassin's Creed", "Call of Duty",
        "The Legend of Zelda", "Mario", "Sonic the Hedgehog", "Halo", "Pokémon",
    ),
    # Engine tasks are useful but sparse; keep only the broadest engines and use
    # them late in the plan rather than as the main L4/L5 source.
    "engine": ("Unreal Engine", "Unity", "Source"),
}

VG_FAST_VISIBLE_LABELS = {
    "genre": (
        "role-playing video game", "shooter game", "platform game", "puzzle video game",
        "strategy video game", "adventure game", "fighting game", "stealth game",
    ),
    "platform": (
        "Microsoft Windows", "PlayStation 4", "Nintendo Switch", "Xbox One",
        "PlayStation 3", "PlayStation 2", "Android", "iOS", "Wii",
        "Nintendo DS", "Game Boy Advance", "Nintendo 3DS",
    ),
    "publisher": (
        "Nintendo", "Electronic Arts", "Ubisoft", "Sega", "Capcom", "Konami",
        "Square Enix", "Activision", "Sony Interactive Entertainment", "Bandai Namco Entertainment",
    ),
    "developer": (
        "Nintendo", "Electronic Arts", "Ubisoft", "Sega", "Capcom", "Konami",
        "Square Enix", "Valve", "Rockstar Games", "FromSoftware",
    ),
}


def _vg_year_window(complexity: str, rng: random.Random) -> Tuple[int, int]:
    # Wider windows for L4/L5 are faster and more stable under <=100 full-gold
    # filtering because hidden bridge + visible constraints are already selective.
    choices = {
        "L1": [(2000, 2008), (2006, 2014), (2012, 2020), (2018, 2025)],
        "L2": [(1996, 2008), (2004, 2013), (2010, 2018), (2016, 2025)],
        "L3": [(1998, 2007), (2006, 2014), (2012, 2020), (2018, 2025)],
        "L4": [(2000, 2015), (2006, 2020), (2010, 2025), (2014, 2025)],
        "L5": [(2000, 2020), (2006, 2025), (2010, 2025), (2014, 2025)],
    }
    return rng.choice(choices.get(complexity, choices["L4"]))


def _vg_pick_from_preferred_labels(prop_name: str, labels: Sequence[str], rng: random.Random) -> Optional[Dict[str, str]]:
    df = _vg_pool(prop_name)
    if df is None or len(df) == 0:
        return None
    labels = tuple(labels or ())
    if labels:
        sub = df[df["label_en"].isin(labels)].copy()
        if sub is not None and len(sub) > 0:
            df = sub
    row = df.sample(1, random_state=rng.randint(0, 10**9)).iloc[0]
    return _vg_entity_from_row(row)


def _vg_pick_seed(bridge: str, rng: random.Random) -> Dict[str, Dict[str, str]]:
    # Prefer high-coverage bridge values and keep trying more than v8/v9 did.
    values = _vg_pool(bridge)
    if values is None or len(values) == 0:
        raise ValueError(f"Empty videogames bridge-value pool for: {bridge}")
    labels = VG_FAST_BRIDGE_LABELS.get(bridge, ())
    if labels:
        sub = values[values["label_en"].isin(labels)].copy()
        if sub is not None and len(sub) > 0:
            values = sub
    order = list(values.sample(frac=1, random_state=rng.randint(0, 10**9)).to_dict("records"))
    last_empty = None
    for row in order[:24]:
        bridge_value = _vg_entity_from_row(row)
        seed_df = _vg_seed_pool_for_bridge_value(bridge, bridge_value)
        if seed_df is None or len(seed_df) == 0:
            last_empty = bridge_value.get("en") or bridge_value.get("qid")
            continue
        seed_row = seed_df.sample(1, random_state=rng.randint(0, 10**9)).iloc[0]
        seed_en = _vg_clean_label(seed_row.seed_label_en)
        bridge_en = _vg_clean_label(seed_row.bridge_label_en)
        if not (seed_en and bridge_en):
            continue
        return {
            "seed": {"qid": seed_row.seed_qid, "en": seed_en, "ru": _vg_display_ru(seed_en, seed_row.seed_label_ru)},
            "bridge_value": {"qid": seed_row.bridge_qid, "en": bridge_en, "ru": _vg_display_ru(bridge_en, seed_row.bridge_label_ru)},
            "bridge": bridge,
        }
    raise ValueError(f"Empty videogames seed pool for bridge: {bridge}; last_empty={last_empty}")


def _vg_pick_visible_for_bridge(bridge: str, bridge_value: Dict[str, str], prop_name: str, rng: random.Random) -> Dict[str, str]:
    # v10 deliberately avoids `videogames_compatible_*` WDQS pools.  Those pools
    # were the hidden source of most L4/L5 latency.  A fast global high-coverage
    # pick plus quick/full gold probes is cheaper and still fully validated.
    ent = _vg_pick_from_preferred_labels(prop_name, VG_FAST_VISIBLE_LABELS.get(prop_name, ()), rng)
    if ent:
        return ent
    return _vg_pick(prop_name, rng)


def _vg_bridge_phrase_ru(bridge: str, seed: Dict[str, str]) -> str:
    return {
        "developer": f"разработчик совпадает с разработчиком игры «{seed['ru']}»",
        "publisher": f"издатель совпадает с издателем игры «{seed['ru']}»",
        "series": f"серия совпадает с серией игры «{seed['ru']}»",
        "engine": f"используется тот же игровой движок, что и в игре «{seed['ru']}»",
    }[bridge]


def _vg_bridge_phrase_en(bridge: str, seed: Dict[str, str]) -> str:
    return {
        "developer": f"the developer is the same as the developer of {seed['en']}",
        "publisher": f"the publisher is the same as the publisher of {seed['en']}",
        "series": f"the series is the same as the series of {seed['en']}",
        "engine": f"the game engine is the same as the game engine of {seed['en']}",
    }[bridge]


def _vg_visible_phrase_ru(prop: str, val: Dict[str, str]) -> str:
    if prop == "genre":
        return _vg_genre_phrase_ru(val)
    if prop == "platform":
        return _vg_platform_phrase_ru(val)
    if prop == "publisher":
        return _vg_publisher_phrase_ru(val)
    if prop == "developer":
        return _vg_developer_phrase_ru(val)
    if prop == "series":
        return _vg_series_phrase_ru(val)
    return f"{prop} — {val.get('ru') or val.get('en')}"


def _vg_visible_phrase_en(prop: str, val: Dict[str, str]) -> str:
    if prop == "genre":
        return _vg_genre_phrase_en(val)
    if prop == "platform":
        return _vg_platform_phrase_en(val)
    if prop == "publisher":
        return _vg_publisher_phrase_en(val)
    if prop == "developer":
        return _vg_developer_phrase_en(val)
    if prop == "series":
        return _vg_series_phrase_en(val)
    return f"{prop} is {val.get('en')}"


def _vg_tpl_same_bridge_as_seed(complexity: str, rng: random.Random, *, bridge: str, visible: Sequence[str] = ()) -> Dict[str, Any]:
    seed_pack = _vg_pick_seed(bridge, rng)
    seed = seed_pack["seed"]
    bridge_value = seed_pack["bridge_value"]
    pid = VG_BRIDGES[bridge]["pid"]
    public_key = VG_BRIDGES[bridge]["public_key"]
    y1, y2 = _vg_year_window(complexity, rng)

    # Direct hidden value is much faster than resolving ?bridgeValue through the
    # seed inside every gold query.  The seed is still stored and shown to the
    # model; the bridge value is kept in gold_collection_meta for auditability.
    where = [
        f"?item wdt:{pid} wd:{bridge_value['qid']} .",
        f"FILTER(?item != wd:{seed['qid']}) .",
    ]
    public_kwargs: Dict[str, Any] = {public_key: seed, "release_year_from": y1, "release_year_to": y2}
    qid_kwargs: Dict[str, Any] = {public_key: seed, "hidden_bridge_value": bridge_value, "release_year_from": y1, "release_year_to": y2}
    ru_parts: List[str] = [_vg_bridge_phrase_ru(bridge, seed)]
    en_parts: List[str] = [_vg_bridge_phrase_en(bridge, seed)]

    for prop in visible:
        if prop not in VG_PROPERTY_PIDS:
            continue
        val = _vg_pick_visible_for_bridge(bridge, bridge_value, prop, rng)
        if not val or not val.get("qid"):
            continue
        where.append(f"?item wdt:{VG_PROPERTY_PIDS[prop]} wd:{val['qid']} .")
        public_kwargs[prop] = val
        qid_kwargs[prop] = val
        ru_parts.append(_vg_visible_phrase_ru(prop, val))
        en_parts.append(_vg_visible_phrase_en(prop, val))

    where += _vg_year_where(y1, y2)
    ru_parts.append(_vg_year_clause_ru(y1, y2))
    en_parts.append(_vg_year_clause_en(y1, y2))
    visible_suffix = "_" + "_".join(visible) if visible else ""
    template_id = f"vg_same_{bridge}_as_seed{visible_suffix}_year"
    k = _vg_requested(complexity)
    return {
        "template_id": template_id,
        "template_family": "hidden_seed_bridge_fast_direct_value",
        "query_text_ru": _vg_make_query_text_ru(k, ru_parts),
        "query_text_en": _vg_make_query_text_en(k, en_parts),
        "constraints": _vg_public_constraints(**public_kwargs),
        "qid_constraints": _vg_constraint_qids(**qid_kwargs),
        "where_lines": where,
        "requested_count": k,
    }

# Re-register single-hidden templates so they use the fast direct-value builder.
VG_TEMPLATE_BUILDERS.update({
    "vg_same_developer_as_seed_year": lambda c, r: _vg_tpl_same_bridge_as_seed(c, r, bridge="developer"),
    "vg_same_publisher_as_seed_year": lambda c, r: _vg_tpl_same_bridge_as_seed(c, r, bridge="publisher"),
    "vg_same_series_as_seed_year": lambda c, r: _vg_tpl_same_bridge_as_seed(c, r, bridge="series"),
    "vg_same_engine_as_seed_year": lambda c, r: _vg_tpl_same_bridge_as_seed(c, r, bridge="engine"),
    "vg_same_developer_as_seed_genre_year": lambda c, r: _vg_tpl_same_bridge_as_seed(c, r, bridge="developer", visible=("genre",)),
    "vg_same_developer_as_seed_platform_year": lambda c, r: _vg_tpl_same_bridge_as_seed(c, r, bridge="developer", visible=("platform",)),
    "vg_same_publisher_as_seed_genre_year": lambda c, r: _vg_tpl_same_bridge_as_seed(c, r, bridge="publisher", visible=("genre",)),
    "vg_same_publisher_as_seed_platform_year": lambda c, r: _vg_tpl_same_bridge_as_seed(c, r, bridge="publisher", visible=("platform",)),
    "vg_same_series_as_seed_genre_year": lambda c, r: _vg_tpl_same_bridge_as_seed(c, r, bridge="series", visible=("genre",)),
    "vg_same_series_as_seed_platform_year": lambda c, r: _vg_tpl_same_bridge_as_seed(c, r, bridge="series", visible=("platform",)),
    "vg_same_engine_as_seed_genre_year": lambda c, r: _vg_tpl_same_bridge_as_seed(c, r, bridge="engine", visible=("genre",)),
    "vg_same_engine_as_seed_platform_year": lambda c, r: _vg_tpl_same_bridge_as_seed(c, r, bridge="engine", visible=("platform",)),
    "vg_same_developer_as_seed_genre_platform_year": lambda c, r: _vg_tpl_same_bridge_as_seed(c, r, bridge="developer", visible=("genre", "platform")),
    "vg_same_publisher_as_seed_genre_platform_year": lambda c, r: _vg_tpl_same_bridge_as_seed(c, r, bridge="publisher", visible=("genre", "platform")),
    "vg_same_series_as_seed_genre_platform_year": lambda c, r: _vg_tpl_same_bridge_as_seed(c, r, bridge="series", visible=("genre", "platform")),
    "vg_same_engine_as_seed_genre_platform_year": lambda c, r: _vg_tpl_same_bridge_as_seed(c, r, bridge="engine", visible=("genre", "platform")),
    "vg_same_developer_as_seed_genre_publisher_year": lambda c, r: _vg_tpl_same_bridge_as_seed(c, r, bridge="developer", visible=("genre", "publisher")),
})

# L4 now prioritizes fast one-hop multihop patterns that reliably produce enough
# clean golds.  L5 starts with stronger two-visible-criterion patterns, then has
# fast L4-style fallbacks so the run does not stall for hours on sparse combos.
VG_TEMPLATE_PLAN_BY_LEVEL = {
    "L1": (),
    "L2": (),
    "L3": (),
    "L4": (
        "vg_same_developer_as_seed_platform_year",
        "vg_same_publisher_as_seed_genre_year",
        "vg_same_developer_as_seed_genre_year",
        "vg_same_publisher_as_seed_platform_year",
        "vg_same_series_as_seed_platform_year",
        "vg_same_series_as_seed_genre_year",
        "vg_same_engine_as_seed_platform_year",
        "vg_same_engine_as_seed_genre_year",
    ),
    "L5": (
        "vg_same_developer_as_seed_genre_platform_year",
        "vg_same_publisher_as_seed_genre_platform_year",
        "vg_same_developer_as_seed_genre_publisher_year",
        "vg_same_series_as_seed_genre_platform_year",
        "vg_same_developer_as_seed_platform_year",
        "vg_same_publisher_as_seed_genre_year",
        "vg_same_developer_as_seed_genre_year",
        "vg_same_publisher_as_seed_platform_year",
        "vg_same_series_as_seed_platform_year",
        "vg_same_engine_as_seed_platform_year",
    ),
}


def _vg_run_l4_l5_sanity_checks() -> None:
    assert VIDEOGAMES_TARGET_PLAN == {"L1": 0, "L2": 0, "L3": 0, "L4": 0, "L5": 30}
    assert sum(VIDEOGAMES_TARGET_PLAN.values()) == 30
    assert VIDEOGAMES_PROBE_LIMIT == VIDEOGAMES_MAX_ACCEPTED_GOLD + 1
    assert VIDEOGAMES_MAX_ACCEPTED_GOLD == 100
    assert len(VG_TEMPLATE_PLAN_BY_LEVEL["L4"]) >= 6
    assert len(VG_TEMPLATE_PLAN_BY_LEVEL["L5"]) >= 8
    # Do not call template builders here: builder calls may need WDQS seed pools.
    # The actual direct-value behavior is enforced by the v10 override of
    # _vg_tpl_same_bridge_as_seed and validated on every accepted record.
    assert callable(VG_TEMPLATE_BUILDERS["vg_same_developer_as_seed_platform_year"])

_vg_run_l4_l5_sanity_checks()

print(
    "✅ videogames v10 fast patch enabled: target 30 L5 only, direct hidden bridge values, "
    "no compatible-pool WDQS step, <=100 complete golds, explicit independent wording"
)


# -----------------------------------------------------------------------------
# v12: L5-only pattern-diversity config.
# v10 was intentionally fast, but too many accepted hard-tail records collapsed into
# the same family: one hidden bridge + one/two visible criteria + year.  v11 keeps
# the v10 speed trick (direct hidden QID values in SPARQL) but restores stronger
# pattern diversity:
#   * L4: one hidden bridge + two visible criteria + year;
#   * L5: two independent hidden bridges + one visible criterion + year first;
#         single-hidden/three-visible templates are only late fallbacks.
# -----------------------------------------------------------------------------

VIDEOGAMES_TEMPLATE_INNER_ATTEMPTS = int(globals().get("VIDEOGAMES_V12_TEMPLATE_INNER_ATTEMPTS", 4))
VIDEOGAMES_SLOT_TRY_WIDTH = int(globals().get("VIDEOGAMES_V12_SLOT_TRY_WIDTH", 99))
VIDEOGAMES_HEARTBEAT_EVERY_ATTEMPTS = int(globals().get("VIDEOGAMES_V12_HEARTBEAT_EVERY_ATTEMPTS", 20))
VIDEOGAMES_LEVEL_ATTEMPT_BUDGET = dict(globals().get("VIDEOGAMES_V12_LEVEL_ATTEMPT_BUDGET", {
    "L1": 0,
    "L2": 0,
    "L3": 0,
    "L4": 0,
    "L5": 6200,
}))

# Let the hard tail produce enough candidates; exact duplicate gold sets and exact
# duplicate public constraints are still rejected.  Final manual pruning will keep
# only the strongest/diverse examples.
VIDEOGAMES_MAX_SAME_HIDDEN_BRIDGE_GLOBAL = int(globals().get("VIDEOGAMES_V12_MAX_SAME_HIDDEN_BRIDGE_GLOBAL", 14))
VIDEOGAMES_MAX_SAME_HIDDEN_BRIDGE_PER_LEVEL = int(globals().get("VIDEOGAMES_V12_MAX_SAME_HIDDEN_BRIDGE_PER_LEVEL", 8))
VIDEOGAMES_MAX_GOLD_JACCARD_OVERLAP = float(globals().get("VIDEOGAMES_V12_MAX_GOLD_JACCARD_OVERLAP", 0.998))

VG_V11_BRIDGE_PAIR_LABELS: Dict[Tuple[str, str], Tuple[Tuple[str, str], ...]] = {
    ("developer", "publisher"): (
        ("Nintendo", "Nintendo"),
        ("Sega", "Sega"),
        ("Capcom", "Capcom"),
        ("Konami", "Konami"),
        ("Square Enix", "Square Enix"),
        ("Ubisoft", "Ubisoft"),
        ("Electronic Arts", "Electronic Arts"),
        ("Activision", "Activision"),
        ("Sony Interactive Entertainment", "Sony Interactive Entertainment"),
        ("Bandai Namco Entertainment", "Bandai Namco Entertainment"),
    ),
    ("publisher", "series"): (
        ("Nintendo", "Mario"),
        ("Nintendo", "The Legend of Zelda"),
        ("Nintendo", "Pokémon"),
        ("Sega", "Sonic the Hedgehog"),
        ("Square Enix", "Final Fantasy"),
        ("Capcom", "Resident Evil"),
        ("Ubisoft", "Assassin's Creed"),
        ("Activision", "Call of Duty"),
        ("Sony Interactive Entertainment", "Uncharted"),
    ),
    ("developer", "series"): (
        ("Nintendo", "Mario"),
        ("Nintendo", "The Legend of Zelda"),
        ("Sega", "Sonic the Hedgehog"),
        ("Square Enix", "Final Fantasy"),
        ("Capcom", "Resident Evil"),
        ("Ubisoft", "Assassin's Creed"),
        ("Valve", "Half-Life"),
        ("Rockstar Games", "Grand Theft Auto"),
        ("FromSoftware", "Dark Souls"),
    ),
}


def _vg_entity_by_exact_en_label(prop_name: str, label_en: str) -> Optional[Dict[str, str]]:
    df = _vg_pool(prop_name)
    if df is None or len(df) == 0:
        return None
    sub = df[df["label_en"].astype(str).eq(str(label_en))].copy()
    if sub is None or len(sub) == 0:
        return None
    return _vg_entity_from_row(sub.iloc[0])


def _vg_seed_for_bridge_value(bridge: str, bridge_value: Dict[str, str], rng: random.Random) -> Optional[Dict[str, str]]:
    seed_df = _vg_seed_pool_for_bridge_value(bridge, bridge_value)
    if seed_df is None or len(seed_df) == 0:
        return None
    # Prefer cleaner / recognizable seed labels when available, but do not spend
    # WDQS time here: this pool is cached and already label-filtered.
    row = seed_df.sample(1, random_state=rng.randint(0, 10**9)).iloc[0]
    seed_en = _vg_clean_label(row.seed_label_en)
    if not seed_en:
        return None
    return {
        "qid": row.seed_qid,
        "en": seed_en,
        "ru": _vg_display_ru(seed_en, row.seed_label_ru),
    }


def _vg_pick_bridge_pair(bridge_a: str, bridge_b: str, rng: random.Random) -> Dict[str, Dict[str, str]]:
    if bridge_a == bridge_b:
        raise ValueError("bridge pair must contain two different bridge types")

    # Curated compatible pairs first.  They make L5 much faster than arbitrary
    # cross-products and keep the questions human-recognizable.
    pairs = list(VG_V11_BRIDGE_PAIR_LABELS.get((bridge_a, bridge_b), ()))
    rng.shuffle(pairs)
    for label_a, label_b in pairs:
        value_a = _vg_entity_by_exact_en_label(bridge_a, label_a)
        value_b = _vg_entity_by_exact_en_label(bridge_b, label_b)
        if not value_a or not value_b:
            continue
        seed_a = _vg_seed_for_bridge_value(bridge_a, value_a, rng)
        seed_b = _vg_seed_for_bridge_value(bridge_b, value_b, rng)
        if seed_a and seed_b:
            return {
                "seed_a": seed_a,
                "value_a": value_a,
                "seed_b": seed_b,
                "value_b": value_b,
                "bridge_a": bridge_a,
                "bridge_b": bridge_b,
            }

    # Last-resort fallback: still direct-value SPARQL, but with independently
    # sampled bridge values.  Gold probes will reject empty intersections.
    pack_a = _vg_pick_seed(bridge_a, rng)
    pack_b = _vg_pick_seed(bridge_b, rng)
    return {
        "seed_a": pack_a["seed"],
        "value_a": pack_a["bridge_value"],
        "seed_b": pack_b["seed"],
        "value_b": pack_b["bridge_value"],
        "bridge_a": bridge_a,
        "bridge_b": bridge_b,
    }


def _vg_tpl_two_seed_bridges_direct(
    complexity: str,
    rng: random.Random,
    *,
    bridge_a: str,
    bridge_b: str,
    visible: Sequence[str] = (),
) -> Dict[str, Any]:
    pack = _vg_pick_bridge_pair(bridge_a, bridge_b, rng)
    seed_a, seed_b = pack["seed_a"], pack["seed_b"]
    value_a, value_b = pack["value_a"], pack["value_b"]
    pid_a = VG_BRIDGES[bridge_a]["pid"]
    pid_b = VG_BRIDGES[bridge_b]["pid"]
    key_a = VG_BRIDGES[bridge_a]["public_key"]
    key_b = VG_BRIDGES[bridge_b]["public_key"]
    y1, y2 = _vg_year_window(complexity, rng)

    where = [
        f"?item wdt:{pid_a} wd:{value_a['qid']} .",
        f"?item wdt:{pid_b} wd:{value_b['qid']} .",
        f"FILTER(?item != wd:{seed_a['qid']} && ?item != wd:{seed_b['qid']}) .",
    ]
    public_kwargs: Dict[str, Any] = {
        key_a: seed_a,
        key_b: seed_b,
        "release_year_from": y1,
        "release_year_to": y2,
    }
    qid_kwargs: Dict[str, Any] = {
        key_a: seed_a,
        key_b: seed_b,
        "hidden_bridge_values": [
            {"bridge": bridge_a, **value_a},
            {"bridge": bridge_b, **value_b},
        ],
        "release_year_from": y1,
        "release_year_to": y2,
    }
    ru_parts: List[str] = [_vg_bridge_phrase_ru(bridge_a, seed_a), _vg_bridge_phrase_ru(bridge_b, seed_b)]
    en_parts: List[str] = [_vg_bridge_phrase_en(bridge_a, seed_a), _vg_bridge_phrase_en(bridge_b, seed_b)]

    for prop in visible:
        if prop not in VG_PROPERTY_PIDS or prop in (bridge_a, bridge_b):
            continue
        # Use the fast v10 picker.  It avoids compatible-pool WDQS scans; the
        # final gold query is the authoritative validator.
        val = _vg_pick_visible_for_bridge(bridge_a, value_a, prop, rng)
        if not val or not val.get("qid"):
            continue
        where.append(f"?item wdt:{VG_PROPERTY_PIDS[prop]} wd:{val['qid']} .")
        public_kwargs[prop] = val
        qid_kwargs[prop] = val
        ru_parts.append(_vg_visible_phrase_ru(prop, val))
        en_parts.append(_vg_visible_phrase_en(prop, val))

    where += _vg_year_where(y1, y2)
    ru_parts.append(_vg_year_clause_ru(y1, y2))
    en_parts.append(_vg_year_clause_en(y1, y2))

    visible_suffix = "_" + "_".join(visible) if visible else ""
    template_id = f"vg_same_{bridge_a}_and_{bridge_b}_as_seeds{visible_suffix}_year"
    k = _vg_requested(complexity)
    return {
        "template_id": template_id,
        "template_family": "double_hidden_seed_bridge_fast_direct_value",
        "query_text_ru": _vg_make_query_text_ru(k, ru_parts),
        "query_text_en": _vg_make_query_text_en(k, en_parts),
        "constraints": _vg_public_constraints(**public_kwargs),
        "qid_constraints": _vg_constraint_qids(**qid_kwargs),
        "where_lines": where,
        "requested_count": k,
    }


# Additional single-hidden templates with two/three visible criteria.  These keep
# L4 harder than L3 and provide late L5 fallback patterns that are not identical
# to the early double-hidden templates.
VG_TEMPLATE_BUILDERS.update({
    "vg_same_developer_as_seed_platform_publisher_year": lambda c, r: _vg_tpl_same_bridge_as_seed(c, r, bridge="developer", visible=("platform", "publisher")),
    "vg_same_developer_as_seed_genre_platform_publisher_year": lambda c, r: _vg_tpl_same_bridge_as_seed(c, r, bridge="developer", visible=("genre", "platform", "publisher")),
    "vg_same_publisher_as_seed_genre_developer_year": lambda c, r: _vg_tpl_same_bridge_as_seed(c, r, bridge="publisher", visible=("genre", "developer")),
    "vg_same_publisher_as_seed_platform_developer_year": lambda c, r: _vg_tpl_same_bridge_as_seed(c, r, bridge="publisher", visible=("platform", "developer")),
    "vg_same_publisher_as_seed_genre_platform_developer_year": lambda c, r: _vg_tpl_same_bridge_as_seed(c, r, bridge="publisher", visible=("genre", "platform", "developer")),
    "vg_same_series_as_seed_platform_publisher_year": lambda c, r: _vg_tpl_same_bridge_as_seed(c, r, bridge="series", visible=("platform", "publisher")),
    "vg_same_series_as_seed_genre_publisher_year": lambda c, r: _vg_tpl_same_bridge_as_seed(c, r, bridge="series", visible=("genre", "publisher")),
    "vg_same_series_as_seed_genre_platform_publisher_year": lambda c, r: _vg_tpl_same_bridge_as_seed(c, r, bridge="series", visible=("genre", "platform", "publisher")),
    "vg_same_engine_as_seed_genre_developer_year": lambda c, r: _vg_tpl_same_bridge_as_seed(c, r, bridge="engine", visible=("genre", "developer")),
    "vg_same_engine_as_seed_platform_developer_year": lambda c, r: _vg_tpl_same_bridge_as_seed(c, r, bridge="engine", visible=("platform", "developer")),
})

# Fast direct double-hidden templates.
VG_TEMPLATE_BUILDERS.update({
    "vg_same_developer_and_publisher_as_seeds_year": lambda c, r: _vg_tpl_two_seed_bridges_direct(c, r, bridge_a="developer", bridge_b="publisher"),
    "vg_same_developer_and_publisher_as_seeds_genre_year": lambda c, r: _vg_tpl_two_seed_bridges_direct(c, r, bridge_a="developer", bridge_b="publisher", visible=("genre",)),
    "vg_same_developer_and_publisher_as_seeds_platform_year": lambda c, r: _vg_tpl_two_seed_bridges_direct(c, r, bridge_a="developer", bridge_b="publisher", visible=("platform",)),
    "vg_same_publisher_and_series_as_seeds_year": lambda c, r: _vg_tpl_two_seed_bridges_direct(c, r, bridge_a="publisher", bridge_b="series"),
    "vg_same_publisher_and_series_as_seeds_genre_year": lambda c, r: _vg_tpl_two_seed_bridges_direct(c, r, bridge_a="publisher", bridge_b="series", visible=("genre",)),
    "vg_same_publisher_and_series_as_seeds_platform_year": lambda c, r: _vg_tpl_two_seed_bridges_direct(c, r, bridge_a="publisher", bridge_b="series", visible=("platform",)),
    "vg_same_developer_and_series_as_seeds_year": lambda c, r: _vg_tpl_two_seed_bridges_direct(c, r, bridge_a="developer", bridge_b="series"),
    "vg_same_developer_and_series_as_seeds_genre_year": lambda c, r: _vg_tpl_two_seed_bridges_direct(c, r, bridge_a="developer", bridge_b="series", visible=("genre",)),
    "vg_same_developer_and_series_as_seeds_platform_year": lambda c, r: _vg_tpl_two_seed_bridges_direct(c, r, bridge_a="developer", bridge_b="series", visible=("platform",)),
})

VG_TEMPLATE_PLAN_BY_LEVEL = {
    "L1": (),
    "L2": (),
    "L3": (),
    # L4 is intentionally disabled in v12; this run is L5-only.
    "L4": (),
    # L5: double hidden first.  Late fallbacks are single hidden + three visible
    # criteria, so even fallback L5 remains stricter than L4.
    "L5": (
        "vg_same_developer_and_publisher_as_seeds_genre_year",
        "vg_same_developer_and_publisher_as_seeds_platform_year",
        "vg_same_publisher_and_series_as_seeds_genre_year",
        "vg_same_publisher_and_series_as_seeds_platform_year",
        "vg_same_developer_and_series_as_seeds_genre_year",
        "vg_same_developer_and_series_as_seeds_platform_year",
        "vg_same_developer_and_publisher_as_seeds_year",
        "vg_same_publisher_and_series_as_seeds_year",
        "vg_same_developer_and_series_as_seeds_year",
        "vg_same_developer_as_seed_genre_platform_publisher_year",
        "vg_same_publisher_as_seed_genre_platform_developer_year",
        "vg_same_series_as_seed_genre_platform_publisher_year",
    ),
}


def _vg_run_l5_v12_sanity_checks() -> None:
    assert VIDEOGAMES_TARGET_PLAN == {"L1": 0, "L2": 0, "L3": 0, "L4": 0, "L5": 30}
    assert sum(VIDEOGAMES_TARGET_PLAN.values()) == 30
    assert VIDEOGAMES_MAX_ACCEPTED_GOLD == 100
    assert VG_TEMPLATE_PLAN_BY_LEVEL["L4"] == ()
    assert all(t in VG_TEMPLATE_BUILDERS for t in VG_TEMPLATE_PLAN_BY_LEVEL["L5"])
    assert any("and_publisher_as_seeds" in t for t in VG_TEMPLATE_PLAN_BY_LEVEL["L5"])
    assert any("and_series_as_seeds" in t for t in VG_TEMPLATE_PLAN_BY_LEVEL["L5"])
    assert not any(t.endswith("_as_seed_platform_year") or t.endswith("_as_seed_genre_year") for t in VG_TEMPLATE_PLAN_BY_LEVEL["L5"])

_vg_run_l5_v12_sanity_checks()

print(
    "✅ videogames v12 L5-only pattern patch enabled: target 30 L5; "
    "double-hidden first + stronger fallbacks; direct hidden QIDs kept for speed"
)


# -----------------------------------------------------------------------------
# v13: QUALITY L5 multihop patch.
# Goals:
#   * generate only L5 records, but make them genuinely hard;
#   * every accepted L5 must have two distinct hidden bridge values;
#   * no L5 year-only double-hidden templates;
#   * require at least one visible content/platform constraint (genre or platform);
#   * use earliest publication date semantics, not "any P577 date";
#   * use curated compatible hidden pairs so generation is fast enough and prompts
#     remain recognizable/human-checkable.
# -----------------------------------------------------------------------------

VIDEOGAMES_V13_QUALITY_L5_PATCH = True
VIDEOGAMES_TARGET_PLAN = {"L1": 0, "L2": 0, "L3": 0, "L4": 0, "L5": 30}
VIDEOGAMES_OUTPUT_PATH = VIDEOGAMES_DOMAIN_OUT_DIR / "videogames_l5_quality.jsonl"
VIDEOGAMES_AUDIT_PATH = VIDEOGAMES_DOMAIN_OUT_DIR / "videogames_l5_quality_generation_audit.json"
VIDEOGAMES_CHECKPOINT_PATH = VIDEOGAMES_DOMAIN_OUT_DIR / "videogames_l5_quality_generation_checkpoint.json"

VIDEOGAMES_TEMPLATE_INNER_ATTEMPTS = int(globals().get("VIDEOGAMES_V13_TEMPLATE_INNER_ATTEMPTS", 4))
VIDEOGAMES_SLOT_TRY_WIDTH = int(globals().get("VIDEOGAMES_V13_SLOT_TRY_WIDTH", 99))
VIDEOGAMES_HEARTBEAT_EVERY_ATTEMPTS = int(globals().get("VIDEOGAMES_V13_HEARTBEAT_EVERY_ATTEMPTS", 15))
VIDEOGAMES_LEVEL_ATTEMPT_BUDGET = dict(globals().get("VIDEOGAMES_V13_LEVEL_ATTEMPT_BUDGET", {
    "L1": 0,
    "L2": 0,
    "L3": 0,
    "L4": 0,
    "L5": 9000,
}))

# Still allow enough hidden-value reuse to finish the tail, but cap exact hidden
# bridge pairs separately below so the output is not 30 Assassin's Creed/Ubisoft tasks.
VIDEOGAMES_MAX_SAME_HIDDEN_BRIDGE_GLOBAL = int(globals().get("VIDEOGAMES_V13_MAX_SAME_HIDDEN_BRIDGE_GLOBAL", 7))
VIDEOGAMES_MAX_SAME_HIDDEN_BRIDGE_PER_LEVEL = int(globals().get("VIDEOGAMES_V13_MAX_SAME_HIDDEN_BRIDGE_PER_LEVEL", 7))
VIDEOGAMES_MAX_SAME_L5_HIDDEN_PAIR = int(globals().get("VIDEOGAMES_V13_MAX_SAME_L5_HIDDEN_PAIR", 3))
VIDEOGAMES_MAX_GOLD_JACCARD_OVERLAP = float(globals().get("VIDEOGAMES_V13_MAX_GOLD_JACCARD_OVERLAP", 0.97))

# A few additional exact-label fallbacks that are useful for high-quality L5 pairs.
# These are only anchor QIDs for choosing constraints; every final answer is still
# validated by SELECT/ASK SPARQL.
VG_ANCHOR_FALLBACK_QIDS.setdefault("publisher", {}).update({
    "Sony Interactive Entertainment": "Q1062635",
})
VG_ANCHOR_FALLBACK_QIDS.setdefault("developer", {}).update({
    "Bethesda Game Studios": "Q3300783",
})

# Make sure the newly added anchors are present in the labels used by the v14 pool.
VG_ANCHOR_LABELS = {k: tuple(sorted(set(v) | set(VG_ANCHOR_FALLBACK_QIDS.get(k, {}).keys()))) for k, v in VG_ANCHOR_LABELS.items()}
_VG_POOL_MEMORY.clear()


def _vg_pool(prop_name: str) -> pd.DataFrame:
    """v13/v14 cache-key pool: avoids reusing older anchor pools missing quality anchors."""
    pid = VG_PROPERTY_PIDS[prop_name]
    cols = ["qid", "label_en", "label_ru"]
    frames: List[pd.DataFrame] = []
    api_anchor = _vg_load_or_build_pool_once(
        f"videogames_{prop_name}_api_anchor_pool_v14",
        lambda: _vg_build_api_anchor_pool(prop_name),
        cols,
    )
    if api_anchor is not None and len(api_anchor) > 0:
        frames.append(api_anchor)
    if VIDEOGAMES_VALIDATE_ANCHOR_POOLS_WITH_WDQS:
        wdqs_anchor = _vg_load_or_build_pool_once(
            f"videogames_{prop_name}_anchor_pool_v14",
            lambda: _vg_build_anchor_property_pool(prop_name, pid),
            cols,
        )
        if wdqs_anchor is not None and len(wdqs_anchor) > 0:
            frames.append(wdqs_anchor)
    if VIDEOGAMES_USE_BROAD_POOLS:
        broad = _vg_load_or_build_pool_once(
            f"videogames_{prop_name}_pool_v14",
            lambda: _vg_build_property_pool(prop_name, pid),
            cols,
        )
        if broad is not None and len(broad) > 0:
            frames.append(broad)
    if not frames:
        return _vg_empty_pool(cols)
    out = pd.concat(frames, ignore_index=True)
    for col in cols:
        if col not in out.columns:
            out[col] = ""
    out["qid"] = out["qid"].map(_vg_clean_qid)
    out["label_en"] = out["label_en"].map(_vg_clean_label)
    out["label_ru"] = [_vg_display_ru(en, ru) for en, ru in zip(out["label_en"], out["label_ru"])]
    out = out[out["qid"].str.fullmatch(r"Q\d+").fillna(False)]
    out = out[out["label_en"].map(_vg_is_good_public_label)]
    out = out[out["label_ru"].map(_vg_is_good_public_label)]
    anchors = set(VG_ANCHOR_LABELS.get(prop_name, ()))
    out["_anchor_rank"] = out["label_en"].map(lambda x: 0 if x in anchors else 1)
    out = out.sort_values(["_anchor_rank", "label_en", "qid"]).drop(columns=["_anchor_rank"])
    return out.drop_duplicates(subset=["qid", "label_en"]).reset_index(drop=True)


def _vg_year_window(complexity: str, rng: random.Random) -> Tuple[int, int]:
    # Earliest-date semantics are stricter than any-P577 semantics, so windows are
    # intentionally broad enough to keep L5 generation feasible.
    choices = {
        "L5": [(1996, 2008), (2001, 2012), (2006, 2018), (2010, 2025), (2014, 2025)],
    }
    return rng.choice(choices.get(complexity, choices["L5"]))


def _vg_year_where(y1: Optional[int], y2: Optional[int]) -> List[str]:
    if y1 is None or y2 is None:
        return []
    return [
        "?item wdt:P577 ?releaseDate .",
        "FILTER NOT EXISTS {",
        "  ?item wdt:P577 ?earlierReleaseDate .",
        "  FILTER(?earlierReleaseDate < ?releaseDate)",
        "}",
        "BIND(YEAR(?releaseDate) AS ?releaseYear) .",
        f"FILTER(?releaseYear >= {int(y1)} && ?releaseYear <= {int(y2)}) .",
    ]


def _vg_year_clause_ru(y1: Optional[int], y2: Optional[int]) -> str:
    if y1 is None or y2 is None:
        return ""
    return f"самая ранняя указанная дата публикации игры — с {int(y1)} по {int(y2)} год включительно"


def _vg_year_clause_en(y1: Optional[int], y2: Optional[int]) -> str:
    if y1 is None or y2 is None:
        return ""
    return f"the earliest listed publication date for the game is from {int(y1)} to {int(y2)} inclusive"


VG_V13_BRIDGE_PAIR_LABELS: Dict[Tuple[str, str], Tuple[Tuple[str, str], ...]] = {
    # Distinct developer/publisher pairs only.  Same-entity pairs like Nintendo+Nintendo
    # are explicitly banned for L5 because they collapse two hops into one obvious fact.
    ("developer", "publisher"): (
        ("BioWare", "Electronic Arts"),
        ("id Software", "Bethesda Softworks"),
        ("Blizzard Entertainment", "Activision"),
        ("Rockstar Games", "Take-Two Interactive"),
        ("Naughty Dog", "Sony Interactive Entertainment"),
    ),
    ("publisher", "series"): (
        ("Square Enix", "Final Fantasy"),
        ("Capcom", "Resident Evil"),
        ("Ubisoft", "Assassin's Creed"),
        ("Activision", "Call of Duty"),
        ("Konami", "Metal Gear"),
        ("Bethesda Softworks", "The Elder Scrolls"),
        ("Nintendo", "The Legend of Zelda"),
    ),
    ("developer", "series"): (
        ("Square Enix", "Final Fantasy"),
        ("Capcom", "Resident Evil"),
        ("Ubisoft", "Assassin's Creed"),
        ("Konami", "Metal Gear"),
        ("Nintendo", "The Legend of Zelda"),
    ),
}

VG_V13_VISIBLE_HINTS_BY_HIDDEN_LABEL: Dict[str, Dict[str, Tuple[str, ...]]] = {
    "Final Fantasy": {
        "genre": ("role-playing video game",),
        "platform": ("Microsoft Windows", "PlayStation 4", "PlayStation 3", "Nintendo Switch"),
    },
    "Resident Evil": {
        "genre": ("survival horror", "adventure game", "shooter game"),
        "platform": ("Microsoft Windows", "PlayStation 4", "PlayStation 3", "Nintendo Switch", "Xbox One"),
    },
    "Assassin's Creed": {
        "genre": ("adventure game", "stealth game"),
        "platform": ("Microsoft Windows", "PlayStation 4", "PlayStation 3", "Xbox One", "Nintendo Switch"),
    },
    "Call of Duty": {
        "genre": ("shooter game",),
        "platform": ("Microsoft Windows", "PlayStation 4", "PlayStation 3", "Xbox One"),
    },
    "Metal Gear": {
        "genre": ("stealth game", "adventure game"),
        "platform": ("Microsoft Windows", "PlayStation 4", "PlayStation 3"),
    },
    "The Elder Scrolls": {
        "genre": ("role-playing video game",),
        "platform": ("Microsoft Windows", "PlayStation 4", "Xbox One", "Nintendo Switch"),
    },
    "The Legend of Zelda": {
        "genre": ("adventure game",),
        "platform": ("Nintendo Switch", "Wii", "Nintendo 3DS", "Nintendo DS"),
    },
    "BioWare": {
        "genre": ("role-playing video game",),
        "platform": ("Microsoft Windows", "PlayStation 3", "PlayStation 4", "Xbox One"),
    },
    "id Software": {
        "genre": ("shooter game",),
        "platform": ("Microsoft Windows", "PlayStation 4", "Xbox One"),
    },
    "Blizzard Entertainment": {
        "genre": ("role-playing video game", "strategy video game", "shooter game"),
        "platform": ("Microsoft Windows",),
    },
    "Rockstar Games": {
        "genre": ("adventure game", "shooter game"),
        "platform": ("Microsoft Windows", "PlayStation 4", "PlayStation 3", "Xbox One"),
    },
    "Naughty Dog": {
        "genre": ("adventure game", "platform game"),
        "platform": ("PlayStation 4", "PlayStation 3"),
    },
}


def _vg_entity_by_exact_en_label(prop_name: str, label_en: str) -> Optional[Dict[str, str]]:
    df = _vg_pool(prop_name)
    if df is None or len(df) == 0:
        return None
    sub = df[df["label_en"].astype(str).eq(str(label_en))].copy()
    if sub is None or len(sub) == 0:
        return None
    return _vg_entity_from_row(sub.iloc[0])


def _vg_seed_label_quality_score(label: str) -> Tuple[int, int, str]:
    label = _vg_clean_label(label)
    bad = re.search(r"\b(VR|remaster(?:ed)?|HD|collection|bundle|demo|multiplayer|mobile|season|episode|chapter|trial)\b", label, flags=re.I)
    colon_penalty = 1 if ":" in label else 0
    return (1 if bad else 0, colon_penalty + min(len(label), 80), label.lower())


def _vg_seed_for_bridge_value(bridge: str, bridge_value: Dict[str, str], rng: random.Random) -> Optional[Dict[str, str]]:
    seed_df = _vg_seed_pool_for_bridge_value(bridge, bridge_value)
    if seed_df is None or len(seed_df) == 0:
        return None
    rows = []
    for _, row in seed_df.iterrows():
        seed_en = _vg_clean_label(getattr(row, "seed_label_en", ""))
        if not seed_en or not _vg_is_good_public_label(seed_en):
            continue
        rows.append(row)
    if not rows:
        return None
    rows = sorted(rows, key=lambda r: _vg_seed_label_quality_score(getattr(r, "seed_label_en", "")))
    # Keep a bit of variety while strongly preferring clean seed names.
    row = rng.choice(rows[: min(12, len(rows))])
    seed_en = _vg_clean_label(row.seed_label_en)
    return {
        "qid": row.seed_qid,
        "en": seed_en,
        "ru": _vg_display_ru(seed_en, row.seed_label_ru),
    }


def _vg_pick_bridge_pair(bridge_a: str, bridge_b: str, rng: random.Random) -> Dict[str, Dict[str, str]]:
    if bridge_a == bridge_b:
        raise ValueError("bridge pair must contain two different bridge types")

    pairs = list(VG_V13_BRIDGE_PAIR_LABELS.get((bridge_a, bridge_b), ()))
    rng.shuffle(pairs)
    for label_a, label_b in pairs:
        value_a = _vg_entity_by_exact_en_label(bridge_a, label_a)
        value_b = _vg_entity_by_exact_en_label(bridge_b, label_b)
        if not value_a or not value_b:
            continue
        if _vg_clean_qid(value_a.get("qid")) == _vg_clean_qid(value_b.get("qid")):
            continue
        seed_a = _vg_seed_for_bridge_value(bridge_a, value_a, rng)
        seed_b = _vg_seed_for_bridge_value(bridge_b, value_b, rng)
        if seed_a and seed_b and seed_a.get("qid") != seed_b.get("qid"):
            return {
                "seed_a": seed_a,
                "value_a": value_a,
                "seed_b": seed_b,
                "value_b": value_b,
                "bridge_a": bridge_a,
                "bridge_b": bridge_b,
            }

    # No same-QID fallback for quality L5. If curated pairs are unavailable, skip.
    raise ValueError(f"No quality curated bridge pair for {bridge_a}+{bridge_b}")


def _vg_pick_l5_visible_value(
    bridge_a: str,
    value_a: Dict[str, str],
    bridge_b: str,
    value_b: Dict[str, str],
    prop_name: str,
    rng: random.Random,
) -> Dict[str, str]:
    hints: List[str] = []
    for label in (value_a.get("en"), value_b.get("en")):
        hints.extend(VG_V13_VISIBLE_HINTS_BY_HIDDEN_LABEL.get(str(label), {}).get(prop_name, ()))
    # Add broad fallback labels after targeted hints.  The full gold SELECT is the
    # validator, so a wrong visible pick is simply rejected by the quick/full probe.
    hints.extend(VG_FAST_VISIBLE_LABELS.get(prop_name, ()))
    seen: set = set()
    labels = [x for x in hints if not (x in seen or seen.add(x))]
    rng.shuffle(labels)
    for label in labels:
        ent = _vg_entity_by_exact_en_label(prop_name, label)
        if ent and ent.get("qid"):
            return ent
    return _vg_pick_visible_for_bridge(bridge_a, value_a, prop_name, rng)


def _vg_tpl_two_seed_bridges_l5_quality(
    complexity: str,
    rng: random.Random,
    *,
    bridge_a: str,
    bridge_b: str,
    visible: Sequence[str],
) -> Dict[str, Any]:
    if complexity != "L5":
        raise ValueError("v13 quality builder is L5-only")
    if not visible or not ({"genre", "platform"} & set(visible)):
        raise ValueError("L5 quality builder requires genre/platform visible constraints")

    pack = _vg_pick_bridge_pair(bridge_a, bridge_b, rng)
    seed_a, seed_b = pack["seed_a"], pack["seed_b"]
    value_a, value_b = pack["value_a"], pack["value_b"]
    if _vg_clean_qid(value_a.get("qid")) == _vg_clean_qid(value_b.get("qid")):
        raise ValueError("L5 hidden bridge values collapsed to the same QID")

    pid_a = VG_BRIDGES[bridge_a]["pid"]
    pid_b = VG_BRIDGES[bridge_b]["pid"]
    key_a = VG_BRIDGES[bridge_a]["public_key"]
    key_b = VG_BRIDGES[bridge_b]["public_key"]
    y1, y2 = _vg_year_window(complexity, rng)

    where = [
        f"?item wdt:{pid_a} wd:{value_a['qid']} .",
        f"?item wdt:{pid_b} wd:{value_b['qid']} .",
        f"FILTER(?item != wd:{seed_a['qid']} && ?item != wd:{seed_b['qid']}) .",
    ]
    public_kwargs: Dict[str, Any] = {
        key_a: seed_a,
        key_b: seed_b,
        "publication_date_basis": "earliest listed publication date",
        "release_year_from": y1,
        "release_year_to": y2,
    }
    qid_kwargs: Dict[str, Any] = {
        key_a: seed_a,
        key_b: seed_b,
        "publication_date_basis": "earliest listed publication date",
        "hidden_bridge_values": [
            {"bridge": bridge_a, **value_a},
            {"bridge": bridge_b, **value_b},
        ],
        "release_year_from": y1,
        "release_year_to": y2,
    }
    ru_parts: List[str] = [_vg_bridge_phrase_ru(bridge_a, seed_a), _vg_bridge_phrase_ru(bridge_b, seed_b)]
    en_parts: List[str] = [_vg_bridge_phrase_en(bridge_a, seed_a), _vg_bridge_phrase_en(bridge_b, seed_b)]

    for prop in visible:
        if prop not in VG_PROPERTY_PIDS or prop in (bridge_a, bridge_b):
            continue
        val = _vg_pick_l5_visible_value(bridge_a, value_a, bridge_b, value_b, prop, rng)
        if not val or not val.get("qid"):
            continue
        where.append(f"?item wdt:{VG_PROPERTY_PIDS[prop]} wd:{val['qid']} .")
        public_kwargs[prop] = val
        qid_kwargs[prop] = val
        ru_parts.append(_vg_visible_phrase_ru(prop, val))
        en_parts.append(_vg_visible_phrase_en(prop, val))

    if not ({"genre", "platform"} & set(public_kwargs.keys())):
        raise ValueError("L5 accepted spec would have no visible genre/platform constraint")

    where += _vg_year_where(y1, y2)
    ru_parts.append(_vg_year_clause_ru(y1, y2))
    en_parts.append(_vg_year_clause_en(y1, y2))

    visible_suffix = "_" + "_".join(visible) if visible else ""
    template_id = f"vg_l5_quality_same_{bridge_a}_and_{bridge_b}_as_seeds{visible_suffix}_earliest_year"
    k = _vg_requested(complexity)
    return {
        "template_id": template_id,
        "template_family": "l5_quality_double_hidden_distinct_earliest_date",
        "query_text_ru": _vg_make_query_text_ru(k, ru_parts),
        "query_text_en": _vg_make_query_text_en(k, en_parts),
        "constraints": _vg_public_constraints(**public_kwargs),
        "qid_constraints": _vg_constraint_qids(**qid_kwargs),
        "where_lines": where,
        "requested_count": k,
    }


VG_TEMPLATE_BUILDERS.update({
    "vg_l5_quality_publisher_series_genre_platform_earliest_year": lambda c, r: _vg_tpl_two_seed_bridges_l5_quality(c, r, bridge_a="publisher", bridge_b="series", visible=("genre", "platform")),
    "vg_l5_quality_publisher_series_platform_earliest_year": lambda c, r: _vg_tpl_two_seed_bridges_l5_quality(c, r, bridge_a="publisher", bridge_b="series", visible=("platform",)),
    "vg_l5_quality_publisher_series_genre_earliest_year": lambda c, r: _vg_tpl_two_seed_bridges_l5_quality(c, r, bridge_a="publisher", bridge_b="series", visible=("genre",)),
    "vg_l5_quality_developer_series_genre_platform_earliest_year": lambda c, r: _vg_tpl_two_seed_bridges_l5_quality(c, r, bridge_a="developer", bridge_b="series", visible=("genre", "platform")),
    "vg_l5_quality_developer_series_platform_earliest_year": lambda c, r: _vg_tpl_two_seed_bridges_l5_quality(c, r, bridge_a="developer", bridge_b="series", visible=("platform",)),
    "vg_l5_quality_developer_series_genre_earliest_year": lambda c, r: _vg_tpl_two_seed_bridges_l5_quality(c, r, bridge_a="developer", bridge_b="series", visible=("genre",)),
    "vg_l5_quality_developer_publisher_genre_platform_earliest_year": lambda c, r: _vg_tpl_two_seed_bridges_l5_quality(c, r, bridge_a="developer", bridge_b="publisher", visible=("genre", "platform")),
    "vg_l5_quality_developer_publisher_platform_earliest_year": lambda c, r: _vg_tpl_two_seed_bridges_l5_quality(c, r, bridge_a="developer", bridge_b="publisher", visible=("platform",)),
    "vg_l5_quality_developer_publisher_genre_earliest_year": lambda c, r: _vg_tpl_two_seed_bridges_l5_quality(c, r, bridge_a="developer", bridge_b="publisher", visible=("genre",)),
})

VG_TEMPLATE_PLAN_BY_LEVEL = {
    "L1": (),
    "L2": (),
    "L3": (),
    "L4": (),
    "L5": (
        "vg_l5_quality_publisher_series_genre_platform_earliest_year",
        "vg_l5_quality_developer_series_genre_platform_earliest_year",
        "vg_l5_quality_developer_publisher_genre_platform_earliest_year",
        "vg_l5_quality_publisher_series_platform_earliest_year",
        "vg_l5_quality_publisher_series_genre_earliest_year",
        "vg_l5_quality_developer_series_platform_earliest_year",
        "vg_l5_quality_developer_series_genre_earliest_year",
        "vg_l5_quality_developer_publisher_platform_earliest_year",
        "vg_l5_quality_developer_publisher_genre_earliest_year",
    ),
}


def _vg_l5_hidden_bridge_values_from_qids(qid_constraints: Dict[str, Any]) -> List[Dict[str, Any]]:
    hvs = qid_constraints.get("hidden_bridge_values") if isinstance(qid_constraints, dict) else None
    if isinstance(hvs, list):
        return [x for x in hvs if isinstance(x, dict) and x.get("qid")]
    return []


def _vg_l5_hidden_pair_key_from_record(ex: BenchmarkExample) -> Tuple[Tuple[str, str], ...]:
    cq = (ex.gold_collection_meta or {}).get("constraints_with_qids") or {}
    hvs = _vg_l5_hidden_bridge_values_from_qids(cq)
    return tuple(sorted((str(x.get("bridge") or ""), str(x.get("qid") or "")) for x in hvs))


def _vg_l5_quality_reject_spec(complexity: str, template_id: str, constraints: Dict[str, Any], qid_constraints: Dict[str, Any]) -> Optional[str]:
    if complexity != "L5":
        return None
    hvs = _vg_l5_hidden_bridge_values_from_qids(qid_constraints)
    if len(hvs) < 2:
        return "l5_quality_reject:not_double_hidden"
    qids = [_vg_clean_qid(x.get("qid")) for x in hvs]
    bridges = [str(x.get("bridge") or "") for x in hvs]
    if len(set(qids)) != len(qids):
        return "l5_quality_reject:hidden_bridge_qid_not_distinct"
    if len(set(bridges)) != len(bridges):
        return "l5_quality_reject:hidden_bridge_type_not_distinct"
    if not ({"genre", "platform"} & set((constraints or {}).keys())):
        return "l5_quality_reject:no_visible_genre_or_platform"
    if template_id.endswith("_as_seeds_year") or "_earliest_year" not in template_id:
        return "l5_quality_reject:weak_or_legacy_template"
    return None


_VG_V13_PREV_PRE_GOLD_REJECT_REASON = _vg_pre_gold_reject_reason


def _vg_pre_gold_reject_reason(
    *,
    complexity: str,
    template_id: str,
    constraints: Dict[str, Any],
    qid_constraints: Dict[str, Any],
    seen_public_keys: set,
    hidden_counts_global: Counter,
    hidden_counts_by_level: Counter,
) -> Optional[str]:
    q_reason = _vg_l5_quality_reject_spec(complexity, template_id, constraints, qid_constraints)
    if q_reason:
        return q_reason
    return _VG_V13_PREV_PRE_GOLD_REJECT_REASON(
        complexity=complexity,
        template_id=template_id,
        constraints=constraints,
        qid_constraints=qid_constraints,
        seen_public_keys=seen_public_keys,
        hidden_counts_global=hidden_counts_global,
        hidden_counts_by_level=hidden_counts_by_level,
    )


_VG_V13_PREV_DIVERSITY_REJECT_REASON = _vg_diversity_reject_reason


def _vg_diversity_reject_reason(
    ex: BenchmarkExample,
    *,
    seen_public_keys: set,
    seen_gold_sets: set,
    accepted_records: Sequence[BenchmarkExample],
    hidden_counts_global: Counter,
    hidden_counts_by_level: Counter,
) -> Optional[str]:
    if ex.complexity == "L5":
        cq = (ex.gold_collection_meta or {}).get("constraints_with_qids") or {}
        q_reason = _vg_l5_quality_reject_spec(ex.complexity, str(ex.template_id), ex.constraints, cq)
        if q_reason:
            return q_reason
        pair_key = _vg_l5_hidden_pair_key_from_record(ex)
        if pair_key:
            same_pair_count = sum(1 for prev in accepted_records if _vg_l5_hidden_pair_key_from_record(prev) == pair_key)
            if same_pair_count >= VIDEOGAMES_MAX_SAME_L5_HIDDEN_PAIR:
                return "l5_quality_reject:hidden_bridge_pair_overused"
    return _VG_V13_PREV_DIVERSITY_REJECT_REASON(
        ex,
        seen_public_keys=seen_public_keys,
        seen_gold_sets=seen_gold_sets,
        accepted_records=accepted_records,
        hidden_counts_global=hidden_counts_global,
        hidden_counts_by_level=hidden_counts_by_level,
    )


_VG_V13_PREV_VALIDATE_RECORD_SCHEMA = _vg_validate_record_schema


def _vg_validate_record_schema(ex: BenchmarkExample) -> None:
    _VG_V13_PREV_VALIDATE_RECORD_SCHEMA(ex)
    if ex.complexity == "L5":
        cq = (ex.gold_collection_meta or {}).get("constraints_with_qids") or {}
        reason = _vg_l5_quality_reject_spec(ex.complexity, str(ex.template_id), ex.constraints, cq)
        assert reason is None, f"L5 quality rule failed: {reason}"
        assert "earliestReleaseDate" in ex.sparql_query or "earlierReleaseDate" in ex.sparql_query, "L5 must use earliest-publication-date semantics"


def _vg_run_l5_v13_quality_sanity_checks() -> None:
    assert VIDEOGAMES_TARGET_PLAN == {"L1": 0, "L2": 0, "L3": 0, "L4": 0, "L5": 30}
    assert VIDEOGAMES_OUTPUT_PATH.name == "videogames_l5_quality.jsonl"
    assert VG_TEMPLATE_PLAN_BY_LEVEL["L1"] == VG_TEMPLATE_PLAN_BY_LEVEL["L2"] == VG_TEMPLATE_PLAN_BY_LEVEL["L3"] == VG_TEMPLATE_PLAN_BY_LEVEL["L4"] == ()
    assert len(VG_TEMPLATE_PLAN_BY_LEVEL["L5"]) >= 8
    assert all(t in VG_TEMPLATE_BUILDERS for t in VG_TEMPLATE_PLAN_BY_LEVEL["L5"])
    assert not any(t.endswith("_as_seeds_year") for t in VG_TEMPLATE_PLAN_BY_LEVEL["L5"])
    assert all("_earliest_year" in t for t in VG_TEMPLATE_PLAN_BY_LEVEL["L5"])
    assert VIDEOGAMES_MAX_ACCEPTED_GOLD == 100 and VIDEOGAMES_PROBE_LIMIT == 101


_vg_run_l5_v13_quality_sanity_checks()

print(
    "✅ videogames v13 QUALITY L5 patch enabled: only L5, distinct double-hidden bridges, "
    "genre/platform visible constraints, earliest publication date semantics, no weak year-only L5 templates"
)



# -----------------------------------------------------------------------------
# v14 hotfix: preserve `bridge` metadata inside qid_constraints["hidden_bridge_values"].
#
# Root cause of the 0/30 run in v13:
#   `_vg_constraint_qids()` normalized every list of dicts into
#   {qid, label_en, label_ru}, unintentionally dropping the `bridge` field from
#   hidden_bridge_values.  The L5 quality gate then saw two hidden bridge types
#   as ["", ""] and rejected every candidate before any gold SPARQL query.
# -----------------------------------------------------------------------------
VIDEOGAMES_V14_HIDDEN_BRIDGE_METADATA_FIX = True
VIDEOGAMES_OUTPUT_PATH = VIDEOGAMES_DOMAIN_OUT_DIR / "videogames_l5_quality_v14.jsonl"
VIDEOGAMES_AUDIT_PATH = VIDEOGAMES_DOMAIN_OUT_DIR / "videogames_l5_quality_v14_generation_audit.json"
VIDEOGAMES_CHECKPOINT_PATH = VIDEOGAMES_DOMAIN_OUT_DIR / "videogames_l5_quality_v14_generation_checkpoint.json"


def _vg_constraint_qids(**kwargs: Any) -> Dict[str, Any]:
    """QID constraints for metadata.

    Public `constraints` stay clean and English-only.  This function is only for
    `gold_collection_meta["constraints_with_qids"]` and therefore may preserve
    technical metadata such as the hidden bridge type.
    """
    out: Dict[str, Any] = {}
    for k, v in kwargs.items():
        if v is None:
            continue

        if k == "hidden_bridge_values" and isinstance(v, (list, tuple)):
            vals: List[Dict[str, Any]] = []
            for x in v:
                if not isinstance(x, dict):
                    continue
                qid = _vg_clean_qid(x.get("qid"))
                if not qid:
                    continue
                row: Dict[str, Any] = {
                    "qid": qid,
                    "label_en": _vg_clean_label(x.get("en") or x.get("label_en")),
                    "label_ru": _vg_clean_label(x.get("ru") or x.get("label_ru")),
                }
                bridge = str(x.get("bridge") or "").strip()
                if bridge:
                    row["bridge"] = bridge
                vals.append(row)
            if vals:
                out[k] = vals
            continue

        if isinstance(v, dict):
            qid = _vg_clean_qid(v.get("qid"))
            if qid:
                out[k] = {
                    "qid": qid,
                    "label_en": _vg_clean_label(v.get("en") or v.get("label_en")),
                    "label_ru": _vg_clean_label(v.get("ru") or v.get("label_ru")),
                }
                bridge = str(v.get("bridge") or "").strip()
                if bridge:
                    out[k]["bridge"] = bridge
        elif isinstance(v, (list, tuple)):
            vals: List[Dict[str, Any]] = []
            for x in v:
                if isinstance(x, dict):
                    qid = _vg_clean_qid(x.get("qid"))
                    if qid:
                        row = {
                            "qid": qid,
                            "label_en": _vg_clean_label(x.get("en") or x.get("label_en")),
                            "label_ru": _vg_clean_label(x.get("ru") or x.get("label_ru")),
                        }
                        bridge = str(x.get("bridge") or "").strip()
                        if bridge:
                            row["bridge"] = bridge
                        vals.append(row)
            if vals:
                out[k] = vals
        else:
            out[k] = v
    return out


def _vg_l5_quality_reject_spec(complexity: str, template_id: str, constraints: Dict[str, Any], qid_constraints: Dict[str, Any]) -> Optional[str]:
    if complexity != "L5":
        return None
    hvs = _vg_l5_hidden_bridge_values_from_qids(qid_constraints)
    if len(hvs) < 2:
        return "l5_quality_reject:not_double_hidden"
    qids = [_vg_clean_qid(x.get("qid")) for x in hvs]
    bridges = [str(x.get("bridge") or "").strip() for x in hvs]
    if len(set(qids)) != len(qids):
        return "l5_quality_reject:hidden_bridge_qid_not_distinct"
    if any(not b for b in bridges):
        return "l5_quality_reject:hidden_bridge_type_missing"
    if len(set(bridges)) != len(bridges):
        return "l5_quality_reject:hidden_bridge_type_not_distinct"
    if not ({"genre", "platform"} & set((constraints or {}).keys())):
        return "l5_quality_reject:no_visible_genre_or_platform"
    if template_id.endswith("_as_seeds_year") or "_earliest_year" not in template_id:
        return "l5_quality_reject:weak_or_legacy_template"
    return None


def _vg_run_l5_v14_metadata_sanity_checks() -> None:
    sample = _vg_constraint_qids(
        hidden_bridge_values=[
            {"bridge": "publisher", "qid": "Q1", "en": "Publisher A", "ru": "Publisher A"},
            {"bridge": "series", "qid": "Q2", "en": "Series B", "ru": "Series B"},
        ],
        genre={"qid": "Q3", "en": "role-playing video game", "ru": "ролевая видеоигра"},
        release_year_from=2010,
        release_year_to=2025,
    )
    hvs = sample.get("hidden_bridge_values") or []
    assert len(hvs) == 2 and hvs[0].get("bridge") == "publisher" and hvs[1].get("bridge") == "series"
    assert _vg_l5_quality_reject_spec(
        "L5",
        "vg_l5_quality_publisher_series_genre_earliest_year",
        {"kind": "video_game", "genre": "role-playing video game", "release_year_from": 2010, "release_year_to": 2025},
        sample,
    ) is None
    collapsed = _vg_constraint_qids(hidden_bridge_values=[
        {"bridge": "publisher", "qid": "Q1", "en": "Nintendo", "ru": "Nintendo"},
        {"bridge": "developer", "qid": "Q1", "en": "Nintendo", "ru": "Nintendo"},
    ])
    assert _vg_l5_quality_reject_spec(
        "L5",
        "vg_l5_quality_developer_publisher_genre_earliest_year",
        {"kind": "video_game", "genre": "platform game", "release_year_from": 2010, "release_year_to": 2025},
        collapsed,
    ) == "l5_quality_reject:hidden_bridge_qid_not_distinct"


_vg_run_l5_v14_metadata_sanity_checks()

print(
    "✅ videogames v14 L5 metadata hotfix enabled: hidden bridge types preserved; "
    "quality gate no longer rejects every double-hidden candidate before gold collection"
)


# -----------------------------------------------------------------------------
# FINAL UNIFIED VIDEOGAMES NOTEBOOK OVERLAY
# -----------------------------------------------------------------------------
# This overlay does not change the core builders above.  It only exposes named
# generation profiles that correspond to the pieces that worked best during the
# domain work:
#   * L1-L2: fast direct/simple generation;
#   * L3-L4: faster multihop generation with direct hidden bridge QIDs;
#   * L5: v14 strict quality generator.
#
# Default profile is "none" and RUN_VIDEOGAMES_GENERATION is False, so Run All is
# safe and only registers the generator/functions.
# -----------------------------------------------------------------------------

VIDEOGAMES_FINAL_UNIFIED_NOTEBOOK = True
VIDEOGAMES_GENERATION_PROFILE = str(globals().get("VIDEOGAMES_GENERATION_PROFILE", "none")).strip().lower()

# Keep the strict v14 L5 plan built above before we overwrite per-profile plans.
_VG_FINAL_L5_QUALITY_PLAN = tuple(VG_TEMPLATE_PLAN_BY_LEVEL.get("L5", ()))

_VG_FINAL_L1_L2_PLAN = {
    "L1": (
        "vg_direct_genre_platform_year",
    ),
    "L2": (
        "vg_direct_genre_platform_year",
        "vg_direct_developer_platform_year",
        "vg_direct_publisher_genre_year",
    ),
    "L3": (),
    "L4": (),
    "L5": (),
}

_VG_FINAL_L3_L4_PLAN = {
    "L1": (),
    "L2": (),
    "L3": (
        "vg_direct_genre_platform_year",
        "vg_direct_developer_platform_year",
        "vg_direct_publisher_genre_year",
        "vg_same_developer_as_seed_year",
        "vg_same_publisher_as_seed_year",
        "vg_same_series_as_seed_year",
        "vg_same_developer_as_seed_platform_year",
        "vg_same_publisher_as_seed_genre_year",
    ),
    "L4": (
        "vg_same_developer_as_seed_genre_platform_year",
        "vg_same_publisher_as_seed_genre_platform_year",
        "vg_same_series_as_seed_genre_platform_year",
        "vg_same_engine_as_seed_genre_platform_year",
        "vg_same_developer_as_seed_platform_year",
        "vg_same_publisher_as_seed_genre_year",
        "vg_same_series_as_seed_platform_year",
        "vg_same_engine_as_seed_genre_year",
    ),
    "L5": (),
}

_VG_FINAL_FULL_PLAN = {
    "L1": _VG_FINAL_L1_L2_PLAN["L1"],
    "L2": _VG_FINAL_L1_L2_PLAN["L2"],
    "L3": _VG_FINAL_L3_L4_PLAN["L3"],
    "L4": _VG_FINAL_L3_L4_PLAN["L4"],
    "L5": _VG_FINAL_L5_QUALITY_PLAN,
}

VIDEOGAMES_FINAL_PROFILES: Dict[str, Dict[str, Any]] = {
    "none": {
        "target_plan": {"L1": 0, "L2": 0, "L3": 0, "L4": 0, "L5": 0},
        "template_plan": _VG_FINAL_FULL_PLAN,
        "output_stem": "videogames_final_unified_no_generation",
        "strict_target": False,
        "attempt_budget": {"L1": 0, "L2": 0, "L3": 0, "L4": 0, "L5": 0},
    },
    "l1_l2_success": {
        "target_plan": {"L1": 10, "L2": 20, "L3": 0, "L4": 0, "L5": 0},
        "template_plan": _VG_FINAL_L1_L2_PLAN,
        "output_stem": "videogames_l1_l2_success",
        "strict_target": True,
        "attempt_budget": {"L1": 260, "L2": 420, "L3": 0, "L4": 0, "L5": 0},
    },
    "l3_l4_success": {
        "target_plan": {"L1": 0, "L2": 0, "L3": 35, "L4": 35, "L5": 0},
        "template_plan": _VG_FINAL_L3_L4_PLAN,
        "output_stem": "videogames_l3_l4_success",
        "strict_target": True,
        "attempt_budget": {"L1": 0, "L2": 0, "L3": 1800, "L4": 2600, "L5": 0},
    },
    "l5_quality": {
        "target_plan": {"L1": 0, "L2": 0, "L3": 0, "L4": 0, "L5": 30},
        "template_plan": {"L1": (), "L2": (), "L3": (), "L4": (), "L5": _VG_FINAL_L5_QUALITY_PLAN},
        "output_stem": "videogames_l5_quality_v14",
        "strict_target": True,
        "attempt_budget": {"L1": 0, "L2": 0, "L3": 0, "L4": 0, "L5": 9000},
    },
    "full_formal": {
        "target_plan": {"L1": 10, "L2": 20, "L3": 35, "L4": 35, "L5": 30},
        "template_plan": _VG_FINAL_FULL_PLAN,
        "output_stem": "videogames_final_unified_full_formal",
        "strict_target": True,
        "attempt_budget": {"L1": 260, "L2": 420, "L3": 1800, "L4": 2600, "L5": 9000},
    },
}


def apply_videogames_generation_profile(profile: str = VIDEOGAMES_GENERATION_PROFILE) -> None:
    """Apply one final generation profile in-place.

    The function intentionally changes only runtime config variables.  It does
    not redefine builders or schema, so JSONL structure remains exactly the same
    as the other benchmark domains based on BenchmarkExample.
    """
    global VIDEOGAMES_GENERATION_PROFILE, VIDEOGAMES_TARGET_PLAN, VG_TEMPLATE_PLAN_BY_LEVEL
    global VIDEOGAMES_OUTPUT_PATH, VIDEOGAMES_AUDIT_PATH, VIDEOGAMES_CHECKPOINT_PATH
    global VIDEOGAMES_STRICT_TARGET, VIDEOGAMES_LEVEL_ATTEMPT_BUDGET

    profile = str(profile or "none").strip().lower()
    if profile not in VIDEOGAMES_FINAL_PROFILES:
        raise ValueError(f"Unknown VIDEOGAMES_GENERATION_PROFILE={profile!r}; choose one of {sorted(VIDEOGAMES_FINAL_PROFILES)}")

    cfg = VIDEOGAMES_FINAL_PROFILES[profile]
    VIDEOGAMES_GENERATION_PROFILE = profile
    VIDEOGAMES_TARGET_PLAN = dict(cfg["target_plan"])
    VG_TEMPLATE_PLAN_BY_LEVEL = {k: tuple(v) for k, v in cfg["template_plan"].items()}

    stem = str(cfg["output_stem"])
    VIDEOGAMES_OUTPUT_PATH = VIDEOGAMES_DOMAIN_OUT_DIR / f"{stem}.jsonl"
    VIDEOGAMES_AUDIT_PATH = VIDEOGAMES_DOMAIN_OUT_DIR / f"{stem}_generation_audit.json"
    VIDEOGAMES_CHECKPOINT_PATH = VIDEOGAMES_DOMAIN_OUT_DIR / f"{stem}_generation_checkpoint.json"
    VIDEOGAMES_STRICT_TARGET = bool(cfg.get("strict_target", VIDEOGAMES_STRICT_TARGET))
    VIDEOGAMES_LEVEL_ATTEMPT_BUDGET = dict(cfg.get("attempt_budget", VIDEOGAMES_LEVEL_ATTEMPT_BUDGET))


apply_videogames_generation_profile(VIDEOGAMES_GENERATION_PROFILE)


def merge_videogames_jsonl_parts(
    part_paths: Sequence[str | Path],
    output_path: str | Path = VIDEOGAMES_DOMAIN_OUT_DIR / "videogames_final_merged.jsonl",
    *,
    dedupe_by: str = "id",
) -> List[Dict[str, Any]]:
    """Merge already-generated videogames JSONL parts without changing records.

    This is useful for the final manual assembly step after separate successful
    L1-L2 / L3-L4 / L5 runs.  It preserves the original BenchmarkExample key
    order and drops duplicate records by `id` by default.
    """
    output_path = Path(output_path)
    merged: List[Dict[str, Any]] = []
    seen_merge_keys: set[str] = set()

    for p in part_paths:
        path = Path(p)
        if not path.exists():
            print(f"[WARN] merge source missing, skipped: {path}")
            continue
        with path.open("r", encoding="utf-8") as f:
            for line_no, line in enumerate(f, start=1):
                if not line.strip():
                    continue
                rec = json.loads(line)
                if list(rec.keys()) != VG_EXPECTED_JSON_KEYS:
                    raise ValueError(f"Bad BenchmarkExample key order in {path}:{line_no}")
                key = str(rec.get(dedupe_by, json.dumps(rec, ensure_ascii=False, sort_keys=True)))
                if key in seen_merge_keys:
                    continue
                seen_merge_keys.add(key)
                merged.append(rec)

    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("w", encoding="utf-8") as f:
        for rec in merged:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
    print(f"Merged {len(merged)} records -> {output_path}")
    return merged


def summarize_videogames_jsonl(path: str | Path) -> Dict[str, Any]:
    """Small offline summary helper for the final generated/merged JSONL."""
    path = Path(path)
    rows: List[Dict[str, Any]] = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return {
        "path": str(path),
        "records": len(rows),
        "by_complexity": dict(Counter(r.get("complexity") for r in rows)),
        "by_template_id": dict(Counter(r.get("template_id") for r in rows)),
        "gold_count_min": min([len(r.get("gold_answer_qids", [])) for r in rows] or [0]),
        "gold_count_max": max([len(r.get("gold_answer_qids", [])) for r in rows] or [0]),
    }

print(
    "✅ videogames FINAL unified overlay loaded: "
    f"profile={VIDEOGAMES_GENERATION_PROFILE!r}, target={VIDEOGAMES_TARGET_PLAN}, "
    f"output={VIDEOGAMES_OUTPUT_PATH}. "
    "Default Run All is safe because RUN_VIDEOGAMES_GENERATION is False."
)

if RUN_VIDEOGAMES_GENERATION and VIDEOGAMES_GENERATION_PROFILE == "none":
    raise ValueError(
        "RUN_VIDEOGAMES_GENERATION=True but VIDEOGAMES_GENERATION_PROFILE='none'. "
        "Set profile to 'l1_l2_success', 'l3_l4_success', 'l5_quality', or 'full_formal'."
    )

if RUN_VIDEOGAMES_GENERATION:
    VIDEOGAMES_RECORDS = generate_videogames_dataset(
        target_plan=VIDEOGAMES_TARGET_PLAN,
        output_path=VIDEOGAMES_OUTPUT_PATH,
        audit_path=VIDEOGAMES_AUDIT_PATH,
        overwrite=OVERWRITE_VIDEOGAMES_OUTPUT,
    )
else:
    print("RUN_VIDEOGAMES_GENERATION is False; generator has been registered but dataset generation did not run.")
